# Notebook 2 — Internal modelling, validation, calibration and model locking

**Purpose.** This notebook develops and internally evaluates the binary HOME-versus-ADMITTED prediction models using the frozen MIMIC-IV ESI-3 abdominal-pain cohort created in `01_data_preparation.ipynb`. It does not reconstruct cohorts, redefine outcomes, remap features or access MC-MED external outcomes.

**Authoritative inputs.** The notebook loads `mimic_train.parquet`, `mimic_validation.parquet`, `mimic_test.parquet`, `feature_manifest.csv`, `split_assignments.parquet` and `pipeline_config.yaml` from `data/processed/`. Every analysis must be reproducible from these files in a fresh kernel without relying on variables retained from Notebook 1.

**Frozen cohort.** The expected MIMIC-IV partitions are 5,985 training encounters, 1,282 validation encounters and 1,283 internal-test encounters. The binary outcome is HOME = 0 and ADMITTED = 1. The internal test set remains untouched until the preprocessing strategy, model, hyperparameters and primary decision threshold have been selected using only the training and validation partitions.

**Frozen feature schema.** The raw model schema contains 2,495 predictors: 9 S0 arrival and triage features, 2,055 S1 historical diagnosis and medication features, and 431 S2 abnormal laboratory and imaging-result features. The current schema uses `age` and excludes the legacy `pain_score` variable. The current structured S0 + S1 + S2 feature order is authoritative; the legacy Pathway 5 order is retained only as an audit.

**Preprocessing plan.** Numerical imputation and categorical encoding are fitted using MIMIC-IV training rows only. The fitted transformation is then applied unchanged to validation and test data. The seven numerical S0 variables use training-derived median imputation. `sex_gender` and `arrival_transport_grouped` use training-derived categorical levels and one-hot encoding with safe handling of unknown categories. The 2,486 binary S1 and S2 predictors pass through unchanged. Under the current training categories, the expected transformed schema contains 2,500 model columns.

**Feature-stage comparison.** Model performance will be evaluated using prespecified cumulative feature blocks: S0 triage information, S0 + S1 history-enhanced information, and S0 + S1 + S2 objective work-up information. Any additional timing comparison will be performed only when supported by the frozen feature manifest.

**Model-development plan.** XGBoost, LightGBM and Random Forest models will be trained using the MIMIC-IV training partition. Hyperparameter comparison will use the validation partition only. Candidate models will be assessed using the same prespecified outcome direction, feature schema and reproducible random seed.

**Model-selection policy.** The primary model will be selected using a documented rule rather than choosing whichever isolated metric appears most favourable. Validation data will be used to select the operating threshold. The internal test set will be evaluated only once after the preprocessing pipeline, model hyperparameters and threshold have been locked.

**Internal evaluation.** The locked model will be assessed using AUROC, AUPRC, sensitivity, specificity, positive predictive value, negative predictive value, F1 score, balanced accuracy, log loss and Brier score. Calibration will be examined using calibration bins, calibration intercept and calibration slope. Patient-level bootstrap resampling will be used to estimate 95% confidence intervals.

**Saved outputs.** This notebook will save the complete fitted preprocessing-plus-model pipeline, optional candidate models, internal probabilities and class predictions, point estimates and confidence intervals, validation-only threshold selection, calibration results, feature-stage comparisons and model metadata. The intended outputs include `final_model_pipeline.joblib`, `candidate_models.joblib`, `internal_predictions.parquet`, `internal_metrics.csv`, `threshold_selection.csv`, `calibration_internal.csv`, `feature_stage_comparison.csv` and `model_metadata.json`.

**External-validation boundary.** `mcmed_external.parquet` and MC-MED labels are not used for model fitting, preprocessing, hyperparameter tuning, model selection or primary-threshold selection in this notebook. The completely locked pipeline and threshold will be transferred to `03_external_validation.ipynb`.

**Reproducibility rules.** The notebook must verify the remote Python executable, project directory, input-file existence, row counts, label support, feature count and feature-order hash before modelling. All random seeds and package versions will be recorded. Every final metric must be reproducible from the saved prediction file. No manually typed performance values will be used in publication tables.

**Completion criterion.** Notebook 2 is complete only when the selected fitted pipeline, locked validation-derived threshold, internal predictions, internal-test metrics, calibration outputs, feature-stage comparison and model metadata have been saved and successfully reloaded from disk.

In [ ]:

from datetime import datetime, timezone
from pathlib import Path
import gc
import hashlib
import json
import os
import platform
import sys
import time
import warnings


import numpy as np
import pandas as pd

from scipy import sparse
from scipy.stats import (
    randint,
    loguniform,
    uniform,
)


import matplotlib.pyplot as plt
import shap
import plotly.graph_objects as go


import joblib


import sklearn
from sklearn.linear_model import LogisticRegression

from sklearn.base import clone
from sklearn.calibration import (
    CalibratedClassifierCV,
    calibration_curve,
)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
)

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    get_scorer_names,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


import lightgbm as lgb
import xgboost as xgb

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


print(
    "NOTEBOOK 2 IMPORTS COMPLETED"
)

print(
    "Python:",
    platform.python_version(),
)

print(
    "pandas:",
    pd.__version__,
)

print(
    "NumPy:",
    np.__version__,
)

print(
    "scikit-learn:",
    sklearn.__version__,
)

print(
    "XGBoost:",
    xgb.__version__,
)

print(
    "LightGBM:",
    lgb.__version__,
)

In [ ]:

PROJECT_ROOT = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_PROJECT_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()

EXPECTED_PYTHON = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_PYTHON",
        sys.executable,
    )
).expanduser().resolve()

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)



actual_python = Path(
    sys.executable
).resolve()

expected_python = (
    EXPECTED_PYTHON
    .resolve()
)


print("Python executable:")
print(actual_python)

print("\nExpected project executable:")
print(expected_python)

print("\nPython version:")
print(sys.version.split()[0])

print("\nOperating system:")
print(platform.platform())

print("\nKernel process ID:")
print(os.getpid())


assert actual_python == expected_python, (
    "Notebook 2 is using the wrong Python environment.\n"
    f"Current:  {actual_python}\n"
    f"Expected: {expected_python}"
)



assert PROJECT_ROOT.exists(), (
    f"Project directory does not exist: {PROJECT_ROOT}"
)

os.chdir(
    PROJECT_ROOT
)


assert Path.cwd().resolve() == (
    PROJECT_ROOT.resolve()
)


print("\nProject working directory:")
print(Path.cwd())



notebook1_memory_objects = [
    "mimic_full_feature_matrix",
    "mcmed_full_feature_matrix",
    "mimic_s0_validated",
    "mcmed_s0_validated",
    "mimic_s1_matrix",
    "mcmed_s1_matrix",
    "mimic_s2_objective_workup_matrix",
    "mcmed_s2_objective_workup_matrix",
    "X_mimic_train_prepared",
    "X_mcmed_external_prepared",
]


unexpected_inherited_objects = [
    object_name
    for object_name in notebook1_memory_objects
    if object_name in globals()
]


assert not unexpected_inherited_objects, (
    "Notebook 2 appears to be sharing Notebook 1's "
    "live kernel memory.\n"
    "Inherited objects found:\n"
    f"{unexpected_inherited_objects}\n\n"
    "Restart Notebook 2 using a separate kernel process."
)


print(
    "\nFresh-kernel check: PASSED"
)



required_handoff_files = {
    "MIMIC-IV training dataset": (
        PROCESSED_DATA_DIR
        / "mimic_train.parquet"
    ),
    "MIMIC-IV validation dataset": (
        PROCESSED_DATA_DIR
        / "mimic_validation.parquet"
    ),
    "MIMIC-IV test dataset": (
        PROCESSED_DATA_DIR
        / "mimic_test.parquet"
    ),
    "MC-MED external dataset": (
        PROCESSED_DATA_DIR
        / "mcmed_external.parquet"
    ),
    "Feature manifest": (
        PROCESSED_DATA_DIR
        / "feature_manifest.csv"
    ),
    "Split assignments": (
        PROCESSED_DATA_DIR
        / "split_assignments.parquet"
    ),
    "Pipeline configuration": (
        PROCESSED_DATA_DIR
        / "pipeline_config.yaml"
    ),
    "Missingness summary": (
        PROCESSED_DATA_DIR
        / "missingness_summary.csv"
    ),
    "Cohort flow": (
        PROCESSED_DATA_DIR
        / "cohort_flow.csv"
    ),
    "Artifact checksums": (
        PROCESSED_DATA_DIR
        / "artifact_checksums.csv"
    ),
}


handoff_file_audit = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "exists": path.exists(),
            "size_mb": (
                round(
                    path.stat().st_size
                    / 1024**2,
                    3,
                )
                if path.exists()
                else None
            ),
            "path": str(path),
        }
        for artifact_name, path
        in required_handoff_files.items()
    ]
)


display(
    handoff_file_audit
)


missing_handoff_files = (
    handoff_file_audit.loc[
        ~handoff_file_audit["exists"],
        "path",
    ]
    .tolist()
)


assert not missing_handoff_files, (
    "Notebook 1 hand-off files are missing:\n"
    + "\n".join(
        missing_handoff_files
    )
)


print(
    "\nNOTEBOOK 2 ENVIRONMENT VERIFIED"
)

print(
    "Same project virtual environment: YES"
)

print(
    "Fresh independent kernel: YES"
)

print(
    "Notebook 1 hand-off files available: YES"
)

print(
    "\nNo Notebook 1 in-memory dataframe was used."
)

In [ ]:
EXPECTED_VENV = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_VENV",
        sys.prefix,
    )
).expanduser().resolve()


print("Raw sys.executable:")
print(sys.executable)

print("\nsys.prefix:")
print(sys.prefix)

print("\nsys.base_prefix:")
print(sys.base_prefix)

print("\nVIRTUAL_ENV:")
print(os.environ.get("VIRTUAL_ENV"))

print("\nSite-package directories:")
for path in site.getsitepackages():
    print(path)


assert (
    Path(sys.prefix).resolve()
    ==
    EXPECTED_VENV.resolve()
), (
    "Notebook 2 is not running from the expected "
    "project virtual environment.\n"
    f"Current sys.prefix: {sys.prefix}\n"
    f"Expected: {EXPECTED_VENV}"
)


assert sys.prefix != sys.base_prefix, (
    "The kernel appears to be using the system Python "
    "rather than an activated virtual environment."
)


print(
    "\nNOTEBOOK 2 VIRTUAL ENVIRONMENT VERIFIED"
)

In [ ]:

EXPECTED_VENV = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_VENV",
        sys.prefix,
    )
).expanduser().resolve()

EXPECTED_PROJECT_ROOT = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_PROJECT_ROOT",
        PROJECT_ROOT,
    )
).expanduser().resolve()

EXPECTED_TMUX_SESSION = (
    "External_val_paper"
)



print("Raw sys.executable:")
print(sys.executable)

print("\nsys.prefix:")
print(sys.prefix)

print("\nsys.base_prefix:")
print(sys.base_prefix)

print("\nVIRTUAL_ENV:")
print(os.environ.get("VIRTUAL_ENV"))


assert Path(
    sys.executable
) == (
    EXPECTED_VENV
    / "bin"
    / "python"
), (
    "Notebook 2 is not using the expected virtual-"
    "environment executable.\n"
    f"Observed: {sys.executable}"
)


assert Path(
    sys.prefix
) == EXPECTED_VENV, (
    "Notebook 2 is not running inside the expected "
    "virtual environment.\n"
    f"Observed sys.prefix: {sys.prefix}\n"
    f"Expected: {EXPECTED_VENV}"
)


assert sys.prefix != sys.base_prefix, (
    "The kernel appears to be using the base Python "
    "environment."
)



def read_process_record(
    pid,
):
    """
    Read PID, parent PID, command and working directory from
    the Linux /proc filesystem.
    """

    process_directory = Path(
        f"/proc/{pid}"
    )


    if not process_directory.exists():
        return None


    try:

        status_lines = (
            process_directory
            .joinpath("status")
            .read_text(
                encoding="utf-8",
                errors="replace",
            )
            .splitlines()
        )


        status_values = {}


        for line in status_lines:

            if ":" not in line:
                continue

            key, value = line.split(
                ":",
                maxsplit=1,
            )

            status_values[
                key.strip()
            ] = value.strip()


        parent_pid = int(
            status_values.get(
                "PPid",
                0,
            )
        )


        process_name = (
            status_values.get(
                "Name",
                "",
            )
        )


        command_bytes = (
            process_directory
            .joinpath("cmdline")
            .read_bytes()
        )


        command = (
            command_bytes
            .replace(
                b"\x00",
                b" ",
            )
            .decode(
                "utf-8",
                errors="replace",
            )
            .strip()
        )


        try:

            working_directory = os.readlink(
                process_directory
                / "cwd"
            )

        except (
            FileNotFoundError,
            PermissionError,
            OSError,
        ):

            working_directory = ""


        return {
            "pid": int(pid),
            "parent_pid": parent_pid,
            "process_name": process_name,
            "command": command,
            "working_directory": (
                working_directory
            ),
        }


    except (
        FileNotFoundError,
        PermissionError,
        ProcessLookupError,
    ):

        return None



kernel_pid = os.getpid()

process_records = []

visited_pids = set()

current_pid = kernel_pid


while (
    current_pid > 0
    and
    current_pid not in visited_pids
):

    visited_pids.add(
        current_pid
    )


    record = read_process_record(
        current_pid
    )


    if record is None:
        break


    process_records.append(
        record
    )


    if record[
        "parent_pid"
    ] in {
        0,
        1,
    }:

        if record[
            "parent_pid"
        ] == 1:

            parent_record = (
                read_process_record(
                    1
                )
            )

            if parent_record is not None:

                process_records.append(
                    parent_record
                )

        break


    current_pid = record[
        "parent_pid"
    ]


kernel_process_ancestry = pd.DataFrame(
    process_records
)


print(
    "\nNotebook 2 kernel process ancestry:"
)

display(
    kernel_process_ancestry
)



def combined_process_text(
    row,
):

    return (
        str(
            row.get(
                "process_name",
                "",
            )
        )
        +
        " "
        +
        str(
            row.get(
                "command",
                "",
            )
        )
    ).lower()


ancestry_text = (
    kernel_process_ancestry
    .apply(
        combined_process_text,
        axis=1,
    )
)


ipykernel_detected = bool(
    ancestry_text.str.contains(
        "ipykernel",
        regex=False,
    ).any()
)


jupyter_detected = bool(
    ancestry_text.str.contains(
        "jupyter",
        regex=False,
    ).any()
)


tmux_detected = bool(
    ancestry_text.str.contains(
        "tmux",
        regex=False,
    ).any()
)



tmux_environment_value = (
    os.environ.get(
        "TMUX",
        "",
    )
)


tmux_session_name = ""


if tmux_environment_value:

    try:

        tmux_session_name = (
            subprocess.check_output(
                [
                    "tmux",
                    "display-message",
                    "-p",
                    "#S",
                ],
                text=True,
                stderr=(
                    subprocess.STDOUT
                ),
            )
            .strip()
        )

    except (
        subprocess.CalledProcessError,
        FileNotFoundError,
    ):

        tmux_session_name = (
            "<unable to query>"
        )


process_runtime_audit = pd.DataFrame(
    [
        {
            "check": (
                "Correct virtual environment"
            ),
            "passed": (
                Path(sys.prefix)
                == EXPECTED_VENV
            ),
            "observed": sys.prefix,
        },
        {
            "check": (
                "Independent ipykernel process"
            ),
            "passed": (
                ipykernel_detected
            ),
            "observed": kernel_pid,
        },
        {
            "check": (
                "Jupyter server in ancestry"
            ),
            "passed": (
                jupyter_detected
            ),
            "observed": (
                "Detected"
                if jupyter_detected
                else "Not detected"
            ),
        },
        {
            "check": (
                "tmux server in ancestry"
            ),
            "passed": (
                tmux_detected
            ),
            "observed": (
                "Detected"
                if tmux_detected
                else "Not detected"
            ),
        },
        {
            "check": (
                "Inherited TMUX environment"
            ),
            "passed": bool(
                tmux_environment_value
            ),
            "observed": (
                tmux_environment_value
                or "<empty>"
            ),
        },
        {
            "check": (
                "tmux session name"
            ),
            "passed": (
                tmux_session_name
                ==
                EXPECTED_TMUX_SESSION
            ),
            "observed": (
                tmux_session_name
                or "<not available>"
            ),
        },
    ]
)


print(
    "\nNotebook 2 runtime audit:"
)

display(
    process_runtime_audit
)


assert ipykernel_detected, (
    "The current process does not appear to be an "
    "ipykernel process."
)


assert jupyter_detected, (
    "A Jupyter server was not detected in the kernel's "
    "process ancestry."
)


assert tmux_detected, (
    "A tmux server was not detected in the kernel's "
    "process ancestry. The Jupyter server may not have "
    "been launched from tmux."
)


if tmux_session_name not in {
    "",
    "<unable to query>",
}:

    assert (
        tmux_session_name
        ==
        EXPECTED_TMUX_SESSION
    ), (
        "Notebook 2 is connected to an unexpected tmux "
        "session.\n"
        f"Observed: {tmux_session_name}\n"
        f"Expected: {EXPECTED_TMUX_SESSION}"
    )


print(
    "\nNOTEBOOK 2 PERSISTENT RUNTIME VERIFIED"
)

print(
    "Virtual environment:",
    EXPECTED_VENV,
)

print(
    "Kernel PID:",
    kernel_pid,
)

print(
    "Jupyter server ancestry: YES"
)

print(
    "tmux ancestry: YES"
)

print(
    "tmux session:",
    tmux_session_name,
)

print(
    "\nNotebook 2 should continue running if the SSH "
    "terminal or VS Code connection temporarily closes."
)

## Load and validate Notebook 1 hand-off files

In [ ]:


PROJECT_ROOT = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_PROJECT_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


NOTEBOOK2_INPUT_PATHS = {
    "mimic_train": (
        PROCESSED_DATA_DIR
        / "mimic_train.parquet"
    ),
    "mimic_validation": (
        PROCESSED_DATA_DIR
        / "mimic_validation.parquet"
    ),
    "mimic_test": (
        PROCESSED_DATA_DIR
        / "mimic_test.parquet"
    ),
    "mcmed_external": (
        PROCESSED_DATA_DIR
        / "mcmed_external.parquet"
    ),
    "feature_manifest": (
        PROCESSED_DATA_DIR
        / "feature_manifest.csv"
    ),
    "split_assignments": (
        PROCESSED_DATA_DIR
        / "split_assignments.parquet"
    ),
    "pipeline_config": (
        PROCESSED_DATA_DIR
        / "pipeline_config.yaml"
    ),
    "artifact_checksums": (
        PROCESSED_DATA_DIR
        / "artifact_checksums.csv"
    ),
}


missing_paths = [
    str(path)
    for path in NOTEBOOK2_INPUT_PATHS.values()
    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(
        "Required Notebook 1 hand-off files are missing:\n"
        +
        "\n".join(
            missing_paths
        )
    )



def file_sha256(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()


artifact_checksums = pd.read_csv(
    NOTEBOOK2_INPUT_PATHS[
        "artifact_checksums"
    ]
)


required_checksum_columns = {
    "artifact",
    "filename",
    "size_bytes",
    "sha256",
}


missing_checksum_columns = (
    required_checksum_columns
    -
    set(
        artifact_checksums.columns
    )
)


if missing_checksum_columns:

    raise ValueError(
        "artifact_checksums.csv is missing columns:\n"
        f"{sorted(missing_checksum_columns)}"
    )


checksum_audit_records = []


for checksum_row in artifact_checksums.itertuples(
    index=False
):

    artifact_path = (
        PROCESSED_DATA_DIR
        /
        str(
            checksum_row.filename
        )
    )


    if not artifact_path.exists():

        raise FileNotFoundError(
            "An official Notebook 1 artifact is missing:\n"
            f"{artifact_path}"
        )


    observed_size_bytes = int(
        artifact_path
        .stat()
        .st_size
    )


    observed_sha256 = file_sha256(
        artifact_path
    )


    size_matches = (
        observed_size_bytes
        ==
        int(
            checksum_row.size_bytes
        )
    )


    checksum_matches = (
        observed_sha256
        ==
        str(
            checksum_row.sha256
        )
    )


    checksum_audit_records.append(
        {
            "artifact": (
                checksum_row.artifact
            ),
            "filename": (
                checksum_row.filename
            ),
            "expected_size_bytes": int(
                checksum_row.size_bytes
            ),
            "observed_size_bytes": (
                observed_size_bytes
            ),
            "size_matches": (
                size_matches
            ),
            "checksum_matches": (
                checksum_matches
            ),
        }
    )


artifact_checksum_audit = pd.DataFrame(
    checksum_audit_records
)


if not artifact_checksum_audit[
    "size_matches"
].all():

    failed_rows = (
        artifact_checksum_audit.loc[
            ~artifact_checksum_audit[
                "size_matches"
            ]
        ]
    )

    raise ValueError(
        "At least one Notebook 1 artifact has an unexpected "
        "file size:\n"
        f"{failed_rows}"
    )


if not artifact_checksum_audit[
    "checksum_matches"
].all():

    failed_rows = (
        artifact_checksum_audit.loc[
            ~artifact_checksum_audit[
                "checksum_matches"
            ]
        ]
    )

    raise ValueError(
        "At least one Notebook 1 artifact failed SHA-256 "
        "verification:\n"
        f"{failed_rows}"
    )



with NOTEBOOK2_INPUT_PATHS[
    "pipeline_config"
].open(
    "r",
    encoding="utf-8",
) as file:

    pipeline_config = json.load(
        file
    )


required_pipeline_config_keys = {
    "pipeline_stage",
    "cohort_sizes",
    "outcome_support",
    "raw_feature_count",
    "feature_groups",
    "feature_order_sha256",
    "mimic_temporal_split_policy",
}


missing_pipeline_config_keys = (
    required_pipeline_config_keys
    -
    set(
        pipeline_config
    )
)


if missing_pipeline_config_keys:

    raise ValueError(
        "pipeline_config.yaml is missing required fields:\n"
        f"{sorted(missing_pipeline_config_keys)}"
    )


assert (
    pipeline_config[
        "pipeline_stage"
    ]
    ==
    "01_data_preparation_complete"
)


assert int(
    pipeline_config[
        "raw_feature_count"
    ]
) == 2495



partition_key_lookup = {
    "MIMIC-IV train": (
        "mimic_train"
    ),
    "MIMIC-IV validation": (
        "mimic_validation"
    ),
    "MIMIC-IV test": (
        "mimic_test"
    ),
}


expected_split_labels = {
    "MIMIC-IV train": "train",
    "MIMIC-IV validation": (
        "validation"
    ),
    "MIMIC-IV test": "test",
}


expected_rows = {}


expected_outcome_support = {}


for partition_name, config_key in (
    partition_key_lookup.items()
):

    if (
        config_key
        not in pipeline_config[
            "cohort_sizes"
        ]
    ):

        raise KeyError(
            f"Missing cohort size for {config_key} "
            "in pipeline_config.yaml."
        )


    if (
        config_key
        not in pipeline_config[
            "outcome_support"
        ]
    ):

        raise KeyError(
            f"Missing outcome support for {config_key} "
            "in pipeline_config.yaml."
        )


    expected_rows[
        partition_name
    ] = int(
        pipeline_config[
            "cohort_sizes"
        ][
            config_key
        ]
    )


    support_config = (
        pipeline_config[
            "outcome_support"
        ][
            config_key
        ]
    )


    expected_outcome_support[
        partition_name
    ] = {
        0: int(
            support_config[
                "home"
            ]
        ),
        1: int(
            support_config[
                "admitted"
            ]
        ),
    }


assert sum(
    expected_rows.values()
) == 8550


assert sum(
    support[0]
    for support
    in expected_outcome_support.values()
) == 6254


assert sum(
    support[1]
    for support
    in expected_outcome_support.values()
) == 2296



feature_manifest = pd.read_csv(
    NOTEBOOK2_INPUT_PATHS[
        "feature_manifest"
    ]
)


required_manifest_columns = {
    "feature_name",
    "family",
    "timing_stage",
    "storage_type",
    "full_position",
    "feature_order_sha256",
}


missing_manifest_columns = (
    required_manifest_columns
    -
    set(
        feature_manifest.columns
    )
)


if missing_manifest_columns:

    raise ValueError(
        "Feature manifest is missing columns:\n"
        f"{sorted(missing_manifest_columns)}"
    )


feature_manifest[
    "full_position"
] = pd.to_numeric(
    feature_manifest[
        "full_position"
    ],
    errors="raise",
)


feature_manifest = (
    feature_manifest
    .sort_values(
        "full_position",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


assert len(
    feature_manifest
) == 2495


assert feature_manifest[
    "feature_name"
].notna().all()


assert feature_manifest[
    "feature_name"
].is_unique


assert feature_manifest[
    "full_position"
].is_unique


raw_feature_order = (
    feature_manifest[
        "feature_name"
    ]
    .astype(str)
    .tolist()
)


assert len(
    raw_feature_order
) == 2495


assert len(
    set(
        raw_feature_order
    )
) == 2495


raw_feature_order_sha256 = (
    hashlib.sha256(
        "\n".join(
            raw_feature_order
        )
        .encode(
            "utf-8"
        )
    )
    .hexdigest()
)


expected_feature_hash = str(
    pipeline_config[
        "feature_order_sha256"
    ]
)


assert (
    raw_feature_order_sha256
    ==
    expected_feature_hash
), (
    "The reconstructed feature-order hash does not match "
    "pipeline_config.yaml."
)


assert feature_manifest[
    "feature_order_sha256"
].nunique() == 1


assert (
    str(
        feature_manifest[
            "feature_order_sha256"
        ]
        .iloc[0]
    )
    ==
    expected_feature_hash
)


expected_family_counts = {
    "S0": 9,
    "S1_diagnosis": 879,
    "S1_medication_name": 812,
    "S1_medication_class": 364,
    "S2_lab_result": 351,
    "S2_imaging_result": 80,
}


observed_family_counts = (
    feature_manifest[
        "family"
    ]
    .value_counts()
    .to_dict()
)


assert (
    observed_family_counts
    ==
    expected_family_counts
), (
    "Feature-family counts differ from the frozen schema.\n"
    f"Observed: {observed_family_counts}"
)



mimic_train_raw = pd.read_parquet(
    NOTEBOOK2_INPUT_PATHS[
        "mimic_train"
    ],
    engine="pyarrow",
)


mimic_validation_raw = pd.read_parquet(
    NOTEBOOK2_INPUT_PATHS[
        "mimic_validation"
    ],
    engine="pyarrow",
)


mimic_test_raw = pd.read_parquet(
    NOTEBOOK2_INPUT_PATHS[
        "mimic_test"
    ],
    engine="pyarrow",
)


frozen_partitions = {
    "MIMIC-IV train": (
        mimic_train_raw
    ),
    "MIMIC-IV validation": (
        mimic_validation_raw
    ),
    "MIMIC-IV test": (
        mimic_test_raw
    ),
}



mimic_split_assignment_columns = [
    "dataset",
    "patient_id",
    "encounter_id",
    "split",
    "outcome",
    "outcome_binary",
    "anchor_year",
    "anchor_year_group",
    "shifted_encounter_year",
    "anchor_to_encounter_year_offset",
    "approx_event_year_start",
    "approx_event_year_end",
    "approx_event_year",
    "approx_event_period",
]


mimic_split_assignments = pd.read_parquet(
    NOTEBOOK2_INPUT_PATHS[
        "split_assignments"
    ],
    columns=(
        mimic_split_assignment_columns
    ),
    filters=[
        (
            "dataset",
            "==",
            "MIMIC-IV",
        )
    ],
    engine="pyarrow",
)


assert len(
    mimic_split_assignments
) == 8550


assert set(
    mimic_split_assignments[
        "dataset"
    ]
    .astype("string")
    .dropna()
    .unique()
) == {
    "MIMIC-IV"
}


assert mimic_split_assignments[
    "patient_id"
].notna().all()


assert mimic_split_assignments[
    "encounter_id"
].notna().all()


assert mimic_split_assignments[
    "patient_id"
].is_unique


assert mimic_split_assignments[
    "encounter_id"
].is_unique



feature_set = set(
    raw_feature_order
)


binary_features = (
    feature_manifest.loc[
        feature_manifest[
            "storage_type"
        ].eq(
            "binary"
        ),
        "feature_name",
    ]
    .tolist()
)


assert len(
    binary_features
) == 2486


required_partition_metadata = {
    "subject_id",
    "stay_id",
    "split",
    "outcome",
    "outcome_binary",
    "anchor_year",
    "anchor_year_group",
    "shifted_encounter_year",
    "anchor_to_encounter_year_offset",
    "approx_event_year_start",
    "approx_event_year_end",
    "approx_event_year",
    "approx_event_period",
}


handoff_validation_records = []


for partition_name, dataframe in (
    frozen_partitions.items()
):

    expected_split = (
        expected_split_labels[
            partition_name
        ]
    )


    assert len(
        dataframe
    ) == expected_rows[
        partition_name
    ], (
        f"{partition_name} row count differs from "
        "pipeline_config.yaml."
    )


    assert not dataframe.columns.duplicated().any()


    missing_metadata = (
        required_partition_metadata
        -
        set(
            dataframe.columns
        )
    )


    if missing_metadata:

        raise ValueError(
            f"{partition_name} is missing metadata columns:\n"
            f"{sorted(missing_metadata)}"
        )


    missing_features = [
        feature_name
        for feature_name
        in raw_feature_order
        if feature_name
        not in dataframe.columns
    ]


    if missing_features:

        raise ValueError(
            f"{partition_name} is missing frozen features:\n"
            f"{missing_features[:20]}"
        )


    observed_feature_order = [
        column_name
        for column_name
        in dataframe.columns
        if column_name
        in feature_set
    ]


    assert (
        observed_feature_order
        ==
        raw_feature_order
    ), (
        f"{partition_name} feature order differs from "
        "the frozen manifest."
    )


    assert dataframe[
        "subject_id"
    ].notna().all()


    assert dataframe[
        "stay_id"
    ].notna().all()


    assert dataframe[
        "subject_id"
    ].is_unique


    assert dataframe[
        "stay_id"
    ].is_unique


    observed_split_values = set(
        dataframe[
            "split"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
        .dropna()
        .unique()
    )


    assert observed_split_values == {
        expected_split
    }


    outcome_values = pd.to_numeric(
        dataframe[
            "outcome_binary"
        ],
        errors="raise",
    ).astype(
        "int8"
    )


    observed_support = (
        outcome_values
        .value_counts()
        .reindex(
            [
                0,
                1,
            ],
            fill_value=0,
        )
        .astype(int)
        .to_dict()
    )


    assert (
        observed_support
        ==
        expected_outcome_support[
            partition_name
        ]
    ), (
        f"{partition_name} outcome support differs from "
        "pipeline_config.yaml.\n"
        f"Observed: {observed_support}\n"
        f"Expected: "
        f"{expected_outcome_support[partition_name]}"
    )


    binary_block = dataframe.loc[
        :,
        binary_features,
    ]


    assert not binary_block.isna().any().any(), (
        f"{partition_name} contains missing binary features."
    )


    assert binary_block.isin(
        [
            0,
            1,
            False,
            True,
        ]
    ).all().all(), (
        f"{partition_name} contains values other than 0/1 "
        "in the frozen binary-feature block."
    )


    approx_event_year = pd.to_numeric(
        dataframe[
            "approx_event_year"
        ],
        errors="raise",
    ).astype(
        int
    )


    assert approx_event_year.notna().all()


    partition_assignments = (
        mimic_split_assignments.loc[
            mimic_split_assignments[
                "split"
            ]
            .astype("string")
            .str.strip()
            .str.lower()
            .eq(
                expected_split
            )
        ]
        .copy()
    )


    assert len(
        partition_assignments
    ) == len(
        dataframe
    )


    parquet_encounter_ids = set(
        dataframe[
            "stay_id"
        ]
        .astype("string")
        .str.strip()
    )


    assignment_encounter_ids = set(
        partition_assignments[
            "encounter_id"
        ]
        .astype("string")
        .str.strip()
    )


    assert (
        parquet_encounter_ids
        ==
        assignment_encounter_ids
    ), (
        f"{partition_name} encounter IDs differ between "
        "the dataset and split_assignments.parquet."
    )


    handoff_validation_records.append(
        {
            "partition": (
                partition_name
            ),
            "split": (
                expected_split
            ),
            "rows": len(
                dataframe
            ),
            "raw_features": len(
                observed_feature_order
            ),
            "metadata_columns": (
                len(
                    dataframe.columns
                )
                -
                len(
                    raw_feature_order
                )
            ),
            "home_support": int(
                observed_support[0]
            ),
            "admitted_support": int(
                observed_support[1]
            ),
            "admitted_prevalence": round(
                observed_support[1]
                /
                len(dataframe),
                6,
            ),
            "approx_event_year_min": int(
                approx_event_year.min()
            ),
            "approx_event_year_max": int(
                approx_event_year.max()
            ),
            "feature_order_sha256": (
                raw_feature_order_sha256
            ),
            "validation_passed": True,
        }
    )


notebook2_handoff_validation = pd.DataFrame(
    handoff_validation_records
)



train_subject_ids = set(
    mimic_train_raw[
        "subject_id"
    ]
)


validation_subject_ids = set(
    mimic_validation_raw[
        "subject_id"
    ]
)


test_subject_ids = set(
    mimic_test_raw[
        "subject_id"
    ]
)


assert train_subject_ids.isdisjoint(
    validation_subject_ids
)


assert train_subject_ids.isdisjoint(
    test_subject_ids
)


assert validation_subject_ids.isdisjoint(
    test_subject_ids
)


train_stay_ids = set(
    mimic_train_raw[
        "stay_id"
    ]
)


validation_stay_ids = set(
    mimic_validation_raw[
        "stay_id"
    ]
)


test_stay_ids = set(
    mimic_test_raw[
        "stay_id"
    ]
)


assert train_stay_ids.isdisjoint(
    validation_stay_ids
)


assert train_stay_ids.isdisjoint(
    test_stay_ids
)


assert validation_stay_ids.isdisjoint(
    test_stay_ids
)


all_subject_ids = pd.concat(
    [
        mimic_train_raw[
            "subject_id"
        ],
        mimic_validation_raw[
            "subject_id"
        ],
        mimic_test_raw[
            "subject_id"
        ],
    ],
    ignore_index=True,
)


all_stay_ids = pd.concat(
    [
        mimic_train_raw[
            "stay_id"
        ],
        mimic_validation_raw[
            "stay_id"
        ],
        mimic_test_raw[
            "stay_id"
        ],
    ],
    ignore_index=True,
)


assert len(
    all_subject_ids
) == 8550


assert all_subject_ids.is_unique


assert all_stay_ids.is_unique



temporal_partition_audit = (
    mimic_split_assignments
    .assign(
        split=(
            mimic_split_assignments[
                "split"
            ]
            .astype("string")
            .str.strip()
            .str.lower()
        ),
        approx_event_year=(
            pd.to_numeric(
                mimic_split_assignments[
                    "approx_event_year"
                ],
                errors="raise",
            )
        ),
    )
    .groupby(
        "split",
        observed=True,
    )[
        "approx_event_year"
    ]
    .agg(
        rows="size",
        approximate_year_min="min",
        approximate_year_max="max",
    )
    .reindex(
        [
            "train",
            "validation",
            "test",
        ]
    )
    .reset_index()
)


assert temporal_partition_audit[
    "rows"
].notna().all()


train_year_max = int(
    temporal_partition_audit.loc[
        temporal_partition_audit[
            "split"
        ].eq(
            "train"
        ),
        "approximate_year_max",
    ].iloc[0]
)


validation_year_min = int(
    temporal_partition_audit.loc[
        temporal_partition_audit[
            "split"
        ].eq(
            "validation"
        ),
        "approximate_year_min",
    ].iloc[0]
)


validation_year_max = int(
    temporal_partition_audit.loc[
        temporal_partition_audit[
            "split"
        ].eq(
            "validation"
        ),
        "approximate_year_max",
    ].iloc[0]
)


test_year_min = int(
    temporal_partition_audit.loc[
        temporal_partition_audit[
            "split"
        ].eq(
            "test"
        ),
        "approximate_year_min",
    ].iloc[0]
)


assert (
    train_year_max
    <
    validation_year_min
), (
    "The corrected training and validation periods overlap."
)


assert (
    validation_year_max
    <
    test_year_min
), (
    "The corrected validation and internal-test periods "
    "overlap."
)



def create_raw_objects(
    dataframe: pd.DataFrame,
):

    indexed = (
        dataframe
        .set_index(
            "stay_id",
            drop=False,
        )
        .copy()
    )


    X = indexed.loc[
        :,
        raw_feature_order,
    ].copy()


    y = pd.to_numeric(
        indexed[
            "outcome_binary"
        ],
        errors="raise",
    ).astype(
        "int8"
    )


    metadata_columns = [
        column_name
        for column_name
        in indexed.columns
        if column_name
        not in feature_set
    ]


    metadata = indexed.loc[
        :,
        metadata_columns,
    ].copy()


    assert X.index.equals(
        y.index
    )


    assert X.index.equals(
        metadata.index
    )


    return X, y, metadata


(
    X_mimic_train_raw,
    y_mimic_train,
    mimic_train_metadata,
) = create_raw_objects(
    mimic_train_raw
)


(
    X_mimic_validation_raw,
    y_mimic_validation,
    mimic_validation_metadata,
) = create_raw_objects(
    mimic_validation_raw
)


(
    X_mimic_test_raw,
    y_mimic_test,
    mimic_test_metadata,
) = create_raw_objects(
    mimic_test_raw
)



s0_raw_feature_order = (
    feature_manifest.loc[
        feature_manifest[
            "family"
        ].eq(
            "S0"
        ),
        "feature_name",
    ]
    .tolist()
)


s1_raw_feature_order = (
    feature_manifest.loc[
        feature_manifest[
            "timing_stage"
        ].eq(
            "S1_HISTORY"
        ),
        "feature_name",
    ]
    .tolist()
)


s2_raw_feature_order = (
    feature_manifest.loc[
        feature_manifest[
            "timing_stage"
        ].eq(
            "S2_OBJECTIVE_WORKUP"
        ),
        "feature_name",
    ]
    .tolist()
)


assert len(
    s0_raw_feature_order
) == 9


assert len(
    s1_raw_feature_order
) == 2055


assert len(
    s2_raw_feature_order
) == 431


assert (
    s0_raw_feature_order
    +
    s1_raw_feature_order
    +
    s2_raw_feature_order
    ==
    raw_feature_order
)



print(
    "Artifact checksum validation:"
)

display(
    artifact_checksum_audit
)


print(
    "\nCorrected Notebook 1 hand-off validation:"
)

display(
    notebook2_handoff_validation
)


print(
    "\nApproximate temporal partition audit:"
)

display(
    temporal_partition_audit
)


print(
    "\nMIMIC temporal split policy:"
)

display(
    pd.DataFrame(
        [
            pipeline_config[
                "mimic_temporal_split_policy"
            ]
        ]
    )
)


print(
    "\nCORRECTED NOTEBOOK 1 HAND-OFF LOADED AND VERIFIED"
)

print(
    "Training rows:",
    len(
        X_mimic_train_raw
    ),
)

print(
    "Validation rows:",
    len(
        X_mimic_validation_raw
    ),
)

print(
    "Internal-test rows:",
    len(
        X_mimic_test_raw
    ),
)

print(
    "Raw features:",
    len(
        raw_feature_order
    ),
)

print(
    "Feature-order SHA-256:",
    raw_feature_order_sha256,
)

print(
    "\nPatient overlap across partitions: NONE"
)

print(
    "Encounter overlap across partitions: NONE"
)

print(
    "Approximate temporal ordering: PASSED"
)

print(
    "MC-MED external file loaded: NO"
)

print(
    "MC-MED checksum verified: YES"
)

print(
    "Preprocessing fitted: NO"
)

print(
    "Model fitted: NO"
)

In [ ]:

required_preprocessing_objects = [
    "feature_manifest",
    "raw_feature_order",
    "X_mimic_train_raw",
    "X_mimic_validation_raw",
    "X_mimic_test_raw",
    "y_mimic_train",
    "y_mimic_validation",
    "y_mimic_test",
    "s0_raw_feature_order",
    "s1_raw_feature_order",
    "s2_raw_feature_order",
]


missing_preprocessing_objects = [
    object_name
    for object_name in required_preprocessing_objects
    if object_name not in globals()
]


if missing_preprocessing_objects:
    raise RuntimeError(
        "Run the complete corrected hand-off loading cell first.\n\n"
        "Missing objects:\n"
        + "\n".join(
            missing_preprocessing_objects
        )
    )


assert len(
    raw_feature_order
) == 2495


assert len(
    set(
        raw_feature_order
    )
) == 2495



s0_numeric_features = (
    feature_manifest.loc[
        feature_manifest[
            "storage_type"
        ].eq(
            "numeric_raw"
        ),
        "feature_name",
    ]
    .tolist()
)


s0_categorical_features = (
    feature_manifest.loc[
        feature_manifest[
            "storage_type"
        ].eq(
            "categorical_raw"
        ),
        "feature_name",
    ]
    .tolist()
)


binary_feature_order = (
    feature_manifest.loc[
        feature_manifest[
            "storage_type"
        ].eq(
            "binary"
        ),
        "feature_name",
    ]
    .tolist()
)


assert len(
    s0_numeric_features
) == 7


assert len(
    s0_categorical_features
) == 2


assert len(
    binary_feature_order
) == 2486


assert (
    len(
        s0_numeric_features
    )
    +
    len(
        s0_categorical_features
    )
    +
    len(
        binary_feature_order
    )
) == 2495


assert set(
    s0_numeric_features
    +
    s0_categorical_features
    +
    binary_feature_order
) == set(
    raw_feature_order
)



def standardise_raw_feature_dtypes(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Apply deterministic dtype cleaning only.

    This function does not estimate medians, category levels or
    any other parameter from the supplied data.
    """

    prepared = dataframe.loc[
        :,
        raw_feature_order,
    ].copy()


    for feature_name in s0_numeric_features:

        prepared[
            feature_name
        ] = pd.to_numeric(
            prepared[
                feature_name
            ],
            errors="coerce",
        )


    for feature_name in s0_categorical_features:

        values = (
            prepared[
                feature_name
            ]
            .astype("string")
            .str.strip()
        )


        values = values.mask(
            values.eq(""),
            pd.NA,
        )


        prepared[
            feature_name
        ] = (
            values
            .astype(object)
            .where(
                values.notna(),
                np.nan,
            )
        )


    prepared.loc[
        :,
        binary_feature_order,
    ] = (
        prepared.loc[
            :,
            binary_feature_order,
        ]
        .astype("int8")
    )


    return prepared


X_mimic_train_standardised = (
    standardise_raw_feature_dtypes(
        X_mimic_train_raw
    )
)


X_mimic_validation_standardised = (
    standardise_raw_feature_dtypes(
        X_mimic_validation_raw
    )
)





MISSING_CATEGORY = "__MISSING__"



def create_model_preprocessor() -> ColumnTransformer:
    """
    Return a new, unfitted preprocessing object.

    RandomizedSearchCV will clone and fit this object separately
    inside every cross-validation fold.
    """

    numeric_pipeline = Pipeline(
        steps=[
            (
                "median_imputer",
                SimpleImputer(
                    strategy="median",
                ),
            ),
        ]
    )


    try:

        categorical_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32,
        )

    except TypeError:

        categorical_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
            dtype=np.float32,
        )


    categorical_pipeline = Pipeline(
        steps=[
            (
                "missing_category_imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value=MISSING_CATEGORY,
                ),
            ),
            (
                "one_hot_encoder",
                categorical_encoder,
            ),
        ]
    )


    return ColumnTransformer(
        transformers=[
            (
                "s0_numeric",
                numeric_pipeline,
                s0_numeric_features,
            ),
            (
                "s0_categorical",
                categorical_pipeline,
                s0_categorical_features,
            ),
            (
                "binary_features",
                "passthrough",
                binary_feature_order,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )


preprocessing_blueprint = (
    create_model_preprocessor()
)


assert not hasattr(
    preprocessing_blueprint,
    "transformers_",
), (
    "The preprocessing blueprint should remain unfitted."
)



raw_preprocessing_audit = pd.DataFrame(
    [
        {
            "partition": "MIMIC-IV train",
            "rows": len(
                X_mimic_train_standardised
            ),
            "raw_features": len(
                raw_feature_order
            ),
            "home": int(
                y_mimic_train.eq(0).sum()
            ),
            "admitted": int(
                y_mimic_train.eq(1).sum()
            ),
            "s0_missing_cells": int(
                X_mimic_train_standardised[
                    s0_numeric_features
                    +
                    s0_categorical_features
                ]
                .isna()
                .sum()
                .sum()
            ),
        },
        {
            "partition": "MIMIC-IV validation",
            "rows": len(
                X_mimic_validation_standardised
            ),
            "raw_features": len(
                raw_feature_order
            ),
            "home": int(
                y_mimic_validation.eq(0).sum()
            ),
            "admitted": int(
                y_mimic_validation.eq(1).sum()
            ),
            "s0_missing_cells": int(
                X_mimic_validation_standardised[
                    s0_numeric_features
                    +
                    s0_categorical_features
                ]
                .isna()
                .sum()
                .sum()
            ),
        },
        {
            "partition": "MIMIC-IV internal test",
            "rows": len(
                X_mimic_test_raw
            ),
            "raw_features": len(
                raw_feature_order
            ),
            "home": int(
                y_mimic_test.eq(0).sum()
            ),
            "admitted": int(
                y_mimic_test.eq(1).sum()
            ),
            "s0_missing_cells": (
                "NOT INSPECTED"
            ),
        },
    ]
)


print(
    "Raw preprocessing-input audit:"
)

display(
    raw_preprocessing_audit
)


print(
    "\nUNFITTED PREPROCESSING BLUEPRINT VERIFIED"
)

print(
    "Raw features:",
    len(
        raw_feature_order
    ),
)

print(
    "Numeric S0 features:",
    len(
        s0_numeric_features
    ),
)

print(
    "Categorical S0 features:",
    len(
        s0_categorical_features
    ),
)

print(
    "Binary S1/S2 features:",
    len(
        binary_feature_order
    ),
)

print(
    "\nPreprocessor fitted: NO"
)

print(
    "Validation transformed: NO"
)

print(
    "Internal test transformed: NO"
)

print(
    "Model fitted: NO"
)

print(
    "MC-MED loaded: NO"
)

## Period-aware training-only cross-validation design

Hyperparameter optimisation is restricted to the approximate 2010–2013 MIMIC-IV training partition.

Five reproducible cross-validation folds are created using a fixed random seed. Stratification jointly considers the HOME/ADMITTED outcome and the approximate encounter year, helping every fold retain representation from each outcome class and each training year.

The validation and internal-test partitions are not used to construct these folds. Preprocessing remains unfitted and will subsequently be fitted independently inside every cross-validation fold as part of each model pipeline.

In [ ]:


required_cv_objects = [
    "X_mimic_train_standardised",
    "y_mimic_train",
    "mimic_train_metadata",
    "create_model_preprocessor",
]


missing_cv_objects = [
    object_name
    for object_name in required_cv_objects
    if object_name not in globals()
]


if missing_cv_objects:

    raise RuntimeError(
        "Run the corrected hand-off and unfitted "
        "preprocessing-blueprint cells first.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_cv_objects
        )
    )


assert X_mimic_train_standardised.index.equals(
    y_mimic_train.index
)


assert X_mimic_train_standardised.index.equals(
    mimic_train_metadata.index
)


assert len(
    X_mimic_train_standardised
) == 3561


assert set(
    y_mimic_train.unique()
) == {
    0,
    1,
}



if (
    "approx_event_year"
    not in mimic_train_metadata.columns
):

    raise ValueError(
        "mimic_train_metadata does not contain "
        "approx_event_year."
    )


training_approx_event_year = pd.to_numeric(
    mimic_train_metadata[
        "approx_event_year"
    ],
    errors="raise",
).astype(
    int
)


assert training_approx_event_year.notna().all()


assert training_approx_event_year.between(
    2008,
    2013,
    inclusive="both",
).all()


observed_training_years = sorted(
    training_approx_event_year.unique().tolist()
)



training_stratification_label = (
    training_approx_event_year
    .astype(str)
    +
    "__outcome_"
    +
    y_mimic_train
    .astype(str)
)


training_stratum_support = (
    pd.DataFrame(
        {
            "approx_event_year": (
                training_approx_event_year
            ),
            "outcome_binary": (
                y_mimic_train
            ),
            "stratification_label": (
                training_stratification_label
            ),
        }
    )
    .groupby(
        [
            "approx_event_year",
            "outcome_binary",
            "stratification_label",
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="rows"
    )
    .sort_values(
        [
            "approx_event_year",
            "outcome_binary",
        ]
    )
    .reset_index(
        drop=True
    )
)


N_CV_SPLITS = 5

CV_RANDOM_SEED = 42


minimum_stratum_support = int(
    training_stratum_support[
        "rows"
    ].min()
)


assert (
    minimum_stratum_support
    >=
    N_CV_SPLITS
), (
    "At least one year-outcome stratum contains fewer rows "
    "than the requested number of CV folds.\n\n"
    f"{training_stratum_support}"
)



period_aware_cv_splitter = StratifiedKFold(
    n_splits=N_CV_SPLITS,
    shuffle=True,
    random_state=CV_RANDOM_SEED,
)


training_cv_splits = list(
    period_aware_cv_splitter.split(
        X_mimic_train_standardised,
        training_stratification_label,
    )
)


assert len(
    training_cv_splits
) == N_CV_SPLITS



cv_fold_records = []

cv_fold_stratum_records = []


training_year_array = (
    training_approx_event_year
    .to_numpy(
        dtype=int
    )
)


training_outcome_array = (
    y_mimic_train
    .to_numpy(
        dtype=np.int8
    )
)


training_stratum_array = (
    training_stratification_label
    .astype(str)
    .to_numpy()
)


all_evaluation_positions = []


for fold_number, (
    fold_training_positions,
    fold_evaluation_positions,
) in enumerate(
    training_cv_splits,
    start=1,
):

    all_evaluation_positions.append(
        fold_evaluation_positions
    )


    fold_training_outcomes = (
        training_outcome_array[
            fold_training_positions
        ]
    )


    fold_evaluation_outcomes = (
        training_outcome_array[
            fold_evaluation_positions
        ]
    )


    fold_training_years = (
        training_year_array[
            fold_training_positions
        ]
    )


    fold_evaluation_years = (
        training_year_array[
            fold_evaluation_positions
        ]
    )


    fold_training_strata = (
        training_stratum_array[
            fold_training_positions
        ]
    )


    fold_evaluation_strata = (
        training_stratum_array[
            fold_evaluation_positions
        ]
    )


    cv_fold_records.append(
        {
            "fold": fold_number,
            "training_rows": len(
                fold_training_positions
            ),
            "training_home": int(
                np.sum(
                    fold_training_outcomes
                    ==
                    0
                )
            ),
            "training_admitted": int(
                np.sum(
                    fold_training_outcomes
                    ==
                    1
                )
            ),
            "training_admitted_prevalence": float(
                np.mean(
                    fold_training_outcomes
                )
            ),
            "evaluation_rows": len(
                fold_evaluation_positions
            ),
            "evaluation_home": int(
                np.sum(
                    fold_evaluation_outcomes
                    ==
                    0
                )
            ),
            "evaluation_admitted": int(
                np.sum(
                    fold_evaluation_outcomes
                    ==
                    1
                )
            ),
            "evaluation_admitted_prevalence": float(
                np.mean(
                    fold_evaluation_outcomes
                )
            ),
            "training_year_min": int(
                fold_training_years.min()
            ),
            "training_year_max": int(
                fold_training_years.max()
            ),
            "evaluation_year_min": int(
                fold_evaluation_years.min()
            ),
            "evaluation_year_max": int(
                fold_evaluation_years.max()
            ),
            "training_strata_represented": int(
                np.unique(
                    fold_training_strata
                ).size
            ),
            "evaluation_strata_represented": int(
                np.unique(
                    fold_evaluation_strata
                ).size
            ),
        }
    )


    fold_evaluation_stratum_counts = (
        pd.Series(
            fold_evaluation_strata
        )
        .value_counts()
        .sort_index()
    )


    for (
        stratum_label,
        stratum_rows,
    ) in (
        fold_evaluation_stratum_counts.items()
    ):

        cv_fold_stratum_records.append(
            {
                "fold": fold_number,
                "stratification_label": (
                    stratum_label
                ),
                "evaluation_rows": int(
                    stratum_rows
                ),
            }
        )


period_aware_cv_fold_audit = pd.DataFrame(
    cv_fold_records
)


period_aware_cv_stratum_audit = pd.DataFrame(
    cv_fold_stratum_records
)



all_evaluation_positions = np.concatenate(
    all_evaluation_positions
)


assert len(
    all_evaluation_positions
) == len(
    X_mimic_train_standardised
)


assert len(
    np.unique(
        all_evaluation_positions
    )
) == len(
    X_mimic_train_standardised
), (
    "A training encounter appears in more than one CV "
    "evaluation fold, or an encounter was omitted."
)


assert period_aware_cv_fold_audit[
    "training_home"
].gt(0).all()


assert period_aware_cv_fold_audit[
    "training_admitted"
].gt(0).all()


assert period_aware_cv_fold_audit[
    "evaluation_home"
].gt(0).all()


assert period_aware_cv_fold_audit[
    "evaluation_admitted"
].gt(0).all()


expected_strata = int(
    training_stratification_label
    .nunique()
)


assert period_aware_cv_fold_audit[
    "training_strata_represented"
].eq(
    expected_strata
).all()


assert period_aware_cv_fold_audit[
    "evaluation_strata_represented"
].eq(
    expected_strata
).all()


assert period_aware_cv_stratum_audit[
    "evaluation_rows"
].gt(0).all()


fresh_preprocessing_check = (
    create_model_preprocessor()
)


assert not hasattr(
    fresh_preprocessing_check,
    "transformers_",
)



print(
    "Training year-outcome strata:"
)

display(
    training_stratum_support
)


print(
    "\nPeriod-aware five-fold CV audit:"
)

display(
    period_aware_cv_fold_audit.round(
        {
            "training_admitted_prevalence": 4,
            "evaluation_admitted_prevalence": 4,
        }
    )
)


print(
    "\nEvaluation support for every stratum and fold:"
)

display(
    period_aware_cv_stratum_audit.pivot(
        index="stratification_label",
        columns="fold",
        values="evaluation_rows",
    )
)


print(
    "\nPERIOD-AWARE TRAINING-ONLY CV FOLDS VERIFIED"
)

print(
    "Training rows:",
    len(
        X_mimic_train_standardised
    ),
)

print(
    "Observed approximate training years:",
    observed_training_years,
)

print(
    "Cross-validation folds:",
    len(
        training_cv_splits
    ),
)

print(
    "Random seed:",
    CV_RANDOM_SEED,
)

print(
    "Stratification variables:",
    "approx_event_year + outcome_binary",
)

print(
    "\nPreprocessor fitted: NO"
)

print(
    "Model fitted: NO"
)

print(
    "Validation used for CV construction: NO"
)

print(
    "Internal test used for CV construction: NO"
)

print(
    "MC-MED loaded: NO"
)

### Training-only randomised hyperparameter optimisation

Random Forest, XGBoost and LightGBM hyperparameters are optimised using the complete S0 + S1 + S2 feature set.

For every sampled hyperparameter configuration, preprocessing and model fitting occur independently inside each of the five period-aware training folds. Numerical imputation and categorical encoding are therefore learned only from the fitting rows of the corresponding fold.

Mean cross-validated AUPRC is the primary optimisation criterion. AUROC, log loss and Brier score are retained as secondary measures. The fixed random seed makes both the sampled hyperparameter configurations and the cross-validation folds reproducible.

The separate 2014–2016 validation partition, 2017–2021 internal-test partition and MC-MED external cohort are not used during hyperparameter optimisation.

In [ ]:



required_tuning_objects = [
    "X_mimic_train_standardised",
    "y_mimic_train",
    "training_cv_splits",
    "create_model_preprocessor",
    "raw_feature_order",
]


missing_tuning_objects = [
    object_name
    for object_name in required_tuning_objects
    if object_name not in globals()
]


if missing_tuning_objects:

    raise RuntimeError(
        "Run the corrected hand-off, preprocessing blueprint "
        "and cross-validation fold cells first.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_tuning_objects
        )
    )


assert len(
    X_mimic_train_standardised
) == 3561


assert X_mimic_train_standardised.shape[
    1
] == 2495


assert X_mimic_train_standardised.index.equals(
    y_mimic_train.index
)


assert len(
    training_cv_splits
) == 5


assert set(
    y_mimic_train.unique()
) == {
    0,
    1,
}



MODEL_RANDOM_SEED = 42

SEARCH_RANDOM_SEED = 42

N_RANDOM_SEARCH_ITERATIONS = 40


AVAILABLE_CPU_THREADS = (
    os.cpu_count()
    or 1
)


SEARCH_N_JOBS = 1

MODEL_N_JOBS = min(
    8,
    AVAILABLE_CPU_THREADS,
)


training_home_support = int(
    y_mimic_train
    .eq(0)
    .sum()
)


training_admitted_support = int(
    y_mimic_train
    .eq(1)
    .sum()
)


training_negative_positive_ratio = float(
    training_home_support
    /
    training_admitted_support
)


tuning_configuration_audit = pd.DataFrame(
    [
        {
            "setting": "Training rows",
            "value": len(
                X_mimic_train_standardised
            ),
        },
        {
            "setting": "Raw features",
            "value": len(
                raw_feature_order
            ),
        },
        {
            "setting": "CV folds",
            "value": len(
                training_cv_splits
            ),
        },
        {
            "setting": "Random configurations per algorithm",
            "value": N_RANDOM_SEARCH_ITERATIONS,
        },
        {
            "setting": "Model random seed",
            "value": MODEL_RANDOM_SEED,
        },
        {
            "setting": "Search random seed",
            "value": SEARCH_RANDOM_SEED,
        },
        {
            "setting": "Threads per fitted model",
            "value": MODEL_N_JOBS,
        },
        {
            "setting": "Training HOME",
            "value": training_home_support,
        },
        {
            "setting": "Training ADMITTED",
            "value": training_admitted_support,
        },
        {
            "setting": "HOME/ADMITTED ratio",
            "value": round(
                training_negative_positive_ratio,
                6,
            ),
        },
        {
            "setting": "Primary tuning metric",
            "value": "Mean CV AUPRC",
        },
    ]
)


print(
    "Hyperparameter-tuning configuration:"
)

display(
    tuning_configuration_audit
)



tuning_scoring = {
    "auprc": "average_precision",
    "auroc": "roc_auc",
    "negative_log_loss": "neg_log_loss",
    "negative_brier": "neg_brier_score",
}


available_scorers = set(
    get_scorer_names()
)


missing_scorers = (
    set(
        tuning_scoring.values()
    )
    -
    available_scorers
)


if missing_scorers:

    raise RuntimeError(
        "The installed scikit-learn version does not provide "
        "the required scorers:\n"
        f"{sorted(missing_scorers)}"
    )



candidate_estimators = {
    "RandomForest": (
        RandomForestClassifier(
            random_state=(
                MODEL_RANDOM_SEED
            ),
            n_jobs=MODEL_N_JOBS,
        )
    ),

    "XGBoost": (
        XGBClassifier(
            objective=(
                "binary:logistic"
            ),
            eval_metric="logloss",
            tree_method="hist",
            random_state=(
                MODEL_RANDOM_SEED
            ),
            n_jobs=MODEL_N_JOBS,
            verbosity=0,
        )
    ),

    "LightGBM": (
        LGBMClassifier(
            objective="binary",
            random_state=(
                MODEL_RANDOM_SEED
            ),
            n_jobs=MODEL_N_JOBS,
            verbosity=-1,
            deterministic=True,
            force_col_wise=True,
            subsample_freq=1,
        )
    ),
}



candidate_parameter_distributions = {

    "RandomForest": {
        "model__n_estimators": randint(
            300,
            1001,
        ),
        "model__criterion": [
            "gini",
            "entropy",
            "log_loss",
        ],
        "model__max_depth": [
            None,
            8,
            12,
            16,
            24,
            32,
        ],
        "model__min_samples_split": randint(
            2,
            21,
        ),
        "model__min_samples_leaf": randint(
            1,
            11,
        ),
        "model__max_features": [
            "sqrt",
            "log2",
            0.10,
            0.20,
            0.30,
            0.50,
        ],
        "model__class_weight": [
            None,
            "balanced",
            "balanced_subsample",
        ],
        "model__bootstrap": [
            True,
        ],
        "model__max_samples": [
            None,
            0.70,
            0.85,
        ],
    },

    "XGBoost": {
        "model__n_estimators": randint(
            200,
            901,
        ),
        "model__max_depth": randint(
            2,
            9,
        ),
        "model__learning_rate": loguniform(
            0.01,
            0.20,
        ),
        "model__min_child_weight": loguniform(
            0.5,
            20.0,
        ),
        "model__subsample": uniform(
            0.60,
            0.40,
        ),
        "model__colsample_bytree": uniform(
            0.50,
            0.50,
        ),
        "model__gamma": [
            0.0,
            0.01,
            0.05,
            0.10,
            0.50,
            1.0,
        ],
        "model__reg_alpha": loguniform(
            1e-4,
            10.0,
        ),
        "model__reg_lambda": loguniform(
            0.10,
            20.0,
        ),
        "model__scale_pos_weight": [
            1.0,
            training_negative_positive_ratio,
        ],
    },

    "LightGBM": {
        "model__n_estimators": randint(
            200,
            901,
        ),
        "model__learning_rate": loguniform(
            0.01,
            0.20,
        ),
        "model__num_leaves": randint(
            15,
            128,
        ),
        "model__max_depth": [
            -1,
            4,
            6,
            8,
            10,
            12,
        ],
        "model__min_child_samples": randint(
            10,
            101,
        ),
        "model__subsample": uniform(
            0.60,
            0.40,
        ),
        "model__colsample_bytree": uniform(
            0.50,
            0.50,
        ),
        "model__reg_alpha": loguniform(
            1e-4,
            10.0,
        ),
        "model__reg_lambda": loguniform(
            0.10,
            20.0,
        ),
        "model__min_split_gain": [
            0.0,
            0.01,
            0.05,
            0.10,
        ],
        "model__class_weight": [
            None,
            "balanced",
        ],
    },
}


assert set(
    candidate_estimators
) == set(
    candidate_parameter_distributions
)



training_cv_searches = {}

training_cv_best_estimators = {}

training_cv_result_tables = []

training_cv_best_summary_records = []


complete_tuning_start_time = (
    time.perf_counter()
)


for algorithm_number, (
    algorithm_name,
    estimator,
) in enumerate(
    candidate_estimators.items(),
    start=1,
):

    print(
        "\n=================================================="
    )

    print(
        f"Algorithm {algorithm_number}/"
        f"{len(candidate_estimators)}:"
    )

    print(
        algorithm_name
    )

    print(
        "=================================================="
    )


    candidate_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                create_model_preprocessor(),
            ),
            (
                "model",
                estimator,
            ),
        ]
    )


    random_search = RandomizedSearchCV(
        estimator=(
            candidate_pipeline
        ),
        param_distributions=(
            candidate_parameter_distributions[
                algorithm_name
            ]
        ),
        n_iter=(
            N_RANDOM_SEARCH_ITERATIONS
        ),
        scoring=(
            tuning_scoring
        ),
        refit="auprc",
        cv=training_cv_splits,
        random_state=(
            SEARCH_RANDOM_SEED
        ),
        n_jobs=SEARCH_N_JOBS,
        verbose=2,
        return_train_score=False,
        error_score="raise",
    )


    algorithm_start_time = (
        time.perf_counter()
    )


    random_search.fit(
        X_mimic_train_standardised,
        y_mimic_train,
    )


    algorithm_elapsed_seconds = (
        time.perf_counter()
        -
        algorithm_start_time
    )


    training_cv_searches[
        algorithm_name
    ] = random_search


    training_cv_best_estimators[
        algorithm_name
    ] = (
        random_search.best_estimator_
    )


    algorithm_results = pd.DataFrame(
        random_search.cv_results_
    )


    algorithm_results.insert(
        0,
        "algorithm",
        algorithm_name,
    )


    algorithm_results.insert(
        1,
        "candidate_number",
        np.arange(
            1,
            len(
                algorithm_results
            )
            +
            1,
        ),
    )


    algorithm_results[
        "mean_cv_log_loss"
    ] = -algorithm_results[
        "mean_test_negative_log_loss"
    ]


    algorithm_results[
        "mean_cv_brier_score"
    ] = -algorithm_results[
        "mean_test_negative_brier"
    ]


    algorithm_results[
        "std_cv_log_loss"
    ] = algorithm_results[
        "std_test_negative_log_loss"
    ]


    algorithm_results[
        "std_cv_brier_score"
    ] = algorithm_results[
        "std_test_negative_brier"
    ]


    training_cv_result_tables.append(
        algorithm_results
    )


    best_result_position = int(
        random_search.best_index_
    )


    best_result = algorithm_results.iloc[
        best_result_position
    ]


    training_cv_best_summary_records.append(
        {
            "algorithm": algorithm_name,
            "best_candidate_number": int(
                best_result[
                    "candidate_number"
                ]
            ),
            "mean_cv_auprc": float(
                best_result[
                    "mean_test_auprc"
                ]
            ),
            "std_cv_auprc": float(
                best_result[
                    "std_test_auprc"
                ]
            ),
            "mean_cv_auroc": float(
                best_result[
                    "mean_test_auroc"
                ]
            ),
            "std_cv_auroc": float(
                best_result[
                    "std_test_auroc"
                ]
            ),
            "mean_cv_log_loss": float(
                best_result[
                    "mean_cv_log_loss"
                ]
            ),
            "mean_cv_brier_score": float(
                best_result[
                    "mean_cv_brier_score"
                ]
            ),
            "search_seconds": float(
                algorithm_elapsed_seconds
            ),
            "best_parameters": (
                random_search.best_params_
            ),
        }
    )


    print(
        "\nBest mean CV AUPRC:",
        round(
            random_search.best_score_,
            6,
        ),
    )

    print(
        "Algorithm elapsed minutes:",
        round(
            algorithm_elapsed_seconds
            /
            60,
            3,
        ),
    )


training_cv_total_seconds = (
    time.perf_counter()
    -
    complete_tuning_start_time
)



training_cv_results = pd.concat(
    training_cv_result_tables,
    ignore_index=True,
)


training_cv_best_summary = pd.DataFrame(
    training_cv_best_summary_records
)


training_cv_best_summary = (
    training_cv_best_summary
    .sort_values(
        [
            "mean_cv_auprc",
            "mean_cv_auroc",
            "mean_cv_log_loss",
            "mean_cv_brier_score",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


training_cv_best_summary.insert(
    0,
    "training_cv_rank",
    np.arange(
        1,
        len(
            training_cv_best_summary
        )
        +
        1,
    ),
)


training_cv_results[
    "overall_training_cv_rank"
] = (
    training_cv_results[
        "mean_test_auprc"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)


assert len(
    training_cv_results
) == (
    N_RANDOM_SEARCH_ITERATIONS
    *
    len(
        candidate_estimators
    )
)


assert len(
    training_cv_searches
) == 3


assert len(
    training_cv_best_estimators
) == 3



tuning_boundary_audit = pd.DataFrame(
    [
        {
            "training_rows_used": len(
                X_mimic_train_standardised
            ),
            "training_cv_folds": len(
                training_cv_splits
            ),
            "algorithms_tuned": len(
                candidate_estimators
            ),
            "random_configurations_per_algorithm": (
                N_RANDOM_SEARCH_ITERATIONS
            ),
            "total_hyperparameter_configurations": (
                len(
                    training_cv_results
                )
            ),
            "total_cv_model_fits": (
                len(
                    training_cv_results
                )
                *
                len(
                    training_cv_splits
                )
            ),
            "validation_rows_used": 0,
            "internal_test_rows_used": 0,
            "mcmed_rows_used": 0,
            "preprocessing_fitted_inside_cv": True,
            "total_elapsed_minutes": round(
                training_cv_total_seconds
                /
                60,
                3,
            ),
        }
    ]
)



print(
    "\nTraining-only hyperparameter-search audit:"
)

display(
    tuning_boundary_audit
)


print(
    "\nBest tuned configuration for each algorithm:"
)

display(
    training_cv_best_summary[
        [
            "training_cv_rank",
            "algorithm",
            "best_candidate_number",
            "mean_cv_auprc",
            "std_cv_auprc",
            "mean_cv_auroc",
            "std_cv_auroc",
            "mean_cv_log_loss",
            "mean_cv_brier_score",
            "search_seconds",
            "best_parameters",
        ]
    ]
)


print(
    "\nTop ten configurations across all algorithms:"
)

display(
    training_cv_results[
        [
            "overall_training_cv_rank",
            "algorithm",
            "candidate_number",
            "mean_test_auprc",
            "std_test_auprc",
            "mean_test_auroc",
            "std_test_auroc",
            "mean_cv_log_loss",
            "mean_cv_brier_score",
            "mean_fit_time",
            "params",
        ]
    ]
    .sort_values(
        [
            "mean_test_auprc",
            "mean_test_auroc",
            "mean_cv_log_loss",
            "mean_cv_brier_score",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
    )
    .head(10)
    .reset_index(
        drop=True
    )
)


print(
    "\nTRAINING-ONLY HYPERPARAMETER OPTIMISATION COMPLETED"
)

print(
    "Algorithms tuned:",
    list(
        training_cv_searches.keys()
    ),
)

print(
    "Configurations evaluated:",
    len(
        training_cv_results
    ),
)

print(
    "Five-fold model fits:",
    (
        len(
            training_cv_results
        )
        *
        len(
            training_cv_splits
        )
    ),
)

print(
    "Best fitted pipelines retained:",
    len(
        training_cv_best_estimators
    ),
)

print(
    "Total elapsed minutes:",
    round(
        training_cv_total_seconds
        /
        60,
        3,
    ),
)

print(
    "\nValidation used: NO"
)

print(
    "Internal test used: NO"
)

print(
    "MC-MED used: NO"
)

print(
    "Final algorithm selected: NO"
)

print(
    "Decision threshold selected: NO"
)

print(
    "Files written: NO"
)

## Validation-period comparison of the tuned algorithms

The best Random Forest, XGBoost and LightGBM pipelines selected through training-only cross-validation are evaluated on the separate approximate 2014–2016 validation period.

Each pipeline already contains preprocessing refitted on the complete 2010–2013 training partition. The validation data are transformed using those training-derived preprocessing parameters without refitting.

Validation AUPRC is the primary algorithm-selection measure. AUROC, log loss and Brier score are retained as secondary deterministic criteria. No classification threshold, calibration model or internal-test prediction is used in this section.

In [ ]:



required_validation_objects = [
    "training_cv_best_estimators",
    "training_cv_best_summary",
    "X_mimic_validation_standardised",
    "y_mimic_validation",
    "mimic_validation_metadata",
]


missing_validation_objects = [
    object_name
    for object_name in required_validation_objects
    if object_name not in globals()
]


if missing_validation_objects:

    raise RuntimeError(
        "Run the training-only hyperparameter-optimisation "
        "section before validation comparison.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_validation_objects
        )
    )


expected_algorithms = {
    "RandomForest",
    "XGBoost",
    "LightGBM",
}


assert set(
    training_cv_best_estimators
) == expected_algorithms


assert len(
    X_mimic_validation_standardised
) == 3138


assert X_mimic_validation_standardised.index.equals(
    y_mimic_validation.index
)


assert X_mimic_validation_standardised.index.equals(
    mimic_validation_metadata.index
)


assert set(
    y_mimic_validation.unique()
) == {
    0,
    1,
}



validation_rows = len(
    y_mimic_validation
)


validation_home_support = int(
    y_mimic_validation
    .eq(0)
    .sum()
)


validation_admitted_support = int(
    y_mimic_validation
    .eq(1)
    .sum()
)


validation_admitted_prevalence = float(
    y_mimic_validation.mean()
)


assert validation_rows == 3138

assert validation_home_support == 2279

assert validation_admitted_support == 859



validation_prediction_table = pd.DataFrame(
    {
        "outcome_binary": (
            y_mimic_validation
            .to_numpy(
                dtype=np.int8
            )
        ),
    },
    index=y_mimic_validation.index,
)


validation_prediction_table.index.name = (
    y_mimic_validation.index.name
)


validation_metric_records = []


for algorithm_name in [
    "RandomForest",
    "XGBoost",
    "LightGBM",
]:

    fitted_pipeline = (
        training_cv_best_estimators[
            algorithm_name
        ]
    )


    validation_probability = (
        fitted_pipeline
        .predict_proba(
            X_mimic_validation_standardised
        )[
            :,
            1,
        ]
    )


    validation_probability = np.asarray(
        validation_probability,
        dtype=float,
    )


    assert validation_probability.ndim == 1

    assert len(
        validation_probability
    ) == validation_rows


    assert np.isfinite(
        validation_probability
    ).all()


    assert (
        (
            validation_probability
            >=
            0
        )
        &
        (
            validation_probability
            <=
            1
        )
    ).all()


    validation_prediction_table[
        f"{algorithm_name}_probability"
    ] = validation_probability.astype(
        np.float32
    )


    validation_auprc = (
        average_precision_score(
            y_mimic_validation,
            validation_probability,
        )
    )


    validation_auroc = (
        roc_auc_score(
            y_mimic_validation,
            validation_probability,
        )
    )


    validation_log_loss = log_loss(
        y_mimic_validation,
        validation_probability,
        labels=[
            0,
            1,
        ],
    )


    validation_brier_score = (
        brier_score_loss(
            y_mimic_validation,
            validation_probability,
        )
    )


    validation_metric_records.append(
        {
            "algorithm": algorithm_name,
            "validation_rows": (
                validation_rows
            ),
            "validation_home": (
                validation_home_support
            ),
            "validation_admitted": (
                validation_admitted_support
            ),
            "validation_admitted_prevalence": (
                validation_admitted_prevalence
            ),
            "validation_auprc": float(
                validation_auprc
            ),
            "validation_auroc": float(
                validation_auroc
            ),
            "validation_log_loss": float(
                validation_log_loss
            ),
            "validation_brier_score": float(
                validation_brier_score
            ),
            "auprc_above_prevalence": float(
                validation_auprc
                -
                validation_admitted_prevalence
            ),
            "auprc_prevalence_ratio": float(
                validation_auprc
                /
                validation_admitted_prevalence
            ),
            "mean_predicted_probability": float(
                validation_probability.mean()
            ),
        }
    )


validation_model_comparison = pd.DataFrame(
    validation_metric_records
)




validation_model_comparison = (
    validation_model_comparison
    .sort_values(
        [
            "validation_auprc",
            "validation_auroc",
            "validation_log_loss",
            "validation_brier_score",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


validation_model_comparison.insert(
    0,
    "validation_rank",
    np.arange(
        1,
        len(
            validation_model_comparison
        )
        +
        1,
    ),
)


validation_selected_algorithm = str(
    validation_model_comparison
    .iloc[0][
        "algorithm"
    ]
)


validation_selected_pipeline = (
    training_cv_best_estimators[
        validation_selected_algorithm
    ]
)


validation_selected_probability = (
    validation_prediction_table[
        f"{validation_selected_algorithm}_probability"
    ]
    .to_numpy(
        dtype=float
    )
)


assert len(
    validation_selected_probability
) == validation_rows





required_validation_results = [
    "training_cv_best_summary",
    "training_cv_best_estimators",
    "validation_model_comparison",
    "validation_prediction_table",
    "validation_selected_algorithm",
    "validation_selected_pipeline",
    "validation_selected_probability",
    "X_mimic_train_standardised",
    "X_mimic_validation_standardised",
    "y_mimic_validation",
]


missing_validation_results = [
    object_name
    for object_name in required_validation_results
    if object_name not in globals()
]


if missing_validation_results:

    raise RuntimeError(
        "The preceding part of the validation-comparison cell "
        "must run successfully first.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_validation_results
        )
    )


expected_algorithms = {
    "RandomForest",
    "XGBoost",
    "LightGBM",
}


assert set(
    training_cv_best_summary[
        "algorithm"
    ]
) == expected_algorithms


assert set(
    validation_model_comparison[
        "algorithm"
    ]
) == expected_algorithms


assert training_cv_best_summary[
    "algorithm"
].is_unique


assert validation_model_comparison[
    "algorithm"
].is_unique



cv_validation_comparison = (
    training_cv_best_summary[
        [
            "algorithm",
            "mean_cv_auprc",
            "std_cv_auprc",
            "mean_cv_auroc",
            "std_cv_auroc",
            "mean_cv_log_loss",
            "mean_cv_brier_score",
            "best_parameters",
        ]
    ]
    .merge(
        validation_model_comparison,
        on="algorithm",
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        "validation_rank"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    cv_validation_comparison
) == 3


assert cv_validation_comparison[
    "algorithm"
].is_unique


assert (
    cv_validation_comparison
    .iloc[0][
        "algorithm"
    ]
    ==
    validation_selected_algorithm
)



validation_rows = len(
    y_mimic_validation
)


validation_admitted_prevalence = float(
    y_mimic_validation.mean()
)


validation_comparison_audit = pd.DataFrame(
    [
        {
            "training_rows_used_for_model_fitting": len(
                X_mimic_train_standardised
            ),
            "validation_rows_evaluated": (
                validation_rows
            ),
            "algorithms_compared": len(
                validation_model_comparison
            ),
            "validation_prevalence": (
                validation_admitted_prevalence
            ),
            "internal_test_rows_used": 0,
            "mcmed_rows_used": 0,
            "threshold_applied": False,
            "calibration_fitted": False,
            "selected_algorithm": (
                validation_selected_algorithm
            ),
        }
    ]
)



print(
    "Validation-period comparison audit:"
)

display(
    validation_comparison_audit
)


print(
    "\nTraining CV and later-period validation comparison:"
)

display(
    cv_validation_comparison[
        [
            "validation_rank",
            "algorithm",
            "mean_cv_auprc",
            "std_cv_auprc",
            "validation_auprc",
            "mean_cv_auroc",
            "std_cv_auroc",
            "validation_auroc",
            "mean_cv_log_loss",
            "validation_log_loss",
            "mean_cv_brier_score",
            "validation_brier_score",
            "mean_predicted_probability",
            "validation_admitted_prevalence",
            "auprc_prevalence_ratio",
        ]
    ]
)


print(
    "\nFull best hyperparameters:"
)


for algorithm_name in [
    "RandomForest",
    "XGBoost",
    "LightGBM",
]:

    parameters = (
        training_cv_best_summary.loc[
            training_cv_best_summary[
                "algorithm"
            ].eq(
                algorithm_name
            ),
            "best_parameters",
        ]
        .iloc[0]
    )


    print(
        f"\n{algorithm_name}:"
    )


    for parameter_name, parameter_value in sorted(
        parameters.items()
    ):

        print(
            f"  {parameter_name}: "
            f"{parameter_value}"
        )


print(
    "\nVALIDATION-PERIOD MODEL COMPARISON COMPLETED"
)

print(
    "Algorithms compared:",
    list(
        validation_model_comparison[
            "algorithm"
        ]
    ),
)

print(
    "Validation-selected algorithm:",
    validation_selected_algorithm,
)

print(
    "Primary selection metric:",
    "Validation AUPRC",
)

print(
    "\nValidation predictions generated: YES"
)

print(
    "Calibration fitted: NO"
)

print(
    "Decision threshold selected: NO"
)

print(
    "Internal test transformed or predicted: NO"
)

print(
    "MC-MED loaded: NO"
)

print(
    "Files written: NO"
)

### Probability calibration and validation-derived threshold locking

The validation-selected XGBoost pipeline is calibrated using Platt scaling. The fitted training model remains unchanged; calibration learns only a monotonic correction between its predicted probabilities and the observed validation outcomes.

Five-fold out-of-fold calibration is used within the validation partition. For each fold, the calibration model is fitted on four validation folds and applied to the remaining fold. These out-of-fold calibrated probabilities are used to choose the classification threshold, avoiding threshold optimisation on probabilities produced by a calibrator fitted to the same rows.

The operating threshold is selected by maximising balanced accuracy, with sensitivity used as the first deterministic tie-breaker. After threshold selection, one final Platt calibrator is fitted using the complete validation partition. The internal-test and MC-MED cohorts remain untouched.

In [ ]:



required_calibration_objects = [
    "validation_selected_algorithm",
    "validation_selected_pipeline",
    "validation_selected_probability",
    "y_mimic_validation",
    "mimic_validation_metadata",
]


missing_calibration_objects = [
    object_name
    for object_name in required_calibration_objects
    if object_name not in globals()
]


if missing_calibration_objects:

    raise RuntimeError(
        "Run the validation-period model-comparison section "
        "before calibration.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_calibration_objects
        )
    )


assert (
    validation_selected_algorithm
    ==
    "XGBoost"
), (
    "The validation-selected algorithm differs from the "
    "expected AUPRC winner."
)


validation_outcome = (
    y_mimic_validation
    .to_numpy(
        dtype=np.int8
    )
)


validation_raw_probability = np.asarray(
    validation_selected_probability,
    dtype=float,
)


assert len(
    validation_raw_probability
) == len(
    validation_outcome
)


assert np.isfinite(
    validation_raw_probability
).all()



PROBABILITY_EPSILON = 1e-6


validation_raw_probability_clipped = np.clip(
    validation_raw_probability,
    PROBABILITY_EPSILON,
    1.0 - PROBABILITY_EPSILON,
)


validation_raw_logit = np.log(
    validation_raw_probability_clipped
    /
    (
        1.0
        -
        validation_raw_probability_clipped
    )
).reshape(
    -1,
    1,
)



CALIBRATION_RANDOM_SEED = 42

N_CALIBRATION_FOLDS = 5


calibration_splitter = StratifiedKFold(
    n_splits=N_CALIBRATION_FOLDS,
    shuffle=True,
    random_state=CALIBRATION_RANDOM_SEED,
)


validation_oof_calibrated_probability = np.full(
    len(
        validation_outcome
    ),
    np.nan,
    dtype=float,
)


calibration_fold_records = []


for fold_number, (
    calibration_fit_positions,
    calibration_evaluation_positions,
) in enumerate(
    calibration_splitter.split(
        validation_raw_logit,
        validation_outcome,
    ),
    start=1,
):

    fold_calibrator = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
        random_state=(
            CALIBRATION_RANDOM_SEED
        ),
    )


    fold_calibrator.fit(
        validation_raw_logit[
            calibration_fit_positions
        ],
        validation_outcome[
            calibration_fit_positions
        ],
    )


    fold_calibrated_probability = (
        fold_calibrator.predict_proba(
            validation_raw_logit[
                calibration_evaluation_positions
            ]
        )[
            :,
            1,
        ]
    )


    validation_oof_calibrated_probability[
        calibration_evaluation_positions
    ] = fold_calibrated_probability


    fold_evaluation_outcome = (
        validation_outcome[
            calibration_evaluation_positions
        ]
    )


    calibration_fold_records.append(
        {
            "fold": fold_number,
            "calibration_fit_rows": len(
                calibration_fit_positions
            ),
            "calibration_evaluation_rows": len(
                calibration_evaluation_positions
            ),
            "evaluation_home": int(
                np.sum(
                    fold_evaluation_outcome
                    ==
                    0
                )
            ),
            "evaluation_admitted": int(
                np.sum(
                    fold_evaluation_outcome
                    ==
                    1
                )
            ),
            "evaluation_prevalence": float(
                np.mean(
                    fold_evaluation_outcome
                )
            ),
            "calibrator_intercept": float(
                fold_calibrator.intercept_[0]
            ),
            "calibrator_slope": float(
                fold_calibrator.coef_[0][0]
            ),
        }
    )


assert np.isfinite(
    validation_oof_calibrated_probability
).all()


assert (
    (
        validation_oof_calibrated_probability
        >=
        0
    )
    &
    (
        validation_oof_calibrated_probability
        <=
        1
    )
).all()


calibration_fold_audit = pd.DataFrame(
    calibration_fold_records
)



def probability_metric_record(
    probability_name,
    probability,
):

    return {
        "probability_type": (
            probability_name
        ),
        "rows": len(
            validation_outcome
        ),
        "observed_prevalence": float(
            validation_outcome.mean()
        ),
        "mean_predicted_probability": float(
            probability.mean()
        ),
        "auprc": float(
            average_precision_score(
                validation_outcome,
                probability,
            )
        ),
        "auroc": float(
            roc_auc_score(
                validation_outcome,
                probability,
            )
        ),
        "log_loss": float(
            log_loss(
                validation_outcome,
                probability,
                labels=[
                    0,
                    1,
                ],
            )
        ),
        "brier_score": float(
            brier_score_loss(
                validation_outcome,
                probability,
            )
        ),
    }


validation_calibration_comparison = pd.DataFrame(
    [
        probability_metric_record(
            "Raw XGBoost probability",
            validation_raw_probability,
        ),
        probability_metric_record(
            "Out-of-fold Platt-calibrated probability",
            validation_oof_calibrated_probability,
        ),
    ]
)



candidate_thresholds = np.linspace(
    0.001,
    0.999,
    999,
)


threshold_records = []


for threshold in candidate_thresholds:

    predicted_class = (
        validation_oof_calibrated_probability
        >=
        threshold
    ).astype(
        np.int8
    )


    tn, fp, fn, tp = confusion_matrix(
        validation_outcome,
        predicted_class,
        labels=[
            0,
            1,
        ],
    ).ravel()


    sensitivity = (
        tp
        /
        (
            tp
            +
            fn
        )
        if (
            tp
            +
            fn
        ) > 0
        else np.nan
    )


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
        if (
            tn
            +
            fp
        ) > 0
        else np.nan
    )


    positive_predictive_value = (
        tp
        /
        (
            tp
            +
            fp
        )
        if (
            tp
            +
            fp
        ) > 0
        else np.nan
    )


    negative_predictive_value = (
        tn
        /
        (
            tn
            +
            fn
        )
        if (
            tn
            +
            fn
        ) > 0
        else np.nan
    )


    f1 = (
        2
        *
        positive_predictive_value
        *
        sensitivity
        /
        (
            positive_predictive_value
            +
            sensitivity
        )
        if (
            positive_predictive_value
            +
            sensitivity
        ) > 0
        else 0.0
    )


    balanced_accuracy = (
        sensitivity
        +
        specificity
    ) / 2.0


    threshold_records.append(
        {
            "threshold": float(
                threshold
            ),
            "true_negative": int(
                tn
            ),
            "false_positive": int(
                fp
            ),
            "false_negative": int(
                fn
            ),
            "true_positive": int(
                tp
            ),
            "sensitivity": float(
                sensitivity
            ),
            "specificity": float(
                specificity
            ),
            "positive_predictive_value": float(
                positive_predictive_value
            ),
            "negative_predictive_value": float(
                negative_predictive_value
            ),
            "f1": float(
                f1
            ),
            "balanced_accuracy": float(
                balanced_accuracy
            ),
        }
    )


validation_threshold_table = pd.DataFrame(
    threshold_records
)



validation_threshold_table = (
    validation_threshold_table
    .sort_values(
        [
            "balanced_accuracy",
            "sensitivity",
            "threshold",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


validation_selected_threshold_metrics = (
    validation_threshold_table
    .iloc[
        [
            0
        ]
    ]
    .copy()
)


locked_decision_threshold = float(
    validation_selected_threshold_metrics[
        "threshold"
    ]
    .iloc[0]
)


assert (
    0
    <
    locked_decision_threshold
    <
    1
)



validation_platt_calibrator = LogisticRegression(
    C=1e6,
    solver="lbfgs",
    max_iter=2000,
    random_state=(
        CALIBRATION_RANDOM_SEED
    ),
)


validation_platt_calibrator.fit(
    validation_raw_logit,
    validation_outcome,
)


final_calibrator_intercept = float(
    validation_platt_calibrator
    .intercept_[
        0
    ]
)


final_calibrator_slope = float(
    validation_platt_calibrator
    .coef_[
        0
    ][
        0
    ]
)



locked_model_package = {
    "algorithm": (
        validation_selected_algorithm
    ),
    "fitted_training_pipeline": (
        validation_selected_pipeline
    ),
    "probability_calibrator": (
        validation_platt_calibrator
    ),
    "calibration_method": (
        "Platt scaling"
    ),
    "calibration_input": (
        "logit of raw model probability"
    ),
    "decision_threshold": (
        locked_decision_threshold
    ),
    "threshold_selection_metric": (
        "Maximum out-of-fold validation balanced accuracy"
    ),
    "model_training_rows": 3561,
    "calibration_rows": len(
        validation_outcome
    ),
    "internal_test_used": False,
    "mcmed_used": False,
}



calibration_and_threshold_audit = pd.DataFrame(
    [
        {
            "selected_algorithm": (
                validation_selected_algorithm
            ),
            "calibration_method": (
                "Platt scaling"
            ),
            "calibration_folds": (
                N_CALIBRATION_FOLDS
            ),
            "validation_rows": len(
                validation_outcome
            ),
            "observed_validation_prevalence": float(
                validation_outcome.mean()
            ),
            "final_calibrator_intercept": (
                final_calibrator_intercept
            ),
            "final_calibrator_slope": (
                final_calibrator_slope
            ),
            "locked_threshold": (
                locked_decision_threshold
            ),
            "threshold_rule": (
                "Maximum out-of-fold balanced accuracy"
            ),
            "internal_test_rows_used": 0,
            "mcmed_rows_used": 0,
        }
    ]
)



print(
    "Calibration-fold audit:"
)

display(
    calibration_fold_audit.round(
        6
    )
)


print(
    "\nRaw versus calibrated validation probabilities:"
)

display(
    validation_calibration_comparison.round(
        6
    )
)


print(
    "\nLocked validation-derived threshold:"
)

display(
    validation_selected_threshold_metrics.round(
        6
    )
)


print(
    "\nCalibration and threshold audit:"
)

display(
    calibration_and_threshold_audit.round(
        6
    )
)


print(
    "\nVALIDATION CALIBRATION AND THRESHOLD LOCKING COMPLETED"
)

print(
    "Selected algorithm:",
    validation_selected_algorithm,
)

print(
    "Calibration method:",
    "Platt scaling",
)

print(
    "Locked threshold:",
    round(
        locked_decision_threshold,
        6,
    ),
)

print(
    "\nThreshold selected from out-of-fold "
    "calibrated validation probabilities: YES"
)

print(
    "Final calibrator fitted on complete validation set: YES"
)

print(
    "Internal test transformed or predicted: NO"
)

print(
    "MC-MED loaded: NO"
)

print(
    "Files written: NO"
)

### Locked internal temporal test evaluation

The locked XGBoost training pipeline, validation-fitted Platt calibrator and validation-derived decision threshold are now applied once to the approximate 2017–2021 internal temporal test period.

No preprocessing, model, calibrator or threshold is refitted. Performance is reported using discrimination, probability accuracy and threshold-dependent classification measures. Patient-level bootstrap resampling provides 95% confidence intervals; because the cohort contains one encounter per patient, encounter-level and patient-level resampling are equivalent.

These results constitute the final internal temporal evaluation and must not be used to revise the model-development decisions.

In [ ]:



required_test_objects = [
    "locked_model_package",
    "validation_selected_algorithm",
    "validation_selected_pipeline",
    "validation_platt_calibrator",
    "locked_decision_threshold",
    "X_mimic_test_raw",
    "y_mimic_test",
    "mimic_test_metadata",
    "standardise_raw_feature_dtypes",
]


missing_test_objects = [
    object_name
    for object_name in required_test_objects
    if object_name not in globals()
]


if missing_test_objects:

    raise RuntimeError(
        "Complete model selection, calibration and threshold "
        "locking before internal-test evaluation.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_test_objects
        )
    )


assert validation_selected_algorithm == "XGBoost"


assert (
    locked_model_package[
        "algorithm"
    ]
    ==
    "XGBoost"
)


assert (
    locked_model_package[
        "decision_threshold"
    ]
    ==
    locked_decision_threshold
)


assert np.isclose(
    locked_model_package[
        "decision_threshold"
    ],
    locked_decision_threshold,
    rtol=0.0,
    atol=1e-12,
), (
    "The threshold stored in locked_model_package differs "
    "from locked_decision_threshold."
)


assert (
    0.0
    <
    locked_decision_threshold
    <
    1.0
)


print(
    "Locked threshold verified:",
    repr(
        locked_decision_threshold
    ),
)


assert len(
    X_mimic_test_raw
) == 1851


assert len(
    y_mimic_test
) == 1851


assert X_mimic_test_raw.index.equals(
    y_mimic_test.index
)


assert X_mimic_test_raw.index.equals(
    mimic_test_metadata.index
)


assert set(
    y_mimic_test.unique()
) == {
    0,
    1,
}




X_mimic_test_standardised = (
    standardise_raw_feature_dtypes(
        X_mimic_test_raw
    )
)


assert X_mimic_test_standardised.shape == (
    1851,
    2495,
)



internal_test_raw_probability = (
    validation_selected_pipeline
    .predict_proba(
        X_mimic_test_standardised
    )[
        :,
        1,
    ]
)


internal_test_raw_probability = np.asarray(
    internal_test_raw_probability,
    dtype=float,
)


assert len(
    internal_test_raw_probability
) == 1851


assert np.isfinite(
    internal_test_raw_probability
).all()


assert (
    (
        internal_test_raw_probability
        >=
        0
    )
    &
    (
        internal_test_raw_probability
        <=
        1
    )
).all()



internal_test_raw_probability_clipped = np.clip(
    internal_test_raw_probability,
    PROBABILITY_EPSILON,
    1.0 - PROBABILITY_EPSILON,
)


internal_test_raw_logit = np.log(
    internal_test_raw_probability_clipped
    /
    (
        1.0
        -
        internal_test_raw_probability_clipped
    )
).reshape(
    -1,
    1,
)


internal_test_calibrated_probability = (
    validation_platt_calibrator
    .predict_proba(
        internal_test_raw_logit
    )[
        :,
        1,
    ]
)


internal_test_calibrated_probability = np.asarray(
    internal_test_calibrated_probability,
    dtype=float,
)


assert len(
    internal_test_calibrated_probability
) == 1851


assert np.isfinite(
    internal_test_calibrated_probability
).all()


assert (
    (
        internal_test_calibrated_probability
        >=
        0
    )
    &
    (
        internal_test_calibrated_probability
        <=
        1
    )
).all()



internal_test_prediction = (
    internal_test_calibrated_probability
    >=
    locked_decision_threshold
).astype(
    np.int8
)


internal_test_outcome = (
    y_mimic_test
    .to_numpy(
        dtype=np.int8
    )
)


assert len(
    internal_test_prediction
) == len(
    internal_test_outcome
)



def calculate_locked_test_metrics(
    y_true,
    raw_probability,
    calibrated_probability,
    prediction,
):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[
            0,
            1,
        ],
    ).ravel()


    sensitivity = (
        tp
        /
        (
            tp
            +
            fn
        )
    )


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
    )


    positive_predictive_value = (
        tp
        /
        (
            tp
            +
            fp
        )
        if (
            tp
            +
            fp
        ) > 0
        else np.nan
    )


    negative_predictive_value = (
        tn
        /
        (
            tn
            +
            fn
        )
        if (
            tn
            +
            fn
        ) > 0
        else np.nan
    )


    return {
        "rows": int(
            len(y_true)
        ),
        "home_support": int(
            np.sum(
                y_true == 0
            )
        ),
        "admitted_support": int(
            np.sum(
                y_true == 1
            )
        ),
        "admitted_prevalence": float(
            np.mean(
                y_true
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                calibrated_probability,
            )
        ),
        "auroc": float(
            roc_auc_score(
                y_true,
                calibrated_probability,
            )
        ),
        "raw_log_loss": float(
            log_loss(
                y_true,
                raw_probability,
                labels=[
                    0,
                    1,
                ],
            )
        ),
        "calibrated_log_loss": float(
            log_loss(
                y_true,
                calibrated_probability,
                labels=[
                    0,
                    1,
                ],
            )
        ),
        "raw_brier_score": float(
            brier_score_loss(
                y_true,
                raw_probability,
            )
        ),
        "calibrated_brier_score": float(
            brier_score_loss(
                y_true,
                calibrated_probability,
            )
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                prediction,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                prediction,
            )
        ),
        "sensitivity": float(
            sensitivity
        ),
        "specificity": float(
            specificity
        ),
        "positive_predictive_value": float(
            positive_predictive_value
        ),
        "negative_predictive_value": float(
            negative_predictive_value
        ),
        "f1": float(
            f1_score(
                y_true,
                prediction,
                zero_division=0,
            )
        ),
        "predicted_admitted_rate": float(
            np.mean(
                prediction
            )
        ),
        "mean_raw_probability": float(
            np.mean(
                raw_probability
            )
        ),
        "mean_calibrated_probability": float(
            np.mean(
                calibrated_probability
            )
        ),
        "true_negative": int(
            tn
        ),
        "false_positive": int(
            fp
        ),
        "false_negative": int(
            fn
        ),
        "true_positive": int(
            tp
        ),
    }


internal_test_point_metrics = (
    calculate_locked_test_metrics(
        y_true=(
            internal_test_outcome
        ),
        raw_probability=(
            internal_test_raw_probability
        ),
        calibrated_probability=(
            internal_test_calibrated_probability
        ),
        prediction=(
            internal_test_prediction
        ),
    )
)



BOOTSTRAP_RANDOM_SEED = 42

N_BOOTSTRAP_REPLICATES = 2000


bootstrap_rng = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)


bootstrap_metric_names = [
    "auprc",
    "auroc",
    "calibrated_log_loss",
    "calibrated_brier_score",
    "accuracy",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "positive_predictive_value",
    "negative_predictive_value",
    "f1",
]


bootstrap_metric_values = {
    metric_name: []
    for metric_name
    in bootstrap_metric_names
}


completed_bootstrap_replicates = 0


for bootstrap_number in range(
    N_BOOTSTRAP_REPLICATES
):

    bootstrap_positions = (
        bootstrap_rng.integers(
            low=0,
            high=len(
                internal_test_outcome
            ),
            size=len(
                internal_test_outcome
            ),
        )
    )


    bootstrap_outcome = (
        internal_test_outcome[
            bootstrap_positions
        ]
    )


    if np.unique(
        bootstrap_outcome
    ).size < 2:

        continue


    bootstrap_raw_probability = (
        internal_test_raw_probability[
            bootstrap_positions
        ]
    )


    bootstrap_calibrated_probability = (
        internal_test_calibrated_probability[
            bootstrap_positions
        ]
    )


    bootstrap_prediction = (
        internal_test_prediction[
            bootstrap_positions
        ]
    )


    bootstrap_metrics = (
        calculate_locked_test_metrics(
            y_true=(
                bootstrap_outcome
            ),
            raw_probability=(
                bootstrap_raw_probability
            ),
            calibrated_probability=(
                bootstrap_calibrated_probability
            ),
            prediction=(
                bootstrap_prediction
            ),
        )
    )


    for metric_name in bootstrap_metric_names:

        metric_value = bootstrap_metrics[
            metric_name
        ]


        if np.isfinite(
            metric_value
        ):

            bootstrap_metric_values[
                metric_name
            ].append(
                metric_value
            )


    completed_bootstrap_replicates += 1


assert (
    completed_bootstrap_replicates
    ==
    N_BOOTSTRAP_REPLICATES
)



internal_test_metric_rows = []


for metric_name in bootstrap_metric_names:

    bootstrap_values = np.asarray(
        bootstrap_metric_values[
            metric_name
        ],
        dtype=float,
    )


    internal_test_metric_rows.append(
        {
            "metric": metric_name,
            "estimate": float(
                internal_test_point_metrics[
                    metric_name
                ]
            ),
            "ci_95_lower": float(
                np.percentile(
                    bootstrap_values,
                    2.5,
                )
            ),
            "ci_95_upper": float(
                np.percentile(
                    bootstrap_values,
                    97.5,
                )
            ),
            "bootstrap_replicates": int(
                len(
                    bootstrap_values
                )
            ),
        }
    )


internal_test_metrics_with_ci = pd.DataFrame(
    internal_test_metric_rows
)



internal_test_predictions = (
    mimic_test_metadata.loc[
        :,
        [
            "subject_id",
            "stay_id",
            "split",
            "outcome",
            "outcome_binary",
            "approx_event_year",
            "approx_event_period",
        ],
    ]
    .copy()
)


internal_test_predictions[
    "raw_probability"
] = (
    internal_test_raw_probability
)


internal_test_predictions[
    "calibrated_probability"
] = (
    internal_test_calibrated_probability
)


internal_test_predictions[
    "locked_threshold"
] = (
    locked_decision_threshold
)


internal_test_predictions[
    "predicted_outcome_binary"
] = (
    internal_test_prediction
)


internal_test_predictions[
    "correct_prediction"
] = (
    internal_test_predictions[
        "outcome_binary"
    ]
    .astype(int)
    .eq(
        internal_test_predictions[
            "predicted_outcome_binary"
        ]
    )
)


assert len(
    internal_test_predictions
) == 1851



internal_test_evaluation_audit = pd.DataFrame(
    [
        {
            "algorithm": (
                validation_selected_algorithm
            ),
            "calibration_method": (
                "Platt scaling"
            ),
            "locked_threshold": (
                locked_decision_threshold
            ),
            "test_rows": len(
                internal_test_outcome
            ),
            "test_home": int(
                np.sum(
                    internal_test_outcome
                    ==
                    0
                )
            ),
            "test_admitted": int(
                np.sum(
                    internal_test_outcome
                    ==
                    1
                )
            ),
            "test_prevalence": float(
                np.mean(
                    internal_test_outcome
                )
            ),
            "bootstrap_replicates": (
                completed_bootstrap_replicates
            ),
            "model_refitted": False,
            "preprocessor_refitted": False,
            "calibrator_refitted": False,
            "threshold_changed": False,
            "mcmed_used": False,
        }
    ]
)



print(
    "Locked internal temporal test audit:"
)

display(
    internal_test_evaluation_audit
)


print(
    "\nInternal temporal test metrics with "
    "patient-level 95% bootstrap confidence intervals:"
)

display(
    internal_test_metrics_with_ci.round(
        6
    )
)


print(
    "\nInternal temporal test probability summary:"
)

display(
    pd.DataFrame(
        [
            {
                "observed_prevalence": (
                    internal_test_point_metrics[
                        "admitted_prevalence"
                    ]
                ),
                "mean_raw_probability": (
                    internal_test_point_metrics[
                        "mean_raw_probability"
                    ]
                ),
                "mean_calibrated_probability": (
                    internal_test_point_metrics[
                        "mean_calibrated_probability"
                    ]
                ),
                "predicted_admitted_rate": (
                    internal_test_point_metrics[
                        "predicted_admitted_rate"
                    ]
                ),
            }
        ]
    ).round(
        6
    )
)


print(
    "\nInternal temporal test confusion matrix:"
)

display(
    pd.DataFrame(
        [
            {
                "true_negative": (
                    internal_test_point_metrics[
                        "true_negative"
                    ]
                ),
                "false_positive": (
                    internal_test_point_metrics[
                        "false_positive"
                    ]
                ),
                "false_negative": (
                    internal_test_point_metrics[
                        "false_negative"
                    ]
                ),
                "true_positive": (
                    internal_test_point_metrics[
                        "true_positive"
                    ]
                ),
            }
        ]
    )
)


print(
    "\nLOCKED INTERNAL TEMPORAL TEST EVALUATION COMPLETED"
)

print(
    "Test evaluated exactly once: YES"
)

print(
    "Model refitted: NO"
)

print(
    "Preprocessor refitted: NO"
)

print(
    "Calibrator refitted: NO"
)

print(
    "Threshold changed: NO"
)

print(
    "MC-MED loaded: NO"
)

print(
    "Files written: NO"
)

print(
    "\nThe internal-test results must not be used to revise "
    "model development decisions."
)

## Notebook 2 locked-model and results hand-off

The final internal model-development decisions are now frozen. This section saves the training-fitted XGBoost pipeline, validation-fitted Platt calibrator, validation-derived decision threshold, model-selection results, internal temporal-test results and patient-level predictions.

The saved model package will be used unchanged for MC-MED external validation. Saving these objects does not refit the model, recalibrate probabilities, alter the threshold or reevaluate the internal test set.

In [ ]:



required_notebook2_objects = [
    "PROJECT_ROOT",
    "pipeline_config",
    "feature_manifest",
    "raw_feature_order",
    "raw_feature_order_sha256",
    "training_cv_results",
    "training_cv_best_summary",
    "training_cv_best_estimators",
    "tuning_boundary_audit",
    "period_aware_cv_fold_audit",
    "period_aware_cv_stratum_audit",
    "validation_model_comparison",
    "cv_validation_comparison",
    "validation_prediction_table",
    "validation_calibration_comparison",
    "calibration_fold_audit",
    "calibration_and_threshold_audit",
    "validation_threshold_table",
    "validation_selected_threshold_metrics",
    "validation_selected_algorithm",
    "validation_selected_pipeline",
    "validation_platt_calibrator",
    "locked_decision_threshold",
    "locked_model_package",
    "internal_test_point_metrics",
    "internal_test_metrics_with_ci",
    "internal_test_evaluation_audit",
    "internal_test_predictions",
]


missing_notebook2_objects = [
    object_name
    for object_name in required_notebook2_objects
    if object_name not in globals()
]


if missing_notebook2_objects:

    raise RuntimeError(
        "Complete all Notebook 2 modelling sections before "
        "saving the locked hand-off.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_notebook2_objects
        )
    )


assert validation_selected_algorithm == "XGBoost"


assert (
    locked_model_package[
        "algorithm"
    ]
    ==
    validation_selected_algorithm
)


assert np.isclose(
    float(
        locked_model_package[
            "decision_threshold"
        ]
    ),
    float(
        locked_decision_threshold
    ),
    rtol=0.0,
    atol=1e-12,
)


assert (
    0.0
    <
    float(
        locked_decision_threshold
    )
    <
    1.0
)


assert len(
    raw_feature_order
) == 2495


assert len(
    internal_test_predictions
) == 1851


assert len(
    internal_test_metrics_with_ci
) == 11



NOTEBOOK2_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "notebook2_locked"
)


NOTEBOOK2_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook2"
)


NOTEBOOK2_PREDICTION_DIR = (
    PROJECT_ROOT
    / "results"
    / "predictions"
    / "notebook2"
)


for output_directory in [
    NOTEBOOK2_MODEL_DIR,
    NOTEBOOK2_TABLE_DIR,
    NOTEBOOK2_PREDICTION_DIR,
]:

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )



def make_json_safe(
    value,
):

    if isinstance(
        value,
        dict,
    ):

        return {
            str(key): make_json_safe(
                item
            )
            for key, item
            in value.items()
        }


    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):

        return [
            make_json_safe(
                item
            )
            for item
            in value
        ]


    if isinstance(
        value,
        np.ndarray,
    ):

        return [
            make_json_safe(
                item
            )
            for item
            in value.tolist()
        ]


    if isinstance(
        value,
        np.generic,
    ):

        return value.item()


    if isinstance(
        value,
        Path,
    ):

        return str(
            value
        )


    if pd.isna(
        value
    ):

        return None


    return value



selected_hyperparameters = (
    training_cv_best_summary.loc[
        training_cv_best_summary[
            "algorithm"
        ].eq(
            validation_selected_algorithm
        ),
        "best_parameters",
    ]
    .iloc[0]
)


selected_training_cv_result = (
    training_cv_best_summary.loc[
        training_cv_best_summary[
            "algorithm"
        ].eq(
            validation_selected_algorithm
        )
    ]
    .iloc[0]
)


selected_validation_result = (
    validation_model_comparison.loc[
        validation_model_comparison[
            "algorithm"
        ].eq(
            validation_selected_algorithm
        )
    ]
    .iloc[0]
)



internal_test_probability_comparison = pd.DataFrame(
    [
        {
            "probability_type": (
                "Raw XGBoost probability"
            ),
            "rows": int(
                internal_test_point_metrics[
                    "rows"
                ]
            ),
            "observed_prevalence": float(
                internal_test_point_metrics[
                    "admitted_prevalence"
                ]
            ),
            "mean_predicted_probability": float(
                internal_test_point_metrics[
                    "mean_raw_probability"
                ]
            ),
            "auprc": float(
                average_precision_score(
                    internal_test_predictions[
                        "outcome_binary"
                    ],
                    internal_test_predictions[
                        "raw_probability"
                    ],
                )
            ),
            "auroc": float(
                roc_auc_score(
                    internal_test_predictions[
                        "outcome_binary"
                    ],
                    internal_test_predictions[
                        "raw_probability"
                    ],
                )
            ),
            "log_loss": float(
                internal_test_point_metrics[
                    "raw_log_loss"
                ]
            ),
            "brier_score": float(
                internal_test_point_metrics[
                    "raw_brier_score"
                ]
            ),
        },
        {
            "probability_type": (
                "Platt-calibrated probability"
            ),
            "rows": int(
                internal_test_point_metrics[
                    "rows"
                ]
            ),
            "observed_prevalence": float(
                internal_test_point_metrics[
                    "admitted_prevalence"
                ]
            ),
            "mean_predicted_probability": float(
                internal_test_point_metrics[
                    "mean_calibrated_probability"
                ]
            ),
            "auprc": float(
                internal_test_point_metrics[
                    "auprc"
                ]
            ),
            "auroc": float(
                internal_test_point_metrics[
                    "auroc"
                ]
            ),
            "log_loss": float(
                internal_test_point_metrics[
                    "calibrated_log_loss"
                ]
            ),
            "brier_score": float(
                internal_test_point_metrics[
                    "calibrated_brier_score"
                ]
            ),
        },
    ]
)


internal_test_probability_summary = pd.DataFrame(
    [
        {
            "observed_prevalence": float(
                internal_test_point_metrics[
                    "admitted_prevalence"
                ]
            ),
            "mean_raw_probability": float(
                internal_test_point_metrics[
                    "mean_raw_probability"
                ]
            ),
            "mean_calibrated_probability": float(
                internal_test_point_metrics[
                    "mean_calibrated_probability"
                ]
            ),
            "predicted_admitted_rate": float(
                internal_test_point_metrics[
                    "predicted_admitted_rate"
                ]
            ),
        }
    ]
)


internal_test_confusion_matrix = pd.DataFrame(
    [
        {
            "true_negative": int(
                internal_test_point_metrics[
                    "true_negative"
                ]
            ),
            "false_positive": int(
                internal_test_point_metrics[
                    "false_positive"
                ]
            ),
            "false_negative": int(
                internal_test_point_metrics[
                    "false_negative"
                ]
            ),
            "true_positive": int(
                internal_test_point_metrics[
                    "true_positive"
                ]
            ),
        }
    ]
)



training_cv_results_to_save = (
    training_cv_results.copy()
)


if (
    "params"
    in training_cv_results_to_save.columns
):

    training_cv_results_to_save[
        "params"
    ] = training_cv_results_to_save[
        "params"
    ].map(
        lambda value: json.dumps(
            make_json_safe(
                value
            ),
            sort_keys=True,
        )
    )


training_cv_best_summary_to_save = (
    training_cv_best_summary.copy()
)


if (
    "best_parameters"
    in training_cv_best_summary_to_save.columns
):

    training_cv_best_summary_to_save[
        "best_parameters"
    ] = training_cv_best_summary_to_save[
        "best_parameters"
    ].map(
        lambda value: json.dumps(
            make_json_safe(
                value
            ),
            sort_keys=True,
        )
    )


cv_validation_comparison_to_save = (
    cv_validation_comparison.copy()
)


if (
    "best_parameters"
    in cv_validation_comparison_to_save.columns
):

    cv_validation_comparison_to_save[
        "best_parameters"
    ] = cv_validation_comparison_to_save[
        "best_parameters"
    ].map(
        lambda value: json.dumps(
            make_json_safe(
                value
            ),
            sort_keys=True,
        )
    )



locked_model_package_to_save = {
    "package_name": (
        "MIMIC-IV ESI-3 abdominal pain "
        "HOME versus ADMITTED"
    ),
    "package_stage": (
        "Notebook 2 internal model locked"
    ),
    "algorithm": (
        validation_selected_algorithm
    ),
    "fitted_training_pipeline": (
        validation_selected_pipeline
    ),
    "probability_calibrator": (
        validation_platt_calibrator
    ),
    "calibration_method": (
        "Platt scaling"
    ),
    "calibration_input": (
        "Logit of raw XGBoost probability"
    ),
    "decision_threshold": float(
        locked_decision_threshold
    ),
    "threshold_selection_rule": (
        "Maximum out-of-fold validation balanced accuracy; "
        "higher sensitivity then lower threshold used as "
        "deterministic tie-breakers"
    ),
    "raw_feature_order": list(
        raw_feature_order
    ),
    "raw_feature_count": int(
        len(
            raw_feature_order
        )
    ),
    "feature_order_sha256": (
        raw_feature_order_sha256
    ),
    "selected_hyperparameters": (
        selected_hyperparameters
    ),
    "training_rows": 3561,
    "validation_rows": 3138,
    "internal_test_rows": 1851,
    "training_period_observed": (
        "2010-2013 approximate encounter years"
    ),
    "validation_period_observed": (
        "2014-2016 approximate encounter years"
    ),
    "internal_test_period_observed": (
        "2017-2021 approximate encounter years"
    ),
    "outcome_mapping": {
        "HOME": 0,
        "ADMITTED": 1,
    },
    "dtype_standardisation_required": True,
    "internal_test_evaluated": True,
    "model_development_locked": True,
    "external_data_used": False,
}



notebook2_model_config = {
    "pipeline_stage": (
        "02_internal_modeling_complete"
    ),
    "saved_at_utc": (
        datetime
        .now(
            timezone.utc
        )
        .isoformat()
    ),
    "project_root": str(
        PROJECT_ROOT
    ),
    "algorithm": (
        validation_selected_algorithm
    ),
    "selected_hyperparameters": (
        make_json_safe(
            selected_hyperparameters
        )
    ),
    "raw_feature_count": int(
        len(
            raw_feature_order
        )
    ),
    "feature_order_sha256": (
        raw_feature_order_sha256
    ),
    "cohort_sizes": {
        "mimic_train": 3561,
        "mimic_validation": 3138,
        "mimic_internal_test": 1851,
    },
    "outcome_support": {
        "mimic_train": {
            "home": 2369,
            "admitted": 1192,
        },
        "mimic_validation": {
            "home": 2279,
            "admitted": 859,
        },
        "mimic_internal_test": {
            "home": 1606,
            "admitted": 245,
        },
    },
    "temporal_design": {
        "method": (
            "Approximate encounter-year temporal split"
        ),
        "training_observed_years": [
            2010,
            2011,
            2012,
            2013,
        ],
        "validation_observed_years": [
            2014,
            2015,
            2016,
        ],
        "internal_test_observed_years": [
            2017,
            2018,
            2019,
            2020,
            2021,
        ],
        "exact_original_dates_available": False,
        "split_is_approximate": True,
    },
    "training_cv": {
        "folds": int(
            len(
                training_cv_splits
            )
        ),
        "stratification": (
            "Approximate encounter year plus outcome"
        ),
        "random_seed": int(
            CV_RANDOM_SEED
        ),
    },
    "hyperparameter_search": {
        "method": (
            "RandomizedSearchCV"
        ),
        "algorithms": [
            "RandomForest",
            "XGBoost",
            "LightGBM",
        ],
        "iterations_per_algorithm": int(
            N_RANDOM_SEARCH_ITERATIONS
        ),
        "total_configurations": int(
            len(
                training_cv_results
            )
        ),
        "total_cv_fits": int(
            len(
                training_cv_results
            )
            *
            len(
                training_cv_splits
            )
        ),
        "primary_metric": (
            "Mean training-CV AUPRC"
        ),
        "model_random_seed": int(
            MODEL_RANDOM_SEED
        ),
        "search_random_seed": int(
            SEARCH_RANDOM_SEED
        ),
    },
    "algorithm_selection": {
        "selection_partition": (
            "MIMIC-IV validation"
        ),
        "primary_metric": (
            "Validation AUPRC"
        ),
        "selected_algorithm": (
            validation_selected_algorithm
        ),
        "training_cv_auprc": float(
            selected_training_cv_result[
                "mean_cv_auprc"
            ]
        ),
        "validation_auprc": float(
            selected_validation_result[
                "validation_auprc"
            ]
        ),
        "validation_auroc": float(
            selected_validation_result[
                "validation_auroc"
            ]
        ),
    },
    "calibration": {
        "method": (
            "Platt scaling"
        ),
        "folds": int(
            N_CALIBRATION_FOLDS
        ),
        "random_seed": int(
            CALIBRATION_RANDOM_SEED
        ),
        "final_intercept": float(
            validation_platt_calibrator
            .intercept_[
                0
            ]
        ),
        "final_slope": float(
            validation_platt_calibrator
            .coef_[
                0
            ][
                0
            ]
        ),
    },
    "threshold": {
        "value": float(
            locked_decision_threshold
        ),
        "selection_rule": (
            "Maximum out-of-fold validation "
            "balanced accuracy"
        ),
    },
    "internal_test": {
        "evaluated_once": True,
        "bootstrap_replicates": int(
            N_BOOTSTRAP_REPLICATES
        ),
        "bootstrap_random_seed": int(
            BOOTSTRAP_RANDOM_SEED
        ),
        "point_metrics": (
            make_json_safe(
                internal_test_point_metrics
            )
        ),
    },
    "software": {
        "python": (
            platform.python_version()
        ),
        "pandas": (
            pd.__version__
        ),
        "numpy": (
            np.__version__
        ),
        "scikit_learn": (
            sklearn.__version__
        ),
        "xgboost": (
            xgb.__version__
        ),
        "lightgbm": (
            lgb.__version__
        ),
        "joblib": (
            joblib.__version__
        ),
    },
    "development_boundaries": {
        "internal_test_used_for_model_selection": False,
        "internal_test_used_for_calibration": False,
        "internal_test_used_for_threshold_selection": False,
        "mcmed_used": False,
        "model_development_locked": True,
    },
}



notebook2_output_paths = {
    "locked_xgboost_pipeline": (
        NOTEBOOK2_MODEL_DIR
        / "locked_xgboost_pipeline.joblib"
    ),
    "platt_calibrator": (
        NOTEBOOK2_MODEL_DIR
        / "platt_calibrator.joblib"
    ),
    "locked_model_package": (
        NOTEBOOK2_MODEL_DIR
        / "locked_model_package.joblib"
    ),
    "model_config": (
        NOTEBOOK2_MODEL_DIR
        / "notebook2_model_config.json"
    ),

    "training_cv_results": (
        NOTEBOOK2_TABLE_DIR
        / "training_cv_results.csv"
    ),
    "training_cv_best_summary": (
        NOTEBOOK2_TABLE_DIR
        / "training_cv_best_summary.csv"
    ),
    "training_cv_fold_audit": (
        NOTEBOOK2_TABLE_DIR
        / "training_cv_fold_audit.csv"
    ),
    "training_cv_stratum_audit": (
        NOTEBOOK2_TABLE_DIR
        / "training_cv_stratum_audit.csv"
    ),
    "tuning_boundary_audit": (
        NOTEBOOK2_TABLE_DIR
        / "tuning_boundary_audit.csv"
    ),
    "validation_model_comparison": (
        NOTEBOOK2_TABLE_DIR
        / "validation_model_comparison.csv"
    ),
    "cv_validation_comparison": (
        NOTEBOOK2_TABLE_DIR
        / "cv_validation_comparison.csv"
    ),

    "validation_predictions": (
        NOTEBOOK2_PREDICTION_DIR
        / "validation_predictions.parquet"
    ),
    "validation_calibration_comparison": (
        NOTEBOOK2_TABLE_DIR
        / "validation_calibration_comparison.csv"
    ),
    "calibration_fold_audit": (
        NOTEBOOK2_TABLE_DIR
        / "calibration_fold_audit.csv"
    ),
    "calibration_threshold_audit": (
        NOTEBOOK2_TABLE_DIR
        / "calibration_threshold_audit.csv"
    ),
    "validation_threshold_table": (
        NOTEBOOK2_TABLE_DIR
        / "validation_threshold_table.csv"
    ),
    "validation_selected_threshold": (
        NOTEBOOK2_TABLE_DIR
        / "validation_selected_threshold.csv"
    ),

    "internal_test_metrics_with_ci": (
        NOTEBOOK2_TABLE_DIR
        / "internal_test_metrics_with_ci.csv"
    ),
    "internal_test_probability_comparison": (
        NOTEBOOK2_TABLE_DIR
        / "internal_test_probability_comparison.csv"
    ),
    "internal_test_probability_summary": (
        NOTEBOOK2_TABLE_DIR
        / "internal_test_probability_summary.csv"
    ),
    "internal_test_confusion_matrix": (
        NOTEBOOK2_TABLE_DIR
        / "internal_test_confusion_matrix.csv"
    ),
    "internal_test_evaluation_audit": (
        NOTEBOOK2_TABLE_DIR
        / "internal_test_evaluation_audit.csv"
    ),
    "internal_test_predictions": (
        NOTEBOOK2_PREDICTION_DIR
        / "internal_test_predictions.parquet"
    ),

    "artifact_manifest": (
        NOTEBOOK2_MODEL_DIR
        / "notebook2_artifact_manifest.csv"
    ),
}



joblib.dump(
    validation_selected_pipeline,
    notebook2_output_paths[
        "locked_xgboost_pipeline"
    ],
    compress=3,
)


joblib.dump(
    validation_platt_calibrator,
    notebook2_output_paths[
        "platt_calibrator"
    ],
    compress=3,
)


joblib.dump(
    locked_model_package_to_save,
    notebook2_output_paths[
        "locked_model_package"
    ],
    compress=3,
)


with notebook2_output_paths[
    "model_config"
].open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        make_json_safe(
            notebook2_model_config
        ),
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )



training_cv_results_to_save.to_csv(
    notebook2_output_paths[
        "training_cv_results"
    ],
    index=False,
)


training_cv_best_summary_to_save.to_csv(
    notebook2_output_paths[
        "training_cv_best_summary"
    ],
    index=False,
)


period_aware_cv_fold_audit.to_csv(
    notebook2_output_paths[
        "training_cv_fold_audit"
    ],
    index=False,
)


period_aware_cv_stratum_audit.to_csv(
    notebook2_output_paths[
        "training_cv_stratum_audit"
    ],
    index=False,
)


tuning_boundary_audit.to_csv(
    notebook2_output_paths[
        "tuning_boundary_audit"
    ],
    index=False,
)


validation_model_comparison.to_csv(
    notebook2_output_paths[
        "validation_model_comparison"
    ],
    index=False,
)


cv_validation_comparison_to_save.to_csv(
    notebook2_output_paths[
        "cv_validation_comparison"
    ],
    index=False,
)


validation_prediction_table.to_parquet(
    notebook2_output_paths[
        "validation_predictions"
    ],
    index=True,
    engine="pyarrow",
)


validation_calibration_comparison.to_csv(
    notebook2_output_paths[
        "validation_calibration_comparison"
    ],
    index=False,
)


calibration_fold_audit.to_csv(
    notebook2_output_paths[
        "calibration_fold_audit"
    ],
    index=False,
)


calibration_and_threshold_audit.to_csv(
    notebook2_output_paths[
        "calibration_threshold_audit"
    ],
    index=False,
)


validation_threshold_table.to_csv(
    notebook2_output_paths[
        "validation_threshold_table"
    ],
    index=False,
)


validation_selected_threshold_metrics.to_csv(
    notebook2_output_paths[
        "validation_selected_threshold"
    ],
    index=False,
)



internal_test_metrics_with_ci.to_csv(
    notebook2_output_paths[
        "internal_test_metrics_with_ci"
    ],
    index=False,
)


internal_test_probability_comparison.to_csv(
    notebook2_output_paths[
        "internal_test_probability_comparison"
    ],
    index=False,
)


internal_test_probability_summary.to_csv(
    notebook2_output_paths[
        "internal_test_probability_summary"
    ],
    index=False,
)


internal_test_confusion_matrix.to_csv(
    notebook2_output_paths[
        "internal_test_confusion_matrix"
    ],
    index=False,
)


internal_test_evaluation_audit.to_csv(
    notebook2_output_paths[
        "internal_test_evaluation_audit"
    ],
    index=False,
)


internal_test_predictions.to_parquet(
    notebook2_output_paths[
        "internal_test_predictions"
    ],
    index=True,
    engine="pyarrow",
)



reloaded_pipeline = joblib.load(
    notebook2_output_paths[
        "locked_xgboost_pipeline"
    ]
)


reloaded_calibrator = joblib.load(
    notebook2_output_paths[
        "platt_calibrator"
    ]
)


reloaded_package = joblib.load(
    notebook2_output_paths[
        "locked_model_package"
    ]
)


assert hasattr(
    reloaded_pipeline,
    "predict_proba",
)


assert hasattr(
    reloaded_calibrator,
    "predict_proba",
)


assert (
    reloaded_package[
        "algorithm"
    ]
    ==
    "XGBoost"
)


assert np.isclose(
    float(
        reloaded_package[
            "decision_threshold"
        ]
    ),
    float(
        locked_decision_threshold
    ),
    rtol=0.0,
    atol=1e-12,
)


assert (
    reloaded_package[
        "feature_order_sha256"
    ]
    ==
    raw_feature_order_sha256
)


reloaded_test_predictions = pd.read_parquet(
    notebook2_output_paths[
        "internal_test_predictions"
    ],
    engine="pyarrow",
)


assert len(
    reloaded_test_predictions
) == 1851


assert {
    "raw_probability",
    "calibrated_probability",
    "predicted_outcome_binary",
}.issubset(
    reloaded_test_predictions.columns
)


del reloaded_pipeline
del reloaded_calibrator
del reloaded_package
del reloaded_test_predictions



def notebook2_file_sha256(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()


artifact_manifest_records = []


for artifact_name, artifact_path in (
    notebook2_output_paths.items()
):

    if artifact_name == "artifact_manifest":
        continue


    if not artifact_path.exists():

        raise FileNotFoundError(
            "Expected Notebook 2 artifact was not written:\n"
            f"{artifact_path}"
        )


    artifact_manifest_records.append(
        {
            "artifact": artifact_name,
            "filename": (
                artifact_path.name
            ),
            "directory": str(
                artifact_path.parent
            ),
            "size_bytes": int(
                artifact_path
                .stat()
                .st_size
            ),
            "sha256": (
                notebook2_file_sha256(
                    artifact_path
                )
            ),
        }
    )


notebook2_artifact_manifest = pd.DataFrame(
    artifact_manifest_records
)


notebook2_artifact_manifest.to_csv(
    notebook2_output_paths[
        "artifact_manifest"
    ],
    index=False,
)



notebook2_saved_artifact_audit = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "exists": (
                artifact_path.exists()
            ),
            "size_mb": round(
                artifact_path
                .stat()
                .st_size
                /
                (
                    1024
                    **
                    2
                ),
                4,
            ),
            "path": str(
                artifact_path
            ),
        }
        for artifact_name, artifact_path
        in notebook2_output_paths.items()
    ]
)


assert notebook2_saved_artifact_audit[
    "exists"
].all()


print(
    "Notebook 2 saved-artifact audit:"
)

display(
    notebook2_saved_artifact_audit
)


print(
    "\nNotebook 2 SHA-256 artifact manifest:"
)

display(
    notebook2_artifact_manifest
)


print(
    "\nNOTEBOOK 2 LOCKED MODEL AND RESULTS SAVED"
)

print(
    "Selected algorithm:",
    validation_selected_algorithm,
)

print(
    "Calibration:",
    "Platt scaling",
)

print(
    "Locked threshold:",
    round(
        float(
            locked_decision_threshold
        ),
        6,
    ),
)

print(
    "Raw feature count:",
    len(
        raw_feature_order
    ),
)

print(
    "Feature-order SHA-256:",
    raw_feature_order_sha256,
)

print(
    "\nModel pipeline saved: YES"
)

print(
    "Calibrator saved: YES"
)

print(
    "Locked model package saved: YES"
)

print(
    "Reproducibility configuration saved: YES"
)

print(
    "Result tables saved: YES"
)

print(
    "Patient-level predictions saved: YES"
)

print(
    "Artifact checksums saved: YES"
)

print(
    "Figures saved: NO — figures are the next section"
)

print(
    "Model refitted during saving: NO"
)

print(
    "Threshold changed during saving: NO"
)

print(
    "MC-MED loaded: NO"
)

### Locked-model performance figures

This section visualises the prespecified validation and locked internal temporal-test results. All figures use already-generated predictions and do not refit preprocessing, models, calibration or thresholds.

Validation figures document algorithm selection, calibration and threshold choice. Internal-test figures report the single locked evaluation of discrimination, calibration and classification performance.

Figures are saved as 600-dpi PNG files and vector PDF files. Internal-test results are presented descriptively and are not used to revise any model-development decision.

In [ ]:



required_figure_objects = [
    "PROJECT_ROOT",
    "validation_model_comparison",
    "validation_outcome",
    "validation_raw_probability",
    "validation_oof_calibrated_probability",
    "validation_threshold_table",
    "validation_selected_threshold_metrics",
    "locked_decision_threshold",
    "internal_test_predictions",
    "internal_test_point_metrics",
]


missing_figure_objects = [
    object_name
    for object_name in required_figure_objects
    if object_name not in globals()
]


if missing_figure_objects:

    raise RuntimeError(
        "Run all validation, calibration and locked "
        "internal-test sections before creating figures.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_figure_objects
        )
    )


assert len(
    validation_outcome
) == 3138


assert len(
    validation_raw_probability
) == 3138


assert len(
    validation_oof_calibrated_probability
) == 3138


assert len(
    internal_test_predictions
) == 1851


assert np.isclose(
    float(
        locked_decision_threshold
    ),
    float(
        validation_selected_threshold_metrics[
            "threshold"
        ].iloc[0]
    ),
    rtol=0.0,
    atol=1e-12,
)



NOTEBOOK2_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "notebook2"
)


NOTEBOOK2_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FIGURE_DPI = 600



saved_figure_records = []


def save_notebook2_figure(
    figure,
    figure_name,
):

    png_path = (
        NOTEBOOK2_FIGURE_DIR
        /
        f"{figure_name}.png"
    )


    pdf_path = (
        NOTEBOOK2_FIGURE_DIR
        /
        f"{figure_name}.pdf"
    )


    figure.savefig(
        png_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )


    figure.savefig(
        pdf_path,
        bbox_inches="tight",
    )


    saved_figure_records.extend(
        [
            {
                "figure": figure_name,
                "format": "PNG",
                "dpi": FIGURE_DPI,
                "path": str(
                    png_path
                ),
            },
            {
                "figure": figure_name,
                "format": "PDF",
                "dpi": (
                    "vector"
                ),
                "path": str(
                    pdf_path
                ),
            },
        ]
    )


    plt.show()


    plt.close(
        figure
    )



validation_calibration_predictions = pd.DataFrame(
    {
        "outcome_binary": (
            validation_outcome
        ),
        "raw_xgboost_probability": (
            validation_raw_probability
        ),
        "oof_platt_calibrated_probability": (
            validation_oof_calibrated_probability
        ),
    },
    index=y_mimic_validation.index,
)


validation_calibration_predictions.index.name = (
    y_mimic_validation.index.name
)


validation_calibration_prediction_path = (
    PROJECT_ROOT
    / "results"
    / "predictions"
    / "notebook2"
    / "validation_calibration_predictions.parquet"
)


validation_calibration_predictions.to_parquet(
    validation_calibration_prediction_path,
    index=True,
    engine="pyarrow",
)



validation_plot_data = (
    validation_model_comparison
    .sort_values(
        "validation_auprc",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


validation_algorithms = (
    validation_plot_data[
        "algorithm"
    ]
    .tolist()
)


validation_positions = np.arange(
    len(
        validation_algorithms
    )
)


validation_bar_width = 0.36


figure, axis = plt.subplots(
    figsize=(
        9,
        6,
    )
)


auprc_bars = axis.bar(
    validation_positions
    -
    validation_bar_width
    /
    2,
    validation_plot_data[
        "validation_auprc"
    ],
    width=validation_bar_width,
    label="AUPRC",
)


auroc_bars = axis.bar(
    validation_positions
    +
    validation_bar_width
    /
    2,
    validation_plot_data[
        "validation_auroc"
    ],
    width=validation_bar_width,
    label="AUROC",
)


axis.axhline(
    y=float(
        validation_plot_data[
            "validation_admitted_prevalence"
        ].iloc[0]
    ),
    linestyle="--",
    linewidth=1.2,
    label=(
        "Validation prevalence "
        f"({validation_plot_data['validation_admitted_prevalence'].iloc[0]:.3f})"
    ),
)


axis.set_xticks(
    validation_positions
)


axis.set_xticklabels(
    validation_algorithms
)


axis.set_ylim(
    0.0,
    1.0,
)


axis.set_ylabel(
    "Performance"
)


axis.set_xlabel(
    "Algorithm"
)


axis.set_title(
    "Later-period validation performance of tuned algorithms"
)


axis.legend(
    frameon=False
)


axis.bar_label(
    auprc_bars,
    fmt="%.3f",
    padding=3,
)


axis.bar_label(
    auroc_bars,
    fmt="%.3f",
    padding=3,
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


save_notebook2_figure(
    figure,
    "01_validation_algorithm_comparison",
)



validation_raw_fraction_positive, (
    validation_raw_mean_probability
) = calibration_curve(
    validation_outcome,
    validation_raw_probability,
    n_bins=10,
    strategy="quantile",
)


validation_calibrated_fraction_positive, (
    validation_calibrated_mean_probability
) = calibration_curve(
    validation_outcome,
    validation_oof_calibrated_probability,
    n_bins=10,
    strategy="quantile",
)


figure, axis = plt.subplots(
    figsize=(
        7,
        7,
    )
)


axis.plot(
    [
        0,
        1,
    ],
    [
        0,
        1,
    ],
    linestyle="--",
    linewidth=1.2,
    label="Perfect calibration",
)


axis.plot(
    validation_raw_mean_probability,
    validation_raw_fraction_positive,
    marker="o",
    linewidth=1.8,
    label="Raw XGBoost",
)


axis.plot(
    validation_calibrated_mean_probability,
    validation_calibrated_fraction_positive,
    marker="o",
    linewidth=1.8,
    label="OOF Platt calibrated",
)


axis.set_xlim(
    0.0,
    1.0,
)


axis.set_ylim(
    0.0,
    1.0,
)


axis.set_xlabel(
    "Mean predicted admission probability"
)


axis.set_ylabel(
    "Observed admission proportion"
)


axis.set_title(
    "Validation reliability curve"
)


axis.legend(
    frameon=False
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


save_notebook2_figure(
    figure,
    "02_validation_reliability_curve",
)



test_outcome_for_figures = (
    pd.to_numeric(
        internal_test_predictions[
            "outcome_binary"
        ],
        errors="raise",
    )
    .astype(
        "int8"
    )
    .to_numpy()
)


test_raw_probability_for_figures = (
    pd.to_numeric(
        internal_test_predictions[
            "raw_probability"
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=float
    )
)


test_calibrated_probability_for_figures = (
    pd.to_numeric(
        internal_test_predictions[
            "calibrated_probability"
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=float
    )
)


test_prediction_for_figures = (
    pd.to_numeric(
        internal_test_predictions[
            "predicted_outcome_binary"
        ],
        errors="raise",
    )
    .astype(
        "int8"
    )
    .to_numpy()
)


assert len(
    test_outcome_for_figures
) == 1851



test_false_positive_rate, (
    test_true_positive_rate
), _ = roc_curve(
    test_outcome_for_figures,
    test_calibrated_probability_for_figures,
)


test_auroc_for_figure = roc_auc_score(
    test_outcome_for_figures,
    test_calibrated_probability_for_figures,
)


figure, axis = plt.subplots(
    figsize=(
        7,
        7,
    )
)


axis.plot(
    test_false_positive_rate,
    test_true_positive_rate,
    linewidth=2.0,
    label=(
        f"Locked XGBoost "
        f"(AUROC = {test_auroc_for_figure:.3f})"
    ),
)


axis.plot(
    [
        0,
        1,
    ],
    [
        0,
        1,
    ],
    linestyle="--",
    linewidth=1.2,
    label="No-discrimination reference",
)


axis.set_xlim(
    0.0,
    1.0,
)


axis.set_ylim(
    0.0,
    1.0,
)


axis.set_xlabel(
    "False-positive rate"
)


axis.set_ylabel(
    "True-positive rate"
)


axis.set_title(
    "Locked internal temporal-test ROC curve"
)


axis.legend(
    frameon=False,
    loc="lower right",
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


save_notebook2_figure(
    figure,
    "03_internal_test_roc_curve",
)



test_precision, (
    test_recall
), _ = precision_recall_curve(
    test_outcome_for_figures,
    test_calibrated_probability_for_figures,
)


test_auprc_for_figure = (
    average_precision_score(
        test_outcome_for_figures,
        test_calibrated_probability_for_figures,
    )
)


test_prevalence_for_figure = float(
    np.mean(
        test_outcome_for_figures
    )
)


figure, axis = plt.subplots(
    figsize=(
        7,
        7,
    )
)


axis.plot(
    test_recall,
    test_precision,
    linewidth=2.0,
    label=(
        f"Locked XGBoost "
        f"(AUPRC = {test_auprc_for_figure:.3f})"
    ),
)


axis.axhline(
    y=test_prevalence_for_figure,
    linestyle="--",
    linewidth=1.2,
    label=(
        "No-skill prevalence "
        f"({test_prevalence_for_figure:.3f})"
    ),
)


axis.set_xlim(
    0.0,
    1.0,
)


axis.set_ylim(
    0.0,
    1.0,
)


axis.set_xlabel(
    "Recall / sensitivity"
)


axis.set_ylabel(
    "Precision / positive predictive value"
)


axis.set_title(
    "Locked internal temporal-test precision–recall curve"
)


axis.legend(
    frameon=False,
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


save_notebook2_figure(
    figure,
    "04_internal_test_precision_recall_curve",
)



test_raw_fraction_positive, (
    test_raw_mean_probability
) = calibration_curve(
    test_outcome_for_figures,
    test_raw_probability_for_figures,
    n_bins=10,
    strategy="quantile",
)


test_calibrated_fraction_positive, (
    test_calibrated_mean_probability
) = calibration_curve(
    test_outcome_for_figures,
    test_calibrated_probability_for_figures,
    n_bins=10,
    strategy="quantile",
)


figure, axis = plt.subplots(
    figsize=(
        7,
        7,
    )
)


axis.plot(
    [
        0,
        1,
    ],
    [
        0,
        1,
    ],
    linestyle="--",
    linewidth=1.2,
    label="Perfect calibration",
)


axis.plot(
    test_raw_mean_probability,
    test_raw_fraction_positive,
    marker="o",
    linewidth=1.8,
    label="Raw XGBoost",
)


axis.plot(
    test_calibrated_mean_probability,
    test_calibrated_fraction_positive,
    marker="o",
    linewidth=1.8,
    label="Platt calibrated",
)


axis.set_xlim(
    0.0,
    1.0,
)


axis.set_ylim(
    0.0,
    1.0,
)


axis.set_xlabel(
    "Mean predicted admission probability"
)


axis.set_ylabel(
    "Observed admission proportion"
)


axis.set_title(
    "Locked internal temporal-test reliability curve"
)


axis.legend(
    frameon=False,
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


save_notebook2_figure(
    figure,
    "05_internal_test_reliability_curve",
)



test_confusion_matrix_for_figure = (
    confusion_matrix(
        test_outcome_for_figures,
        test_prediction_for_figures,
        labels=[
            0,
            1,
        ],
    )
)


figure, axis = plt.subplots(
    figsize=(
        7,
        6,
    )
)


matrix_image = axis.imshow(
    test_confusion_matrix_for_figure,
)


axis.set_xticks(
    [
        0,
        1,
    ]
)


axis.set_yticks(
    [
        0,
        1,
    ]
)


axis.set_xticklabels(
    [
        "Predicted HOME",
        "Predicted ADMITTED",
    ]
)


axis.set_yticklabels(
    [
        "Actual HOME",
        "Actual ADMITTED",
    ]
)


for row_position in range(
    test_confusion_matrix_for_figure.shape[0]
):

    for column_position in range(
        test_confusion_matrix_for_figure.shape[1]
    ):

        axis.text(
            column_position,
            row_position,
            (
                test_confusion_matrix_for_figure[
                    row_position,
                    column_position
                ]
            ),
            ha="center",
            va="center",
            fontsize=14,
        )


axis.set_title(
    "Locked internal temporal-test confusion matrix"
)


figure.colorbar(
    matrix_image,
    ax=axis,
    label="Number of encounters",
)


figure.tight_layout()


save_notebook2_figure(
    figure,
    "06_internal_test_confusion_matrix",
)



probability_transport_table = pd.DataFrame(
    [
        {
            "partition": (
                "Validation\n2014–2016"
            ),
            "observed_prevalence": float(
                np.mean(
                    validation_outcome
                )
            ),
            "mean_raw_probability": float(
                np.mean(
                    validation_raw_probability
                )
            ),
            "mean_calibrated_probability": float(
                np.mean(
                    validation_oof_calibrated_probability
                )
            ),
        },
        {
            "partition": (
                "Internal test\n2017–2021"
            ),
            "observed_prevalence": float(
                np.mean(
                    test_outcome_for_figures
                )
            ),
            "mean_raw_probability": float(
                np.mean(
                    test_raw_probability_for_figures
                )
            ),
            "mean_calibrated_probability": float(
                np.mean(
                    test_calibrated_probability_for_figures
                )
            ),
        },
    ]
)


transport_positions = np.arange(
    len(
        probability_transport_table
    )
)


transport_bar_width = 0.24


figure, axis = plt.subplots(
    figsize=(
        9,
        6,
    )
)


observed_bars = axis.bar(
    transport_positions
    -
    transport_bar_width,
    probability_transport_table[
        "observed_prevalence"
    ],
    width=transport_bar_width,
    label="Observed prevalence",
)


raw_bars = axis.bar(
    transport_positions,
    probability_transport_table[
        "mean_raw_probability"
    ],
    width=transport_bar_width,
    label="Mean raw probability",
)


calibrated_bars = axis.bar(
    transport_positions
    +
    transport_bar_width,
    probability_transport_table[
        "mean_calibrated_probability"
    ],
    width=transport_bar_width,
    label="Mean calibrated probability",
)


axis.set_xticks(
    transport_positions
)


axis.set_xticklabels(
    probability_transport_table[
        "partition"
    ]
)


axis.set_ylim(
    0.0,
    0.50,
)


axis.set_ylabel(
    "Proportion / mean probability"
)


axis.set_xlabel(
    "Evaluation period"
)


axis.set_title(
    "Probability calibration across later temporal periods"
)


axis.legend(
    frameon=False,
)


axis.bar_label(
    observed_bars,
    fmt="%.3f",
    padding=3,
)


axis.bar_label(
    raw_bars,
    fmt="%.3f",
    padding=3,
)


axis.bar_label(
    calibrated_bars,
    fmt="%.3f",
    padding=3,
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


save_notebook2_figure(
    figure,
    "07_temporal_probability_transport",
)



threshold_plot_data = (
    validation_threshold_table
    .sort_values(
        "threshold"
    )
    .reset_index(
        drop=True
    )
)


figure, axis = plt.subplots(
    figsize=(
        9,
        6,
    )
)


axis.plot(
    threshold_plot_data[
        "threshold"
    ],
    threshold_plot_data[
        "sensitivity"
    ],
    linewidth=1.8,
    label="Sensitivity",
)


axis.plot(
    threshold_plot_data[
        "threshold"
    ],
    threshold_plot_data[
        "specificity"
    ],
    linewidth=1.8,
    label="Specificity",
)


axis.plot(
    threshold_plot_data[
        "threshold"
    ],
    threshold_plot_data[
        "balanced_accuracy"
    ],
    linewidth=1.8,
    label="Balanced accuracy",
)


axis.axvline(
    x=float(
        locked_decision_threshold
    ),
    linestyle="--",
    linewidth=1.3,
    label=(
        "Locked threshold "
        f"({locked_decision_threshold:.3f})"
    ),
)


axis.set_xlim(
    0.0,
    1.0,
)


axis.set_ylim(
    0.0,
    1.0,
)


axis.set_xlabel(
    "Calibrated probability threshold"
)


axis.set_ylabel(
    "Performance"
)


axis.set_title(
    "Validation threshold trade-off"
)


axis.legend(
    frameon=False,
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


save_notebook2_figure(
    figure,
    "08_validation_threshold_tradeoff",
)



notebook2_figure_manifest = pd.DataFrame(
    saved_figure_records
)


def figure_file_sha256(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )


    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()


notebook2_figure_manifest[
    "exists"
] = (
    notebook2_figure_manifest[
        "path"
    ]
    .map(
        lambda value: Path(
            value
        ).exists()
    )
)


notebook2_figure_manifest[
    "size_bytes"
] = (
    notebook2_figure_manifest[
        "path"
    ]
    .map(
        lambda value: int(
            Path(
                value
            )
            .stat()
            .st_size
        )
    )
)


notebook2_figure_manifest[
    "sha256"
] = (
    notebook2_figure_manifest[
        "path"
    ]
    .map(
        figure_file_sha256
    )
)


assert notebook2_figure_manifest[
    "exists"
].all()


notebook2_figure_manifest_path = (
    NOTEBOOK2_FIGURE_DIR
    / "notebook2_figure_manifest.csv"
)


notebook2_figure_manifest.to_csv(
    notebook2_figure_manifest_path,
    index=False,
)



print(
    "Notebook 2 figure manifest:"
)

display(
    notebook2_figure_manifest[
        [
            "figure",
            "format",
            "dpi",
            "size_bytes",
            "sha256",
            "path",
        ]
    ]
)


print(
    "\nNOTEBOOK 2 CORE PERFORMANCE FIGURES SAVED"
)

print(
    "Unique figures:",
    notebook2_figure_manifest[
        "figure"
    ].nunique(),
)

print(
    "Figure files:",
    len(
        notebook2_figure_manifest
    ),
)

print(
    "PNG resolution:",
    f"{FIGURE_DPI} dpi",
)

print(
    "Vector PDFs saved: YES"
)

print(
    "Validation calibration predictions saved: YES"
)

print(
    "Figure manifest and checksums saved: YES"
)

print(
    "\nModel refitted: NO"
)

print(
    "Calibrator refitted: NO"
)

print(
    "Threshold changed: NO"
)

print(
    "Internal test reevaluated for development: NO"
)

print(
    "MC-MED loaded: NO"
)

## Incremental feature-stage analysis

A secondary incremental analysis evaluates the predictive value added across the clinical information pathway: S0 triage information, S0 plus S1 previous diagnoses and medications, and S0 plus S1 plus S2 current laboratory and imaging findings.

The validation-selected XGBoost algorithm and its previously tuned hyperparameters are held constant across all feature stages. Preprocessing is fitted independently within each training cross-validation fold, and every stage uses the same five training-only folds. This design isolates the contribution of additional information rather than differences in algorithms or hyperparameter optimisation.

Performance is assessed using threshold-independent discrimination and probability metrics. Stage-specific calibration and thresholds are not fitted. The locked Platt calibrator and threshold of 0.282 remain exclusive to the final S0 + S1 + S2 model. The internal temporal-test and MC-MED cohorts are not used in this secondary analysis.

In [ ]:



required_incremental_objects = [
    "PROJECT_ROOT",
    "X_mimic_train_standardised",
    "y_mimic_train",
    "X_mimic_validation_standardised",
    "y_mimic_validation",
    "training_cv_splits",
    "validation_selected_algorithm",
    "validation_selected_pipeline",
    "validation_model_comparison",
    "s0_raw_feature_order",
    "s1_raw_feature_order",
    "s2_raw_feature_order",
    "s0_numeric_features",
    "s0_categorical_features",
    "binary_feature_order",
    "raw_feature_order",
    "raw_feature_order_sha256",
    "MISSING_CATEGORY",
]


missing_incremental_objects = [
    object_name
    for object_name in required_incremental_objects
    if object_name not in globals()
]


if missing_incremental_objects:

    raise RuntimeError(
        "Run the hand-off, preprocessing, tuning and "
        "validation-selection sections first.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_incremental_objects
        )
    )


assert validation_selected_algorithm == "XGBoost"


assert len(
    X_mimic_train_standardised
) == 3561


assert len(
    X_mimic_validation_standardised
) == 3138


assert X_mimic_train_standardised.index.equals(
    y_mimic_train.index
)


assert X_mimic_validation_standardised.index.equals(
    y_mimic_validation.index
)


assert len(
    training_cv_splits
) == 5



incremental_stage_feature_orders = {
    "S0": list(
        s0_raw_feature_order
    ),
    "S0_S1": (
        list(
            s0_raw_feature_order
        )
        +
        list(
            s1_raw_feature_order
        )
    ),
    "S0_S1_S2": list(
        raw_feature_order
    ),
}


incremental_stage_display_names = {
    "S0": "S0",
    "S0_S1": "S0 + S1",
    "S0_S1_S2": "S0 + S1 + S2",
}


incremental_stage_descriptions = {
    "S0": (
        "Triage demographics, arrival transport "
        "and initial vital signs"
    ),
    "S0_S1": (
        "S0 plus previous diagnoses and medications"
    ),
    "S0_S1_S2": (
        "S0 plus S1 plus current abnormal laboratory "
        "and imaging-result features"
    ),
}


expected_stage_feature_counts = {
    "S0": 9,
    "S0_S1": 2064,
    "S0_S1_S2": 2495,
}


for stage_name, stage_features in (
    incremental_stage_feature_orders.items()
):

    assert len(
        stage_features
    ) == expected_stage_feature_counts[
        stage_name
    ]


    assert len(
        set(
            stage_features
        )
    ) == len(
        stage_features
    )


    assert set(
        stage_features
    ).issubset(
        set(
            raw_feature_order
        )
    )


assert (
    incremental_stage_feature_orders[
        "S0_S1_S2"
    ]
    ==
    raw_feature_order
)



def create_incremental_stage_preprocessor(
    stage_features,
):
    """
    Return a new unfitted preprocessor containing only the
    features available at the requested clinical stage.
    """

    stage_feature_set = set(
        stage_features
    )


    stage_numeric_features = [
        feature_name
        for feature_name
        in s0_numeric_features
        if feature_name
        in stage_feature_set
    ]


    stage_categorical_features = [
        feature_name
        for feature_name
        in s0_categorical_features
        if feature_name
        in stage_feature_set
    ]


    stage_binary_features = [
        feature_name
        for feature_name
        in binary_feature_order
        if feature_name
        in stage_feature_set
    ]


    transformers = []


    if stage_numeric_features:

        numeric_pipeline = Pipeline(
            steps=[
                (
                    "median_imputer",
                    SimpleImputer(
                        strategy="median",
                    ),
                ),
            ]
        )


        transformers.append(
            (
                "s0_numeric",
                numeric_pipeline,
                stage_numeric_features,
            )
        )


    if stage_categorical_features:

        try:

            categorical_encoder = OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
                dtype=np.float32,
            )

        except TypeError:

            categorical_encoder = OneHotEncoder(
                handle_unknown="ignore",
                sparse=True,
                dtype=np.float32,
            )


        categorical_pipeline = Pipeline(
            steps=[
                (
                    "missing_category_imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value=MISSING_CATEGORY,
                    ),
                ),
                (
                    "one_hot_encoder",
                    categorical_encoder,
                ),
            ]
        )


        transformers.append(
            (
                "s0_categorical",
                categorical_pipeline,
                stage_categorical_features,
            )
        )


    if stage_binary_features:

        transformers.append(
            (
                "binary_features",
                "passthrough",
                stage_binary_features,
            )
        )


    stage_preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=1.0,
    )


    assert not hasattr(
        stage_preprocessor,
        "transformers_",
    )


    return stage_preprocessor



selected_xgboost_model_template = clone(
    validation_selected_pipeline
    .named_steps[
        "model"
    ]
)


assert isinstance(
    selected_xgboost_model_template,
    XGBClassifier,
)


selected_xgboost_parameters = (
    selected_xgboost_model_template
    .get_params(
        deep=False
    )
)



incremental_stage_cv_fold_records = []

incremental_stage_summary_records = []

incremental_stage_fitted_pipelines = {}


incremental_stage_validation_predictions = pd.DataFrame(
    {
        "outcome_binary": (
            y_mimic_validation
            .to_numpy(
                dtype=np.int8
            )
        ),
    },
    index=y_mimic_validation.index,
)


incremental_stage_validation_predictions.index.name = (
    y_mimic_validation.index.name
)


for stage_number, (
    stage_name,
    stage_features,
) in enumerate(
    incremental_stage_feature_orders.items(),
    start=1,
):

    stage_display_name = (
        incremental_stage_display_names[
            stage_name
        ]
    )


    print(
        "\n=================================================="
    )

    print(
        f"Incremental stage {stage_number}/"
        f"{len(incremental_stage_feature_orders)}:"
    )

    print(
        stage_display_name
    )

    print(
        "Raw features:",
        len(
            stage_features
        ),
    )

    print(
        "=================================================="
    )


    X_stage_train = (
        X_mimic_train_standardised.loc[
            :,
            stage_features,
        ]
    )


    X_stage_validation = (
        X_mimic_validation_standardised.loc[
            :,
            stage_features,
        ]
    )


    assert X_stage_train.shape == (
        3561,
        len(
            stage_features
        ),
    )


    assert X_stage_validation.shape == (
        3138,
        len(
            stage_features
        ),
    )


    stage_oof_probability = np.full(
        len(
            X_stage_train
        ),
        np.nan,
        dtype=float,
    )


    stage_cv_start_time = (
        time.perf_counter()
    )


    for fold_number, (
        fold_fit_positions,
        fold_evaluation_positions,
    ) in enumerate(
        training_cv_splits,
        start=1,
    ):

        fold_pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    create_incremental_stage_preprocessor(
                        stage_features
                    ),
                ),
                (
                    "model",
                    clone(
                        selected_xgboost_model_template
                    ),
                ),
            ]
        )


        fold_pipeline.fit(
            X_stage_train.iloc[
                fold_fit_positions
            ],
            y_mimic_train.iloc[
                fold_fit_positions
            ],
        )


        fold_probability = (
            fold_pipeline
            .predict_proba(
                X_stage_train.iloc[
                    fold_evaluation_positions
                ]
            )[
                :,
                1,
            ]
        )


        fold_probability = np.asarray(
            fold_probability,
            dtype=float,
        )


        fold_outcome = (
            y_mimic_train.iloc[
                fold_evaluation_positions
            ]
            .to_numpy(
                dtype=np.int8
            )
        )


        assert len(
            fold_probability
        ) == len(
            fold_outcome
        )


        assert np.isfinite(
            fold_probability
        ).all()


        stage_oof_probability[
            fold_evaluation_positions
        ] = fold_probability


        incremental_stage_cv_fold_records.append(
            {
                "stage": stage_name,
                "stage_display": (
                    stage_display_name
                ),
                "fold": fold_number,
                "raw_features": len(
                    stage_features
                ),
                "fit_rows": len(
                    fold_fit_positions
                ),
                "evaluation_rows": len(
                    fold_evaluation_positions
                ),
                "evaluation_prevalence": float(
                    fold_outcome.mean()
                ),
                "auprc": float(
                    average_precision_score(
                        fold_outcome,
                        fold_probability,
                    )
                ),
                "auroc": float(
                    roc_auc_score(
                        fold_outcome,
                        fold_probability,
                    )
                ),
                "log_loss": float(
                    log_loss(
                        fold_outcome,
                        fold_probability,
                        labels=[
                            0,
                            1,
                        ],
                    )
                ),
                "brier_score": float(
                    brier_score_loss(
                        fold_outcome,
                        fold_probability,
                    )
                ),
            }
        )


    stage_cv_elapsed_seconds = (
        time.perf_counter()
        -
        stage_cv_start_time
    )


    assert np.isfinite(
        stage_oof_probability
    ).all()


    stage_fitted_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                create_incremental_stage_preprocessor(
                    stage_features
                ),
            ),
            (
                "model",
                clone(
                    selected_xgboost_model_template
                ),
            ),
        ]
    )


    stage_full_fit_start_time = (
        time.perf_counter()
    )


    stage_fitted_pipeline.fit(
        X_stage_train,
        y_mimic_train,
    )


    stage_full_fit_seconds = (
        time.perf_counter()
        -
        stage_full_fit_start_time
    )


    stage_validation_probability = (
        stage_fitted_pipeline
        .predict_proba(
            X_stage_validation
        )[
            :,
            1,
        ]
    )


    stage_validation_probability = np.asarray(
        stage_validation_probability,
        dtype=float,
    )


    assert len(
        stage_validation_probability
    ) == 3138


    assert np.isfinite(
        stage_validation_probability
    ).all()


    incremental_stage_fitted_pipelines[
        stage_name
    ] = stage_fitted_pipeline


    incremental_stage_validation_predictions[
        f"{stage_name}_raw_probability"
    ] = stage_validation_probability.astype(
        np.float32
    )


    stage_fold_table = pd.DataFrame(
        incremental_stage_cv_fold_records
    )


    stage_fold_table = stage_fold_table.loc[
        stage_fold_table[
            "stage"
        ].eq(
            stage_name
        )
    ]


    incremental_stage_summary_records.append(
        {
            "stage": stage_name,
            "stage_display": (
                stage_display_name
            ),
            "description": (
                incremental_stage_descriptions[
                    stage_name
                ]
            ),
            "raw_features": len(
                stage_features
            ),
            "training_rows": len(
                X_stage_train
            ),
            "validation_rows": len(
                X_stage_validation
            ),
            "mean_cv_auprc": float(
                stage_fold_table[
                    "auprc"
                ].mean()
            ),
            "std_cv_auprc": float(
                stage_fold_table[
                    "auprc"
                ].std(
                    ddof=0
                )
            ),
            "oof_cv_auprc": float(
                average_precision_score(
                    y_mimic_train,
                    stage_oof_probability,
                )
            ),
            "mean_cv_auroc": float(
                stage_fold_table[
                    "auroc"
                ].mean()
            ),
            "std_cv_auroc": float(
                stage_fold_table[
                    "auroc"
                ].std(
                    ddof=0
                )
            ),
            "oof_cv_auroc": float(
                roc_auc_score(
                    y_mimic_train,
                    stage_oof_probability,
                )
            ),
            "oof_cv_log_loss": float(
                log_loss(
                    y_mimic_train,
                    stage_oof_probability,
                    labels=[
                        0,
                        1,
                    ],
                )
            ),
            "oof_cv_brier_score": float(
                brier_score_loss(
                    y_mimic_train,
                    stage_oof_probability,
                )
            ),
            "validation_prevalence": float(
                y_mimic_validation.mean()
            ),
            "validation_mean_probability": float(
                stage_validation_probability.mean()
            ),
            "validation_auprc": float(
                average_precision_score(
                    y_mimic_validation,
                    stage_validation_probability,
                )
            ),
            "validation_auroc": float(
                roc_auc_score(
                    y_mimic_validation,
                    stage_validation_probability,
                )
            ),
            "validation_log_loss": float(
                log_loss(
                    y_mimic_validation,
                    stage_validation_probability,
                    labels=[
                        0,
                        1,
                    ],
                )
            ),
            "validation_brier_score": float(
                brier_score_loss(
                    y_mimic_validation,
                    stage_validation_probability,
                )
            ),
            "cv_seconds": float(
                stage_cv_elapsed_seconds
            ),
            "full_training_fit_seconds": float(
                stage_full_fit_seconds
            ),
        }
    )



incremental_stage_cv_fold_results = pd.DataFrame(
    incremental_stage_cv_fold_records
)


incremental_feature_stage_summary = pd.DataFrame(
    incremental_stage_summary_records
)


stage_order_lookup = {
    "S0": 1,
    "S0_S1": 2,
    "S0_S1_S2": 3,
}


incremental_feature_stage_summary[
    "stage_order"
] = (
    incremental_feature_stage_summary[
        "stage"
    ]
    .map(
        stage_order_lookup
    )
)


incremental_feature_stage_summary = (
    incremental_feature_stage_summary
    .sort_values(
        "stage_order"
    )
    .reset_index(
        drop=True
    )
)


incremental_feature_stage_summary[
    "delta_validation_auprc_vs_previous_stage"
] = (
    incremental_feature_stage_summary[
        "validation_auprc"
    ]
    .diff()
)


incremental_feature_stage_summary[
    "delta_validation_auroc_vs_previous_stage"
] = (
    incremental_feature_stage_summary[
        "validation_auroc"
    ]
    .diff()
)


incremental_feature_stage_summary[
    "delta_validation_log_loss_vs_previous_stage"
] = (
    incremental_feature_stage_summary[
        "validation_log_loss"
    ]
    .diff()
)


incremental_feature_stage_summary[
    "delta_validation_brier_vs_previous_stage"
] = (
    incremental_feature_stage_summary[
        "validation_brier_score"
    ]
    .diff()
)


incremental_feature_stage_summary[
    "auprc_above_validation_prevalence"
] = (
    incremental_feature_stage_summary[
        "validation_auprc"
    ]
    -
    incremental_feature_stage_summary[
        "validation_prevalence"
    ]
)



previous_locked_validation_result = (
    validation_model_comparison.loc[
        validation_model_comparison[
            "algorithm"
        ].eq(
            "XGBoost"
        )
    ]
    .iloc[0]
)


full_stage_result = (
    incremental_feature_stage_summary.loc[
        incremental_feature_stage_summary[
            "stage"
        ].eq(
            "S0_S1_S2"
        )
    ]
    .iloc[0]
)


full_stage_reproduction_audit = pd.DataFrame(
    [
        {
            "metric": "Validation AUPRC",
            "previous_locked_analysis": float(
                previous_locked_validation_result[
                    "validation_auprc"
                ]
            ),
            "incremental_full_stage": float(
                full_stage_result[
                    "validation_auprc"
                ]
            ),
        },
        {
            "metric": "Validation AUROC",
            "previous_locked_analysis": float(
                previous_locked_validation_result[
                    "validation_auroc"
                ]
            ),
            "incremental_full_stage": float(
                full_stage_result[
                    "validation_auroc"
                ]
            ),
        },
        {
            "metric": "Validation log loss",
            "previous_locked_analysis": float(
                previous_locked_validation_result[
                    "validation_log_loss"
                ]
            ),
            "incremental_full_stage": float(
                full_stage_result[
                    "validation_log_loss"
                ]
            ),
        },
        {
            "metric": "Validation Brier score",
            "previous_locked_analysis": float(
                previous_locked_validation_result[
                    "validation_brier_score"
                ]
            ),
            "incremental_full_stage": float(
                full_stage_result[
                    "validation_brier_score"
                ]
            ),
        },
    ]
)


full_stage_reproduction_audit[
    "absolute_difference"
] = (
    full_stage_reproduction_audit[
        "incremental_full_stage"
    ]
    -
    full_stage_reproduction_audit[
        "previous_locked_analysis"
    ]
).abs()


assert full_stage_reproduction_audit[
    "absolute_difference"
].lt(
    1e-5
).all(), (
    "The full incremental stage does not reproduce the "
    "previously selected XGBoost validation result.\n\n"
    f"{full_stage_reproduction_audit}"
)



INCREMENTAL_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook2"
)


INCREMENTAL_PREDICTION_DIR = (
    PROJECT_ROOT
    / "results"
    / "predictions"
    / "notebook2"
)


INCREMENTAL_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "notebook2"
)


for directory in [
    INCREMENTAL_TABLE_DIR,
    INCREMENTAL_PREDICTION_DIR,
    INCREMENTAL_FIGURE_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


incremental_output_paths = {
    "cv_fold_results": (
        INCREMENTAL_TABLE_DIR
        / "incremental_feature_stage_cv_fold_results.csv"
    ),
    "stage_summary": (
        INCREMENTAL_TABLE_DIR
        / "incremental_feature_stage_summary.csv"
    ),
    "full_stage_reproduction_audit": (
        INCREMENTAL_TABLE_DIR
        / "incremental_full_stage_reproduction_audit.csv"
    ),
    "validation_predictions": (
        INCREMENTAL_PREDICTION_DIR
        / "incremental_feature_stage_validation_predictions.parquet"
    ),
    "validation_discrimination_png": (
        INCREMENTAL_FIGURE_DIR
        / "09_incremental_stage_validation_discrimination.png"
    ),
    "validation_discrimination_pdf": (
        INCREMENTAL_FIGURE_DIR
        / "09_incremental_stage_validation_discrimination.pdf"
    ),
    "cv_validation_auprc_png": (
        INCREMENTAL_FIGURE_DIR
        / "10_incremental_stage_cv_validation_auprc.png"
    ),
    "cv_validation_auprc_pdf": (
        INCREMENTAL_FIGURE_DIR
        / "10_incremental_stage_cv_validation_auprc.pdf"
    ),
    "artifact_manifest": (
        INCREMENTAL_TABLE_DIR
        / "incremental_feature_stage_artifact_manifest.csv"
    ),
}


incremental_stage_cv_fold_results.to_csv(
    incremental_output_paths[
        "cv_fold_results"
    ],
    index=False,
)


incremental_feature_stage_summary.to_csv(
    incremental_output_paths[
        "stage_summary"
    ],
    index=False,
)


full_stage_reproduction_audit.to_csv(
    incremental_output_paths[
        "full_stage_reproduction_audit"
    ],
    index=False,
)


incremental_stage_validation_predictions.to_parquet(
    incremental_output_paths[
        "validation_predictions"
    ],
    index=True,
    engine="pyarrow",
)



stage_plot_positions = np.arange(
    len(
        incremental_feature_stage_summary
    )
)


stage_plot_width = 0.36


figure, axis = plt.subplots(
    figsize=(
        10,
        6,
    )
)


stage_auprc_bars = axis.bar(
    stage_plot_positions
    -
    stage_plot_width
    /
    2,
    incremental_feature_stage_summary[
        "validation_auprc"
    ],
    width=stage_plot_width,
    label="Validation AUPRC",
)


stage_auroc_bars = axis.bar(
    stage_plot_positions
    +
    stage_plot_width
    /
    2,
    incremental_feature_stage_summary[
        "validation_auroc"
    ],
    width=stage_plot_width,
    label="Validation AUROC",
)


axis.axhline(
    y=float(
        y_mimic_validation.mean()
    ),
    linestyle="--",
    linewidth=1.2,
    label=(
        "Validation prevalence "
        f"({y_mimic_validation.mean():.3f})"
    ),
)


axis.set_xticks(
    stage_plot_positions
)


axis.set_xticklabels(
    incremental_feature_stage_summary[
        "stage_display"
    ]
)


axis.set_ylim(
    0.0,
    1.0,
)


axis.set_xlabel(
    "Clinical information stage"
)


axis.set_ylabel(
    "Performance"
)


axis.set_title(
    "Incremental validation discrimination across feature stages"
)


axis.legend(
    frameon=False
)


axis.bar_label(
    stage_auprc_bars,
    fmt="%.3f",
    padding=3,
)


axis.bar_label(
    stage_auroc_bars,
    fmt="%.3f",
    padding=3,
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    incremental_output_paths[
        "validation_discrimination_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    incremental_output_paths[
        "validation_discrimination_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



figure, axis = plt.subplots(
    figsize=(
        10,
        6,
    )
)


axis.plot(
    stage_plot_positions,
    incremental_feature_stage_summary[
        "oof_cv_auprc"
    ],
    marker="o",
    linewidth=2.0,
    label="Training OOF CV AUPRC",
)


axis.plot(
    stage_plot_positions,
    incremental_feature_stage_summary[
        "validation_auprc"
    ],
    marker="o",
    linewidth=2.0,
    label="Later-period validation AUPRC",
)


axis.axhline(
    y=float(
        y_mimic_validation.mean()
    ),
    linestyle="--",
    linewidth=1.2,
    label=(
        "Validation prevalence "
        f"({y_mimic_validation.mean():.3f})"
    ),
)


axis.set_xticks(
    stage_plot_positions
)


axis.set_xticklabels(
    incremental_feature_stage_summary[
        "stage_display"
    ]
)


axis.set_ylim(
    0.0,
    1.0,
)


axis.set_xlabel(
    "Clinical information stage"
)


axis.set_ylabel(
    "AUPRC"
)


axis.set_title(
    "Incremental training-CV and validation AUPRC"
)


axis.legend(
    frameon=False
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    incremental_output_paths[
        "cv_validation_auprc_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    incremental_output_paths[
        "cv_validation_auprc_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



def incremental_file_sha256(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )


    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()


incremental_artifact_records = []


for artifact_name, artifact_path in (
    incremental_output_paths.items()
):

    if artifact_name == "artifact_manifest":
        continue


    if not artifact_path.exists():

        raise FileNotFoundError(
            "Expected incremental-analysis artifact was not "
            f"written:\n{artifact_path}"
        )


    incremental_artifact_records.append(
        {
            "artifact": artifact_name,
            "filename": artifact_path.name,
            "directory": str(
                artifact_path.parent
            ),
            "size_bytes": int(
                artifact_path
                .stat()
                .st_size
            ),
            "sha256": incremental_file_sha256(
                artifact_path
            ),
        }
    )


incremental_feature_stage_artifact_manifest = (
    pd.DataFrame(
        incremental_artifact_records
    )
)


incremental_feature_stage_artifact_manifest.to_csv(
    incremental_output_paths[
        "artifact_manifest"
    ],
    index=False,
)



print(
    "Incremental feature-stage results:"
)

display(
    incremental_feature_stage_summary[
        [
            "stage_display",
            "raw_features",
            "mean_cv_auprc",
            "std_cv_auprc",
            "oof_cv_auprc",
            "validation_auprc",
            "delta_validation_auprc_vs_previous_stage",
            "oof_cv_auroc",
            "validation_auroc",
            "delta_validation_auroc_vs_previous_stage",
            "validation_log_loss",
            "validation_brier_score",
            "validation_mean_probability",
        ]
    ].round(
        6
    )
)


print(
    "\nFull-stage reproduction audit:"
)

display(
    full_stage_reproduction_audit.round(
        10
    )
)


print(
    "\nIncremental-analysis artifact manifest:"
)

display(
    incremental_feature_stage_artifact_manifest
)


print(
    "\nINCREMENTAL FEATURE-STAGE ANALYSIS COMPLETED"
)

print(
    "Algorithm:",
    "XGBoost",
)

print(
    "Hyperparameters changed across stages: NO"
)

print(
    "Training CV folds changed across stages: NO"
)

print(
    "Stage-specific calibration fitted: NO"
)

print(
    "Stage-specific threshold selected: NO"
)

print(
    "Locked full-model threshold remains:",
    round(
        float(
            locked_decision_threshold
        ),
        6,
    ),
)

print(
    "\nInternal test used: NO"
)

print(
    "MC-MED used: NO"
)

print(
    "Final locked model changed: NO"
)

print(
    "Tables saved: YES"
)

print(
    "Validation predictions saved: YES"
)

print(
    "Figures saved at 600 dpi and vector PDF: YES"
)

print(
    "Stage-specific fitted pipelines saved: NO"
)

print(
    "\nThis is a secondary explanatory analysis and must not "
    "change the locked primary model."
)

## Locked full-model global SHAP interpretation

Global SHAP analysis is performed for the locked S0 + S1 + S2 XGBoost model using the complete internal temporal-test cohort.

SHAP values quantify how individual features move the model prediction toward ADMITTED or toward HOME. Positive SHAP values increase the model’s predicted admission risk, while negative values decrease it.

The analysis is descriptive only. It does not refit preprocessing, retrain the model, recalibrate probabilities, alter the locked threshold or influence any model-development decision. One-hot encoded categorical contributions are additionally aggregated back to their original raw features for clinically interpretable ranking.

In [ ]:



required_shap_objects = [
    "PROJECT_ROOT",
    "validation_selected_algorithm",
    "validation_selected_pipeline",
    "X_mimic_test_standardised",
    "y_mimic_test",
    "internal_test_predictions",
    "feature_manifest",
    "raw_feature_order",
    "raw_feature_order_sha256",
    "s0_numeric_features",
    "s0_categorical_features",
    "binary_feature_order",
]


missing_shap_objects = [
    object_name
    for object_name in required_shap_objects
    if object_name not in globals()
]


if missing_shap_objects:

    raise RuntimeError(
        "Run the complete locked internal-test evaluation "
        "before performing SHAP interpretation.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_shap_objects
        )
    )


assert validation_selected_algorithm == "XGBoost"


assert set(
    s0_categorical_features
) == {
    "sex_gender",
    "arrival_transport_grouped",
}


assert len(
    s0_numeric_features
) == 7


assert len(
    binary_feature_order
) == 2486


assert len(
    raw_feature_order
) == 2495


assert len(
    set(
        raw_feature_order
    )
) == 2495


assert X_mimic_test_standardised.shape == (
    1851,
    2495,
)


assert X_mimic_test_standardised.index.equals(
    y_mimic_test.index
)


assert len(
    internal_test_predictions
) == 1851



locked_shap_preprocessor = (
    validation_selected_pipeline
    .named_steps[
        "preprocessor"
    ]
)


locked_shap_model = (
    validation_selected_pipeline
    .named_steps[
        "model"
    ]
)


assert isinstance(
    locked_shap_model,
    XGBClassifier,
)


assert hasattr(
    locked_shap_preprocessor,
    "transformers_",
)


locked_xgboost_booster = (
    locked_shap_model
    .get_booster()
)



X_internal_test_transformed = (
    locked_shap_preprocessor
    .transform(
        X_mimic_test_standardised
    )
)


if sparse.issparse(
    X_internal_test_transformed
):

    X_internal_test_transformed = (
        X_internal_test_transformed
        .tocsr()
        .astype(
            np.float32
        )
    )

else:

    X_internal_test_transformed = np.asarray(
        X_internal_test_transformed,
        dtype=np.float32,
    )


assert X_internal_test_transformed.shape[0] == 1851


if sparse.issparse(
    X_internal_test_transformed
):

    assert np.isfinite(
        X_internal_test_transformed.data
    ).all()

else:

    assert np.isfinite(
        X_internal_test_transformed
    ).all()



fitted_categorical_encoder = (
    locked_shap_preprocessor
    .named_transformers_[
        "s0_categorical"
    ]
    .named_steps[
        "one_hot_encoder"
    ]
)


fitted_category_levels = (
    fitted_categorical_encoder
    .categories_
)


assert len(
    fitted_category_levels
) == len(
    s0_categorical_features
)


one_hot_category_audit_records = []


for feature_name, category_levels in zip(
    s0_categorical_features,
    fitted_category_levels,
):

    one_hot_category_audit_records.append(
        {
            "raw_categorical_feature": (
                feature_name
            ),
            "training_categories": int(
                len(
                    category_levels
                )
            ),
            "category_levels": " | ".join(
                [
                    str(
                        category
                    )
                    for category
                    in category_levels
                ]
            ),
        }
    )


one_hot_category_audit = pd.DataFrame(
    one_hot_category_audit_records
)


encoded_categorical_feature_names = (
    fitted_categorical_encoder
    .get_feature_names_out(
        s0_categorical_features
    )
    .tolist()
)


observed_encoded_categorical_count = len(
    encoded_categorical_feature_names
)


expected_dynamic_prepared_feature_count = (
    len(
        s0_numeric_features
    )
    +
    observed_encoded_categorical_count
    +
    len(
        binary_feature_order
    )
)


assert observed_encoded_categorical_count == 5


assert expected_dynamic_prepared_feature_count == 2498


assert (
    X_internal_test_transformed.shape[1]
    ==
    expected_dynamic_prepared_feature_count
), (
    "Transformed feature width does not match the fitted "
    "preprocessor structure.\n\n"
    f"Observed transformed columns: "
    f"{X_internal_test_transformed.shape[1]}\n"
    f"Expected dynamically: "
    f"{expected_dynamic_prepared_feature_count}"
)


print(
    "Training-derived one-hot category audit:"
)

display(
    one_hot_category_audit
)


print(
    "Numeric S0 columns:",
    len(
        s0_numeric_features
    ),
)

print(
    "One-hot categorical columns:",
    observed_encoded_categorical_count,
)

print(
    "Binary S1/S2 columns:",
    len(
        binary_feature_order
    ),
)

print(
    "Prepared model columns:",
    expected_dynamic_prepared_feature_count,
)

print(
    "Raw grouped features:",
    len(
        raw_feature_order
    ),
)



prepared_feature_mapping_records = []


prepared_position = 0


for feature_name in s0_numeric_features:

    prepared_feature_mapping_records.append(
        {
            "prepared_position": prepared_position,
            "prepared_feature_name": feature_name,
            "source_raw_feature": feature_name,
            "category_level": pd.NA,
            "is_grouped_categorical": False,
        }
    )


    prepared_position += 1



encoded_name_offset = 0


for feature_name, category_levels in zip(
    s0_categorical_features,
    fitted_category_levels,
):

    number_of_categories = len(
        category_levels
    )


    feature_encoded_names = (
        encoded_categorical_feature_names[
            encoded_name_offset:
            encoded_name_offset
            +
            number_of_categories
        ]
    )


    assert len(
        feature_encoded_names
    ) == number_of_categories


    for encoded_name, category_level in zip(
        feature_encoded_names,
        category_levels,
    ):

        prepared_feature_mapping_records.append(
            {
                "prepared_position": prepared_position,
                "prepared_feature_name": encoded_name,
                "source_raw_feature": feature_name,
                "category_level": str(
                    category_level
                ),
                "is_grouped_categorical": True,
            }
        )


        prepared_position += 1


    encoded_name_offset += number_of_categories


assert (
    encoded_name_offset
    ==
    observed_encoded_categorical_count
)


for feature_name in binary_feature_order:

    prepared_feature_mapping_records.append(
        {
            "prepared_position": prepared_position,
            "prepared_feature_name": feature_name,
            "source_raw_feature": feature_name,
            "category_level": pd.NA,
            "is_grouped_categorical": False,
        }
    )


    prepared_position += 1


prepared_shap_feature_mapping = pd.DataFrame(
    prepared_feature_mapping_records
)


assert len(
    prepared_shap_feature_mapping
) == expected_dynamic_prepared_feature_count


assert prepared_shap_feature_mapping[
    "prepared_position"
].tolist() == list(
    range(
        expected_dynamic_prepared_feature_count
    )
)


prepared_shap_feature_order = (
    prepared_shap_feature_mapping[
        "prepared_feature_name"
    ]
    .tolist()
)


assert len(
    prepared_shap_feature_order
) == X_internal_test_transformed.shape[1]


assert len(
    set(
        prepared_shap_feature_order
    )
) == len(
    prepared_shap_feature_order
)


assert set(
    prepared_shap_feature_mapping[
        "source_raw_feature"
    ]
) == set(
    raw_feature_order
)



manifest_metadata = (
    feature_manifest[
        [
            "feature_name",
            "family",
            "timing_stage",
            "storage_type",
        ]
    ]
    .rename(
        columns={
            "feature_name": (
                "source_raw_feature"
            ),
        }
    )
)


prepared_shap_feature_mapping = (
    prepared_shap_feature_mapping
    .merge(
        manifest_metadata,
        on="source_raw_feature",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        "prepared_position"
    )
    .reset_index(
        drop=True
    )
)


assert prepared_shap_feature_mapping[
    "family"
].notna().all()


assert prepared_shap_feature_mapping[
    "timing_stage"
].notna().all()



shap_start_time = time.perf_counter()


internal_test_dmatrix = xgb.DMatrix(
    X_internal_test_transformed
)


native_prediction_kwargs = {
    "validate_features": False,
}



try:

    locked_best_iteration = (
        locked_shap_model
        .best_iteration
    )

except Exception:

    locked_best_iteration = None


if locked_best_iteration is not None:

    native_prediction_kwargs[
        "iteration_range"
    ] = (
        0,
        int(
            locked_best_iteration
        )
        +
        1,
    )


native_contribution_output = (
    locked_xgboost_booster
    .predict(
        internal_test_dmatrix,
        pred_contribs=True,
        approx_contribs=False,
        **native_prediction_kwargs,
    )
)


native_contribution_output = np.asarray(
    native_contribution_output,
    dtype=np.float32,
)



if native_contribution_output.ndim == 3:

    if native_contribution_output.shape[1] == 1:

        native_contribution_output = (
            native_contribution_output[
                :,
                0,
                :,
            ]
        )

    elif native_contribution_output.shape[2] == 1:

        native_contribution_output = (
            native_contribution_output[
                :,
                :,
                0,
            ]
        )

    else:

        raise ValueError(
            "Unexpected native XGBoost contribution shape:\n"
            f"{native_contribution_output.shape}"
        )


assert native_contribution_output.ndim == 2


assert native_contribution_output.shape == (
    1851,
    expected_dynamic_prepared_feature_count
    +
    1,
), (
    "Native XGBoost contribution matrix has an unexpected "
    "shape.\n\n"
    f"Observed: {native_contribution_output.shape}\n"
    f"Expected: "
    f"{(1851, expected_dynamic_prepared_feature_count + 1)}"
)


internal_test_prepared_shap_values = (
    native_contribution_output[
        :,
        :-1,
    ]
)


internal_test_native_bias = (
    native_contribution_output[
        :,
        -1,
    ]
)


assert internal_test_prepared_shap_values.shape == (
    1851,
    2498,
)


assert internal_test_native_bias.shape == (
    1851,
)


assert np.isfinite(
    internal_test_prepared_shap_values
).all()


assert np.isfinite(
    internal_test_native_bias
).all()



internal_test_native_margin = (
    locked_xgboost_booster
    .predict(
        internal_test_dmatrix,
        output_margin=True,
        **native_prediction_kwargs,
    )
)


internal_test_native_margin = np.asarray(
    internal_test_native_margin,
    dtype=np.float32,
).reshape(
    -1
)


internal_test_reconstructed_margin = (
    internal_test_native_bias
    +
    internal_test_prepared_shap_values.sum(
        axis=1
    )
)


native_additivity_absolute_error = np.abs(
    internal_test_native_margin
    -
    internal_test_reconstructed_margin
)


assert np.allclose(
    internal_test_native_margin,
    internal_test_reconstructed_margin,
    rtol=1e-5,
    atol=1e-4,
), (
    "Native XGBoost SHAP contributions do not reconstruct "
    "the model margin.\n\n"
    f"Maximum absolute error: "
    f"{native_additivity_absolute_error.max()}"
)



internal_test_native_probability = (
    locked_xgboost_booster
    .predict(
        internal_test_dmatrix,
        **native_prediction_kwargs,
    )
)


internal_test_native_probability = np.asarray(
    internal_test_native_probability,
    dtype=float,
).reshape(
    -1
)


saved_locked_raw_probability = (
    pd.to_numeric(
        internal_test_predictions[
            "raw_probability"
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=float
    )
)


maximum_probability_difference = float(
    np.max(
        np.abs(
            internal_test_native_probability
            -
            saved_locked_raw_probability
        )
    )
)


assert np.allclose(
    internal_test_native_probability,
    saved_locked_raw_probability,
    rtol=1e-6,
    atol=1e-6,
), (
    "Native XGBoost prediction does not reproduce the saved "
    "locked-model raw probability.\n\n"
    f"Maximum absolute difference: "
    f"{maximum_probability_difference}"
)



raw_feature_position_lookup = {
    feature_name: position
    for position, feature_name
    in enumerate(
        raw_feature_order
    )
}


internal_test_grouped_raw_shap_values = np.zeros(
    (
        len(
            X_mimic_test_standardised
        ),
        len(
            raw_feature_order
        ),
    ),
    dtype=np.float32,
)


for mapping_row in (
    prepared_shap_feature_mapping
    .itertuples(
        index=False
    )
):

    raw_position = (
        raw_feature_position_lookup[
            mapping_row.source_raw_feature
        ]
    )


    internal_test_grouped_raw_shap_values[
        :,
        raw_position,
    ] += internal_test_prepared_shap_values[
        :,
        mapping_row.prepared_position,
    ]


assert internal_test_grouped_raw_shap_values.shape == (
    1851,
    2495,
)


assert np.isfinite(
    internal_test_grouped_raw_shap_values
).all()



prepared_total_contribution = (
    internal_test_prepared_shap_values
    .sum(
        axis=1
    )
)


grouped_total_contribution = (
    internal_test_grouped_raw_shap_values
    .sum(
        axis=1
    )
)


grouping_absolute_error = np.abs(
    prepared_total_contribution
    -
    grouped_total_contribution
)


assert np.allclose(
    prepared_total_contribution,
    grouped_total_contribution,
    rtol=1e-5,
    atol=1e-5,
), (
    "Patient-level SHAP totals changed during one-hot "
    "grouping.\n\n"
    f"Maximum absolute difference: "
    f"{grouping_absolute_error.max()}"
)


shap_elapsed_seconds = (
    time.perf_counter()
    -
    shap_start_time
)



internal_test_grouped_raw_feature_values = np.full(
    (
        len(
            X_mimic_test_standardised
        ),
        len(
            raw_feature_order
        ),
    ),
    np.nan,
    dtype=np.float32,
)


number_of_numeric_columns = len(
    s0_numeric_features
)


if sparse.issparse(
    X_internal_test_transformed
):

    transformed_numeric_values = (
        X_internal_test_transformed[
            :,
            :number_of_numeric_columns,
        ]
        .toarray()
        .astype(
            np.float32
        )
    )

else:

    transformed_numeric_values = np.asarray(
        X_internal_test_transformed[
            :,
            :number_of_numeric_columns,
        ],
        dtype=np.float32,
    )


numeric_raw_positions = [
    raw_feature_position_lookup[
        feature_name
    ]
    for feature_name
    in s0_numeric_features
]


internal_test_grouped_raw_feature_values[
    :,
    numeric_raw_positions,
] = transformed_numeric_values



categorical_code_mapping_records = []


for feature_name, category_levels in zip(
    s0_categorical_features,
    fitted_category_levels,
):

    category_string_levels = [
        str(
            category
        )
        for category
        in category_levels
    ]


    category_to_code = {
        category_value: category_code
        for category_code, category_value
        in enumerate(
            category_string_levels
        )
    }


    standardised_values = (
        X_mimic_test_standardised[
            feature_name
        ]
        .astype(
            "string"
        )
        .fillna(
            "__MISSING__"
        )
        .astype(str)
    )


    category_codes = (
        standardised_values
        .map(
            category_to_code
        )
        .fillna(
            -1
        )
        .to_numpy(
            dtype=np.float32
        )
    )


    categorical_raw_position = (
        raw_feature_position_lookup[
            feature_name
        ]
    )


    internal_test_grouped_raw_feature_values[
        :,
        categorical_raw_position,
    ] = category_codes


    for category_value, category_code in (
        category_to_code.items()
    ):

        categorical_code_mapping_records.append(
            {
                "raw_categorical_feature": feature_name,
                "beeswarm_category_code": int(
                    category_code
                ),
                "category_level": category_value,
                "seen_during_training": True,
            }
        )


    if np.any(
        category_codes == -1
    ):

        categorical_code_mapping_records.append(
            {
                "raw_categorical_feature": feature_name,
                "beeswarm_category_code": -1,
                "category_level": (
                    "UNKNOWN_TO_TRAINING_ENCODER"
                ),
                "seen_during_training": False,
            }
        )


categorical_beeswarm_code_mapping = pd.DataFrame(
    categorical_code_mapping_records
)


binary_prepared_start = (
    len(
        s0_numeric_features
    )
    +
    observed_encoded_categorical_count
)


binary_prepared_end = (
    binary_prepared_start
    +
    len(
        binary_feature_order
    )
)


if sparse.issparse(
    X_internal_test_transformed
):

    transformed_binary_values = (
        X_internal_test_transformed[
            :,
            binary_prepared_start:
            binary_prepared_end,
        ]
        .toarray()
        .astype(
            np.float32
        )
    )

else:

    transformed_binary_values = np.asarray(
        X_internal_test_transformed[
            :,
            binary_prepared_start:
            binary_prepared_end,
        ],
        dtype=np.float32,
    )


binary_raw_positions = [
    raw_feature_position_lookup[
        feature_name
    ]
    for feature_name
    in binary_feature_order
]


internal_test_grouped_raw_feature_values[
    :,
    binary_raw_positions,
] = transformed_binary_values


assert internal_test_grouped_raw_feature_values.shape == (
    1851,
    2495,
)



prepared_shap_global_importance = (
    prepared_shap_feature_mapping
    .copy()
)


prepared_shap_global_importance[
    "mean_absolute_shap"
] = np.mean(
    np.abs(
        internal_test_prepared_shap_values
    ),
    axis=0,
)


prepared_shap_global_importance[
    "mean_signed_shap"
] = np.mean(
    internal_test_prepared_shap_values,
    axis=0,
)


prepared_shap_global_importance[
    "positive_shap_fraction"
] = np.mean(
    internal_test_prepared_shap_values
    >
    0,
    axis=0,
)


prepared_shap_global_importance = (
    prepared_shap_global_importance
    .sort_values(
        "mean_absolute_shap",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


prepared_shap_global_importance.insert(
    0,
    "prepared_importance_rank",
    np.arange(
        1,
        len(
            prepared_shap_global_importance
        )
        +
        1,
    ),
)



manifest_in_raw_order = (
    feature_manifest
    .set_index(
        "feature_name"
    )
    .loc[
        raw_feature_order
    ]
    .reset_index()
    .rename(
        columns={
            "feature_name": (
                "source_raw_feature"
            ),
        }
    )
)


prepared_column_counts = (
    prepared_shap_feature_mapping
    .groupby(
        "source_raw_feature",
        observed=True,
    )
    .size()
    .rename(
        "prepared_columns_grouped"
    )
)


raw_feature_shap_importance = (
    manifest_in_raw_order
    .merge(
        prepared_column_counts,
        left_on="source_raw_feature",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
)


raw_feature_shap_importance[
    "mean_absolute_shap"
] = np.mean(
    np.abs(
        internal_test_grouped_raw_shap_values
    ),
    axis=0,
)


raw_feature_shap_importance[
    "mean_signed_shap"
] = np.mean(
    internal_test_grouped_raw_shap_values,
    axis=0,
)


raw_feature_shap_importance[
    "positive_shap_fraction"
] = np.mean(
    internal_test_grouped_raw_shap_values
    >
    0,
    axis=0,
)


raw_feature_shap_importance = (
    raw_feature_shap_importance
    .sort_values(
        "mean_absolute_shap",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


raw_feature_shap_importance.insert(
    0,
    "raw_feature_rank",
    np.arange(
        1,
        len(
            raw_feature_shap_importance
        )
        +
        1,
    ),
)


assert len(
    raw_feature_shap_importance
) == 2495


total_grouped_mean_absolute_shap = float(
    raw_feature_shap_importance[
        "mean_absolute_shap"
    ].sum()
)


raw_feature_shap_importance[
    "global_importance_percent"
] = (
    100
    *
    raw_feature_shap_importance[
        "mean_absolute_shap"
    ]
    /
    total_grouped_mean_absolute_shap
)



shap_feature_family_summary = (
    raw_feature_shap_importance
    .groupby(
        [
            "family",
            "timing_stage",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        raw_features=(
            "source_raw_feature",
            "count",
        ),
        total_mean_absolute_shap=(
            "mean_absolute_shap",
            "sum",
        ),
        total_mean_signed_shap=(
            "mean_signed_shap",
            "sum",
        ),
    )
    .sort_values(
        "total_mean_absolute_shap",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


shap_feature_family_summary[
    "importance_share_percent"
] = (
    100
    *
    shap_feature_family_summary[
        "total_mean_absolute_shap"
    ]
    /
    shap_feature_family_summary[
        "total_mean_absolute_shap"
    ].sum()
)


shap_feature_family_summary.insert(
    0,
    "family_importance_rank",
    np.arange(
        1,
        len(
            shap_feature_family_summary
        )
        +
        1,
    ),
)



native_shap_grouping_audit = pd.DataFrame(
    [
        {
            "calculation_method": (
                "XGBoost native pred_contribs"
            ),
            "test_rows": 1851,
            "numeric_prepared_columns": int(
                len(
                    s0_numeric_features
                )
            ),
            "one_hot_prepared_columns": int(
                observed_encoded_categorical_count
            ),
            "binary_prepared_columns": int(
                len(
                    binary_feature_order
                )
            ),
            "total_prepared_columns": int(
                expected_dynamic_prepared_feature_count
            ),
            "grouped_raw_features": int(
                len(
                    raw_feature_order
                )
            ),
            "categorical_raw_features_grouped": int(
                len(
                    s0_categorical_features
                )
            ),
            "maximum_additivity_error": float(
                native_additivity_absolute_error.max()
            ),
            "maximum_probability_difference": float(
                maximum_probability_difference
            ),
            "maximum_grouping_error": float(
                grouping_absolute_error.max()
            ),
            "model_refitted": False,
            "preprocessor_refitted": False,
            "calibrator_used": False,
            "threshold_used": False,
        }
    ]
)



TOP_SHAP_FEATURES = 30


top_raw_features = (
    raw_feature_shap_importance
    .head(
        TOP_SHAP_FEATURES
    )[
        "source_raw_feature"
    ]
    .tolist()
)


top_raw_positions = np.asarray(
    [
        raw_feature_position_lookup[
            feature_name
        ]
        for feature_name
        in top_raw_features
    ],
    dtype=int,
)


top_grouped_raw_shap_values = (
    internal_test_grouped_raw_shap_values[
        :,
        top_raw_positions,
    ]
)


top_grouped_raw_feature_values = (
    internal_test_grouped_raw_feature_values[
        :,
        top_raw_positions,
    ]
)


top_raw_display_names = [
    (
        f"{feature_name} [grouped categorical codes]"
        if feature_name
        in s0_categorical_features
        else feature_name
    )
    for feature_name
    in top_raw_features
]


grouped_raw_shap_explanation = shap.Explanation(
    values=(
        top_grouped_raw_shap_values
    ),
    base_values=(
        internal_test_native_bias
    ),
    data=(
        top_grouped_raw_feature_values
    ),
    feature_names=(
        top_raw_display_names
    ),
)



SHAP_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook2"
)


SHAP_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "notebook2"
)


SHAP_ARRAY_DIR = (
    PROJECT_ROOT
    / "results"
    / "shap"
    / "notebook2"
)


for directory in [
    SHAP_TABLE_DIR,
    SHAP_FIGURE_DIR,
    SHAP_ARRAY_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


shap_output_paths = {
    "one_hot_category_audit": (
        SHAP_TABLE_DIR
        / "shap_one_hot_category_audit.csv"
    ),
    "categorical_code_mapping": (
        SHAP_TABLE_DIR
        / "shap_categorical_beeswarm_code_mapping.csv"
    ),
    "prepared_feature_mapping": (
        SHAP_TABLE_DIR
        / "shap_prepared_feature_mapping.csv"
    ),
    "prepared_feature_importance": (
        SHAP_TABLE_DIR
        / "shap_prepared_feature_importance.csv"
    ),
    "grouped_raw_feature_importance": (
        SHAP_TABLE_DIR
        / "shap_grouped_raw_feature_importance.csv"
    ),
    "feature_family_summary": (
        SHAP_TABLE_DIR
        / "shap_feature_family_summary.csv"
    ),
    "native_shap_grouping_audit": (
        SHAP_TABLE_DIR
        / "shap_native_calculation_and_grouping_audit.csv"
    ),
    "prepared_shap_values": (
        SHAP_ARRAY_DIR
        / "internal_test_prepared_shap_values.npz"
    ),
    "grouped_raw_shap_values": (
        SHAP_ARRAY_DIR
        / "internal_test_grouped_raw_shap_values.npz"
    ),
    "top_raw_features_png": (
        SHAP_FIGURE_DIR
        / "11_shap_grouped_top_raw_features.png"
    ),
    "top_raw_features_pdf": (
        SHAP_FIGURE_DIR
        / "11_shap_grouped_top_raw_features.pdf"
    ),
    "grouped_beeswarm_png": (
        SHAP_FIGURE_DIR
        / "12_shap_grouped_raw_feature_beeswarm.png"
    ),
    "grouped_beeswarm_pdf": (
        SHAP_FIGURE_DIR
        / "12_shap_grouped_raw_feature_beeswarm.pdf"
    ),
    "family_summary_png": (
        SHAP_FIGURE_DIR
        / "13_shap_feature_family_summary.png"
    ),
    "family_summary_pdf": (
        SHAP_FIGURE_DIR
        / "13_shap_feature_family_summary.pdf"
    ),
    "artifact_manifest": (
        SHAP_TABLE_DIR
        / "shap_grouped_artifact_manifest.csv"
    ),
}



one_hot_category_audit.to_csv(
    shap_output_paths[
        "one_hot_category_audit"
    ],
    index=False,
)


categorical_beeswarm_code_mapping.to_csv(
    shap_output_paths[
        "categorical_code_mapping"
    ],
    index=False,
)


prepared_shap_feature_mapping.to_csv(
    shap_output_paths[
        "prepared_feature_mapping"
    ],
    index=False,
)


prepared_shap_global_importance.to_csv(
    shap_output_paths[
        "prepared_feature_importance"
    ],
    index=False,
)


raw_feature_shap_importance.to_csv(
    shap_output_paths[
        "grouped_raw_feature_importance"
    ],
    index=False,
)


shap_feature_family_summary.to_csv(
    shap_output_paths[
        "feature_family_summary"
    ],
    index=False,
)


native_shap_grouping_audit.to_csv(
    shap_output_paths[
        "native_shap_grouping_audit"
    ],
    index=False,
)


np.savez_compressed(
    shap_output_paths[
        "prepared_shap_values"
    ],
    shap_values=(
        internal_test_prepared_shap_values
    ),
    bias=(
        internal_test_native_bias
    ),
    prepared_feature_names=np.asarray(
        prepared_shap_feature_order,
        dtype=str,
    ),
    stay_ids=np.asarray(
        X_mimic_test_standardised.index
    ),
)


np.savez_compressed(
    shap_output_paths[
        "grouped_raw_shap_values"
    ],
    shap_values=(
        internal_test_grouped_raw_shap_values
    ),
    bias=(
        internal_test_native_bias
    ),
    raw_feature_names=np.asarray(
        raw_feature_order,
        dtype=str,
    ),
    stay_ids=np.asarray(
        X_mimic_test_standardised.index
    ),
)



top_raw_feature_plot_data = (
    raw_feature_shap_importance
    .head(
        TOP_SHAP_FEATURES
    )
    .sort_values(
        "mean_absolute_shap",
        ascending=True,
    )
)


figure, axis = plt.subplots(
    figsize=(
        11,
        10,
    )
)


axis.barh(
    top_raw_feature_plot_data[
        "source_raw_feature"
    ],
    top_raw_feature_plot_data[
        "mean_absolute_shap"
    ],
)


axis.set_xlabel(
    "Mean absolute grouped SHAP value"
)


axis.set_ylabel(
    "Raw feature"
)


axis.set_title(
    "Locked XGBoost: grouped raw-feature SHAP importance"
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    shap_output_paths[
        "top_raw_features_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    shap_output_paths[
        "top_raw_features_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



shap.plots.beeswarm(
    grouped_raw_shap_explanation,
    max_display=TOP_SHAP_FEATURES,
    show=False,
)


current_figure = plt.gcf()


current_axis = plt.gca()


current_axis.set_title(
    "Locked XGBoost grouped raw-feature SHAP distribution"
)


current_figure.tight_layout()


current_figure.savefig(
    shap_output_paths[
        "grouped_beeswarm_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


current_figure.savefig(
    shap_output_paths[
        "grouped_beeswarm_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    current_figure
)



family_plot_data = (
    shap_feature_family_summary
    .sort_values(
        "importance_share_percent",
        ascending=True,
    )
)


figure, axis = plt.subplots(
    figsize=(
        10,
        7,
    )
)


family_bars = axis.barh(
    family_plot_data[
        "family"
    ],
    family_plot_data[
        "importance_share_percent"
    ],
)


axis.set_xlabel(
    "Share of total grouped mean absolute SHAP importance (%)"
)


axis.set_ylabel(
    "Feature family"
)


axis.set_title(
    "Locked XGBoost grouped SHAP contribution by feature family"
)


axis.bar_label(
    family_bars,
    fmt="%.1f%%",
    padding=3,
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    shap_output_paths[
        "family_summary_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    shap_output_paths[
        "family_summary_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



def grouped_shap_file_sha256(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )


    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()


grouped_shap_artifact_records = []


for artifact_name, artifact_path in (
    shap_output_paths.items()
):

    if artifact_name == "artifact_manifest":
        continue


    if not artifact_path.exists():

        raise FileNotFoundError(
            "Expected grouped SHAP artifact was not written:\n"
            f"{artifact_path}"
        )


    grouped_shap_artifact_records.append(
        {
            "artifact": artifact_name,
            "filename": artifact_path.name,
            "directory": str(
                artifact_path.parent
            ),
            "size_bytes": int(
                artifact_path
                .stat()
                .st_size
            ),
            "sha256": (
                grouped_shap_file_sha256(
                    artifact_path
                )
            ),
        }
    )


grouped_shap_artifact_manifest = pd.DataFrame(
    grouped_shap_artifact_records
)


grouped_shap_artifact_manifest.to_csv(
    shap_output_paths[
        "artifact_manifest"
    ],
    index=False,
)



print(
    "Native XGBoost SHAP and grouping audit:"
)

display(
    native_shap_grouping_audit
)


print(
    "\nTop 30 grouped raw features by mean absolute SHAP:"
)

display(
    raw_feature_shap_importance[
        [
            "raw_feature_rank",
            "source_raw_feature",
            "prepared_columns_grouped",
            "family",
            "timing_stage",
            "mean_absolute_shap",
            "global_importance_percent",
            "mean_signed_shap",
            "positive_shap_fraction",
        ]
    ]
    .head(
        30
    )
    .round(
        6
    )
)


print(
    "\nGrouped categorical feature audit:"
)

display(
    raw_feature_shap_importance.loc[
        raw_feature_shap_importance[
            "source_raw_feature"
        ].isin(
            s0_categorical_features
        ),
        [
            "raw_feature_rank",
            "source_raw_feature",
            "prepared_columns_grouped",
            "mean_absolute_shap",
            "global_importance_percent",
            "mean_signed_shap",
        ],
    ].round(
        6
    )
)


print(
    "\nCategorical beeswarm code mapping:"
)

display(
    categorical_beeswarm_code_mapping
)


print(
    "\nSHAP importance by feature family:"
)

display(
    shap_feature_family_summary[
        [
            "family_importance_rank",
            "family",
            "timing_stage",
            "raw_features",
            "total_mean_absolute_shap",
            "importance_share_percent",
            "total_mean_signed_shap",
        ]
    ].round(
        6
    )
)


print(
    "\nGrouped SHAP artifact manifest:"
)

display(
    grouped_shap_artifact_manifest
)


print(
    "\nLOCKED FULL-MODEL GROUPED SHAP ANALYSIS COMPLETED"
)

print(
    "SHAP calculation method:",
    "XGBoost native pred_contribs",
)

print(
    "Explanation cohort:",
    "MIMIC-IV internal temporal test",
)

print(
    "Rows explained:",
    len(
        X_mimic_test_standardised
    ),
)

print(
    "Prepared model columns:",
    expected_dynamic_prepared_feature_count,
)

print(
    "Grouped raw features:",
    len(
        raw_feature_order
    ),
)

print(
    "One-hot encoded columns:",
    observed_encoded_categorical_count,
)

print(
    "Grouped categorical features:",
    s0_categorical_features,
)

print(
    "Native SHAP additivity verified: YES"
)

print(
    "Native probability reproduction verified: YES"
)

print(
    "Patient-level SHAP totals preserved after grouping: YES"
)

print(
    "SHAP calculation minutes:",
    round(
        shap_elapsed_seconds
        /
        60,
        3,
    ),
)

print(
    "\nModel refitted: NO"
)

print(
    "Preprocessor refitted: NO"
)

print(
    "Calibrator refitted: NO"
)

print(
    "Threshold changed: NO"
)

print(
    "Internal-test labels used to calculate SHAP: NO"
)

print(
    "MC-MED loaded: NO"
)

print(
    "Grouped tables saved: YES"
)

print(
    "Grouped SHAP arrays saved: YES"
)

print(
    "Grouped figures saved at 600 dpi and vector PDF: YES"
)

print(
    "\nSHAP remains a descriptive interpretation of the "
    "already-locked model."
)

### Interactive grouped SHAP visualisations

Interactive SHAP visualisations are generated from the previously saved patient-level grouped raw-feature SHAP values.

One-hot categorical contributions remain grouped into the original `sex_gender` and `arrival_transport_grouped` features. Categorical features use discrete category-specific colours rather than the continuous high-to-low colour scale used for numeric features.

The interactive files are descriptive only. They do not recalculate SHAP values, refit the model, alter calibration or modify the locked decision threshold.

In [ ]:



required_interactive_shap_objects = [
    "PROJECT_ROOT",
    "raw_feature_order",
    "raw_feature_shap_importance",
    "internal_test_grouped_raw_shap_values",
    "internal_test_grouped_raw_feature_values",
    "X_mimic_test_standardised",
    "internal_test_predictions",
    "s0_categorical_features",
]


missing_interactive_shap_objects = [
    object_name
    for object_name in required_interactive_shap_objects
    if object_name not in globals()
]


if missing_interactive_shap_objects:

    raise RuntimeError(
        "Complete the grouped SHAP analysis before creating "
        "interactive SHAP plots.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_interactive_shap_objects
        )
    )


assert internal_test_grouped_raw_shap_values.shape == (
    1851,
    2495,
)


assert internal_test_grouped_raw_feature_values.shape == (
    1851,
    2495,
)


assert len(
    raw_feature_order
) == 2495


assert X_mimic_test_standardised.index.equals(
    internal_test_predictions.index
)


assert {
    "sex_gender",
    "arrival_transport_grouped",
}.issubset(
    set(
        raw_feature_order
    )
)



try:

    import plotly
    import plotly.graph_objects as go

except ImportError as error:

    raise ImportError(
        "Plotly is required for the interactive SHAP figures.\n"
        "Install it in the current environment using:\n\n"
        "pip install plotly"
    ) from error



INTERACTIVE_SHAP_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "notebook2"
    / "interactive"
)


INTERACTIVE_SHAP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


interactive_shap_paths = {
    "top_summary": (
        INTERACTIVE_SHAP_DIR
        / "14_interactive_grouped_shap_top30.html"
    ),
    "arrival_transport": (
        INTERACTIVE_SHAP_DIR
        / "15_interactive_shap_arrival_transport.html"
    ),
    "sex_gender": (
        INTERACTIVE_SHAP_DIR
        / "16_interactive_shap_sex_gender.html"
    ),
    "colour_config": (
        INTERACTIVE_SHAP_DIR
        / "interactive_shap_colour_config.json"
    ),
    "artifact_manifest": (
        INTERACTIVE_SHAP_DIR
        / "interactive_shap_artifact_manifest.csv"
    ),
}



arrival_transport_colour_map = {
    "EMS": "#009E73",
    "OTHER_OR_UNKNOWN": "#E69F00",
    "SELF": "#8E44AD",
    "__MISSING__": "#7F8C8D",
    "UNKNOWN_TO_TRAINING_ENCODER": "#7F8C8D",
}


sex_gender_colour_map = {
    "F": "#7B2CBF",
    "M": "#F4A261",
    "__MISSING__": "#7F8C8D",
    "UNKNOWN_TO_TRAINING_ENCODER": "#7F8C8D",
}


categorical_colour_maps = {
    "arrival_transport_grouped": (
        arrival_transport_colour_map
    ),
    "sex_gender": (
        sex_gender_colour_map
    ),
}


interactive_shap_colour_config = {
    "continuous_features": {
        "colour_scale": {
    "low": "#008BFB",
    "high": "#FF0051",
    },
        "interpretation": (
            "Relative feature value within each feature row"
        ),
    },
    "arrival_transport_grouped": (
        arrival_transport_colour_map
    ),
    "sex_gender": (
        sex_gender_colour_map
    ),
}


with interactive_shap_paths[
    "colour_config"
].open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        interactive_shap_colour_config,
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )



row_count = len(
    internal_test_predictions
)


row_positions = np.arange(
    row_count,
    dtype=int,
)


hover_outcome = np.where(
    internal_test_predictions[
        "outcome_binary"
    ]
    .astype(int)
    .to_numpy()
    ==
    1,
    "ADMITTED",
    "HOME",
)


hover_raw_probability = (
    pd.to_numeric(
        internal_test_predictions[
            "raw_probability"
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=float
    )
)


hover_calibrated_probability = (
    pd.to_numeric(
        internal_test_predictions[
            "calibrated_probability"
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=float
    )
)



raw_feature_position_lookup = {
    feature_name: feature_position
    for feature_position, feature_name
    in enumerate(
        raw_feature_order
    )
}



INTERACTIVE_SHAP_RANDOM_SEED = 42


interactive_shap_rng = np.random.default_rng(
    INTERACTIVE_SHAP_RANDOM_SEED
)


def create_vertical_jitter(
    size,
    width=0.28,
):

    return interactive_shap_rng.uniform(
        low=-width,
        high=width,
        size=size,
    )



def normalise_feature_values(
    values,
):

    values = np.asarray(
        values,
        dtype=float,
    )


    finite_mask = np.isfinite(
        values
    )


    normalised_values = np.full(
        len(
            values
        ),
        0.5,
        dtype=float,
    )


    if not finite_mask.any():

        return normalised_values


    finite_values = values[
        finite_mask
    ]


    lower_value = float(
        np.percentile(
            finite_values,
            5,
        )
    )


    upper_value = float(
        np.percentile(
            finite_values,
            95,
        )
    )


    if np.isclose(
        lower_value,
        upper_value,
    ):

        minimum_value = float(
            finite_values.min()
        )


        maximum_value = float(
            finite_values.max()
        )


        if np.isclose(
            minimum_value,
            maximum_value,
        ):

            normalised_values[
                finite_mask
            ] = 0.5

        else:

            normalised_values[
                finite_mask
            ] = (
                finite_values
                -
                minimum_value
            ) / (
                maximum_value
                -
                minimum_value
            )


    else:

        clipped_values = np.clip(
            finite_values,
            lower_value,
            upper_value,
        )


        normalised_values[
            finite_mask
        ] = (
            clipped_values
            -
            lower_value
        ) / (
            upper_value
            -
            lower_value
        )


    return normalised_values



TOP_INTERACTIVE_SHAP_FEATURES = 30


top_interactive_features = (
    raw_feature_shap_importance
    .sort_values(
        "mean_absolute_shap",
        ascending=False,
    )
    .head(
        TOP_INTERACTIVE_SHAP_FEATURES
    )[
        "source_raw_feature"
    ]
    .tolist()
)


interactive_summary_figure = go.Figure()


continuous_colourbar_added = False


categorical_legend_entries_added = set()


for feature_rank, feature_name in enumerate(
    top_interactive_features
):

    feature_position = (
        raw_feature_position_lookup[
            feature_name
        ]
    )


    feature_shap_values = (
        internal_test_grouped_raw_shap_values[
            :,
            feature_position,
        ]
    )


    feature_y_position = (
        TOP_INTERACTIVE_SHAP_FEATURES
        -
        1
        -
        feature_rank
    )


    jittered_y = (
        feature_y_position
        +
        create_vertical_jitter(
            row_count,
            width=0.27,
        )
    )



    if feature_name in categorical_colour_maps:

        categorical_values = (
            X_mimic_test_standardised[
                feature_name
            ]
            .astype(
                "string"
            )
            .fillna(
                "__MISSING__"
            )
            .astype(str)
            .to_numpy()
        )


        category_order = list(
            categorical_colour_maps[
                feature_name
            ].keys()
        )


        observed_categories = [
            category_name
            for category_name
            in category_order
            if np.any(
                categorical_values
                ==
                category_name
            )
        ]


        additional_categories = sorted(
            set(
                categorical_values
            )
            -
            set(
                observed_categories
            )
        )


        observed_categories.extend(
            additional_categories
        )


        for category_name in observed_categories:

            category_mask = (
                categorical_values
                ==
                category_name
            )


            if not category_mask.any():

                continue


            category_colour = (
                categorical_colour_maps[
                    feature_name
                ]
                .get(
                    category_name,
                    "#7F8C8D",
                )
            )


            legend_key = (
                feature_name,
                category_name,
            )


            show_category_legend = (
                legend_key
                not in categorical_legend_entries_added
            )


            categorical_legend_entries_added.add(
                legend_key
            )


            category_custom_data = np.column_stack(
                [
                    np.full(
                        category_mask.sum(),
                        feature_name,
                        dtype=object,
                    ),
                    categorical_values[
                        category_mask
                    ],
                    hover_outcome[
                        category_mask
                    ],
                    hover_raw_probability[
                        category_mask
                    ],
                    hover_calibrated_probability[
                        category_mask
                    ],
                ]
            )


            interactive_summary_figure.add_trace(
                go.Scattergl(
                    x=feature_shap_values[
                        category_mask
                    ],
                    y=jittered_y[
                        category_mask
                    ],
                    mode="markers",
                    name=(
                        f"{feature_name}: "
                        f"{category_name}"
                    ),
                    legendgroup=(
                        f"{feature_name}_{category_name}"
                    ),
                    showlegend=(
                        show_category_legend
                    ),
                    marker={
                        "size": 5,
                        "opacity": 0.72,
                        "color": category_colour,
                    },
                    customdata=category_custom_data,
                    hovertemplate=(
                        "<b>%{customdata[0]}</b>"
                        "<br>Category: %{customdata[1]}"
                        "<br>SHAP value: %{x:.5f}"
                        "<br>Outcome: %{customdata[2]}"
                        "<br>Raw probability: "
                        "%{customdata[3]:.4f}"
                        "<br>Calibrated probability: "
                        "%{customdata[4]:.4f}"
                        "<extra></extra>"
                    ),
                )
            )



    else:

        feature_values = (
            internal_test_grouped_raw_feature_values[
                :,
                feature_position,
            ]
        )


        normalised_colour_values = (
            normalise_feature_values(
                feature_values
            )
        )


        continuous_custom_data = np.column_stack(
            [
                np.full(
                    row_count,
                    feature_name,
                    dtype=object,
                ),
                feature_values,
                hover_outcome,
                hover_raw_probability,
                hover_calibrated_probability,
            ]
        )


        marker_configuration = {
            "size": 5,
            "opacity": 0.72,
            "color": normalised_colour_values,
            "colorscale": [
    [0.0, "#008BFB"],
    [1.0, "#FF0051"],
],
            "cmin": 0.0,
            "cmax": 1.0,
            "showscale": (
                not continuous_colourbar_added
            ),
        }


        if not continuous_colourbar_added:

            marker_configuration[
                "colorbar"
            ] = {
                "title": {
                    "text": (
                        "Relative feature value"
                        "<br>within each row"
                    ),
                },
                "tickvals": [
                    0.0,
                    1.0,
                ],
                "ticktext": [
                    "Low",
                    "High",
                ],
                "len": 0.45,
                "y": 0.75,
            }


            continuous_colourbar_added = True


        interactive_summary_figure.add_trace(
            go.Scattergl(
                x=feature_shap_values,
                y=jittered_y,
                mode="markers",
                name=feature_name,
                showlegend=False,
                marker=marker_configuration,
                customdata=continuous_custom_data,
                hovertemplate=(
                    "<b>%{customdata[0]}</b>"
                    "<br>Feature value: "
                    "%{customdata[1]:.5g}"
                    "<br>SHAP value: %{x:.5f}"
                    "<br>Outcome: %{customdata[2]}"
                    "<br>Raw probability: "
                    "%{customdata[3]:.4f}"
                    "<br>Calibrated probability: "
                    "%{customdata[4]:.4f}"
                    "<extra></extra>"
                ),
            )
        )


interactive_summary_figure.add_vline(
    x=0.0,
    line_width=1.2,
    line_dash="dash",
    line_color="grey",
)


interactive_summary_figure.update_layout(
    title={
        "text": (
            "Interactive grouped SHAP summary: "
            "locked internal temporal-test model"
        ),
        "x": 0.5,
    },
    xaxis={
        "title": (
            "Grouped SHAP value "
            "(positive → ADMITTED, negative → HOME)"
        ),
        "zeroline": False,
        "showgrid": True,
        "gridcolor": "rgba(128,128,128,0.15)",
    },
    yaxis={
        "title": "",
        "tickmode": "array",
        "tickvals": list(
            range(
                TOP_INTERACTIVE_SHAP_FEATURES
            )
        ),
        "ticktext": (
            top_interactive_features[
                ::-1
            ]
        ),
        "range": [
            -1,
            TOP_INTERACTIVE_SHAP_FEATURES,
        ],
        "showgrid": False,
    },
    height=1050,
    width=1450,
    template="plotly_white",
    hovermode="closest",
    legend={
        "title": {
            "text": (
                "Categorical feature levels"
            ),
        },
        "orientation": "v",
        "x": 1.02,
        "y": 0.48,
    },
    margin={
        "l": 340,
        "r": 320,
        "t": 90,
        "b": 80,
    },
)


interactive_summary_figure.write_html(
    interactive_shap_paths[
        "top_summary"
    ],
    include_plotlyjs=True,
    full_html=True,
    auto_open=False,
)


interactive_summary_figure.show()



def create_interactive_categorical_shap_plot(
    feature_name,
    colour_map,
    title,
    output_path,
):

    feature_position = (
        raw_feature_position_lookup[
            feature_name
        ]
    )


    feature_shap_values = (
        internal_test_grouped_raw_shap_values[
            :,
            feature_position,
        ]
    )


    categorical_values = (
        X_mimic_test_standardised[
            feature_name
        ]
        .astype(
            "string"
        )
        .fillna(
            "__MISSING__"
        )
        .astype(str)
        .to_numpy()
    )


    categorical_figure = go.Figure()


    category_order = [
        category_name
        for category_name
        in colour_map.keys()
        if np.any(
            categorical_values
            ==
            category_name
        )
    ]


    unlisted_categories = sorted(
        set(
            categorical_values
        )
        -
        set(
            category_order
        )
    )


    category_order.extend(
        unlisted_categories
    )


    for category_number, category_name in enumerate(
        category_order
    ):

        category_mask = (
            categorical_values
            ==
            category_name
        )


        if not category_mask.any():

            continue


        category_colour = colour_map.get(
            category_name,
            "#7F8C8D",
        )


        category_jitter = create_vertical_jitter(
            category_mask.sum(),
            width=0.23,
        )


        category_custom_data = np.column_stack(
            [
                hover_outcome[
                    category_mask
                ],
                hover_raw_probability[
                    category_mask
                ],
                hover_calibrated_probability[
                    category_mask
                ],
            ]
        )


        categorical_figure.add_trace(
            go.Scattergl(
                x=feature_shap_values[
                    category_mask
                ],
                y=category_jitter,
                mode="markers",
                name=category_name,
                marker={
                    "size": 8,
                    "opacity": 0.72,
                    "color": category_colour,
                },
                customdata=category_custom_data,
                hovertemplate=(
                    f"<b>{feature_name}</b>"
                    f"<br>Category: {category_name}"
                    "<br>SHAP value: %{x:.5f}"
                    "<br>Outcome: %{customdata[0]}"
                    "<br>Raw probability: "
                    "%{customdata[1]:.4f}"
                    "<br>Calibrated probability: "
                    "%{customdata[2]:.4f}"
                    "<extra></extra>"
                ),
            )
        )


    categorical_figure.add_vline(
        x=0.0,
        line_width=1.2,
        line_dash="dash",
        line_color="grey",
    )


    categorical_figure.update_layout(
        title={
            "text": title,
            "x": 0.5,
        },
        xaxis={
            "title": (
                "Grouped SHAP value "
                "(positive → ADMITTED, negative → HOME)"
            ),
            "zeroline": False,
            "showgrid": True,
            "gridcolor": "rgba(128,128,128,0.15)",
        },
        yaxis={
            "title": "",
            "showticklabels": False,
            "showgrid": False,
            "range": [
                -0.45,
                0.45,
            ],
        },
        template="plotly_white",
        hovermode="closest",
        height=560,
        width=1150,
        legend={
            "title": {
                "text": "Category",
            },
            "orientation": "h",
            "x": 0.5,
            "xanchor": "center",
            "y": -0.16,
        },
        margin={
            "l": 70,
            "r": 60,
            "t": 90,
            "b": 120,
        },
    )


    categorical_figure.write_html(
        output_path,
        include_plotlyjs=True,
        full_html=True,
        auto_open=False,
    )


    categorical_figure.show()


    return categorical_figure



interactive_arrival_transport_figure = (
    create_interactive_categorical_shap_plot(
        feature_name=(
            "arrival_transport_grouped"
        ),
        colour_map=(
            arrival_transport_colour_map
        ),
        title=(
            "Interactive grouped SHAP distribution: "
            "arrival transport"
        ),
        output_path=(
            interactive_shap_paths[
                "arrival_transport"
            ]
        ),
    )
)



interactive_sex_gender_figure = (
    create_interactive_categorical_shap_plot(
        feature_name=(
            "sex_gender"
        ),
        colour_map=(
            sex_gender_colour_map
        ),
        title=(
            "Interactive grouped SHAP distribution: "
            "sex/gender"
        ),
        output_path=(
            interactive_shap_paths[
                "sex_gender"
            ]
        ),
    )
)



interactive_categorical_shap_records = []


for feature_name in [
    "arrival_transport_grouped",
    "sex_gender",
]:

    feature_position = (
        raw_feature_position_lookup[
            feature_name
        ]
    )


    feature_categories = (
        X_mimic_test_standardised[
            feature_name
        ]
        .astype(
            "string"
        )
        .fillna(
            "__MISSING__"
        )
        .astype(str)
        .to_numpy()
    )


    feature_shap_values = (
        internal_test_grouped_raw_shap_values[
            :,
            feature_position,
        ]
    )


    for row_position in range(
        row_count
    ):

        interactive_categorical_shap_records.append(
            {
                "shap_row_position": int(
                    row_position
                ),
                "outcome": (
                    hover_outcome[
                        row_position
                    ]
                ),
                "feature": feature_name,
                "category": (
                    feature_categories[
                        row_position
                    ]
                ),
                "grouped_shap_value": float(
                    feature_shap_values[
                        row_position
                    ]
                ),
                "raw_probability": float(
                    hover_raw_probability[
                        row_position
                    ]
                ),
                "calibrated_probability": float(
                    hover_calibrated_probability[
                        row_position
                    ]
                ),
            }
        )


interactive_categorical_shap_data = pd.DataFrame(
    interactive_categorical_shap_records
)


interactive_categorical_shap_data_path = (
    INTERACTIVE_SHAP_DIR
    / "interactive_categorical_shap_plot_data.parquet"
)


interactive_categorical_shap_data.to_parquet(
    interactive_categorical_shap_data_path,
    index=False,
    engine="pyarrow",
)



def interactive_shap_file_sha256(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )


    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()


interactive_shap_artifacts = {
    "interactive_top30_summary": (
        interactive_shap_paths[
            "top_summary"
        ]
    ),
    "interactive_arrival_transport": (
        interactive_shap_paths[
            "arrival_transport"
        ]
    ),
    "interactive_sex_gender": (
        interactive_shap_paths[
            "sex_gender"
        ]
    ),
    "interactive_colour_config": (
        interactive_shap_paths[
            "colour_config"
        ]
    ),
    "interactive_categorical_plot_data": (
        interactive_categorical_shap_data_path
    ),
}


interactive_shap_artifact_records = []


for artifact_name, artifact_path in (
    interactive_shap_artifacts.items()
):

    if not artifact_path.exists():

        raise FileNotFoundError(
            "Expected interactive SHAP artifact was not "
            f"created:\n{artifact_path}"
        )


    interactive_shap_artifact_records.append(
        {
            "artifact": artifact_name,
            "filename": artifact_path.name,
            "directory": str(
                artifact_path.parent
            ),
            "size_bytes": int(
                artifact_path
                .stat()
                .st_size
            ),
            "sha256": (
                interactive_shap_file_sha256(
                    artifact_path
                )
            ),
        }
    )


interactive_shap_artifact_manifest = pd.DataFrame(
    interactive_shap_artifact_records
)


interactive_shap_artifact_manifest.to_csv(
    interactive_shap_paths[
        "artifact_manifest"
    ],
    index=False,
)



print(
    "Interactive SHAP artifact manifest:"
)

display(
    interactive_shap_artifact_manifest
)


print(
    "\nINTERACTIVE GROUPED SHAP FIGURES COMPLETED"
)

print(
    "Interactive top-30 summary saved: YES"
)

print(
    "Interactive arrival-transport plot saved: YES"
)

print(
    "Interactive gender plot saved: YES"
)

print(
    "Standalone Plotly HTML files: YES"
)

print(
    "Hover patient metadata included: YES"
)

print(
    "Categorical plot data saved: YES"
)

print(
    "\nArrival-transport colours:"
)

for category_name, colour_value in (
    arrival_transport_colour_map.items()
):

    print(
        f"  {category_name}: {colour_value}"
    )


print(
    "\nGender colours:"
)

for category_name, colour_value in (
    sex_gender_colour_map.items()
):

    print(
        f"  {category_name}: {colour_value}"
    )


print(
    "\nOne-hot SHAP values recalculated: NO"
)

print(
    "Grouped SHAP values changed: NO"
)

print(
    "Model refitted: NO"
)

print(
    "Calibration changed: NO"
)

print(
    "Threshold changed: NO"
)

print(
    "MC-MED loaded: NO"
)

### Final Notebook 2 SHAP hand-off and completion audit

The patient-level grouped SHAP matrix, corresponding grouped feature-value matrix, outcome metadata and complete 2,495-feature universe are saved as a self-contained hand-off for subsequent robust SHAP filtering and clinician comparison.

The 2,495 grouped raw features constitute the clinically interpretable analysis space. The 2,498 prepared model columns are retained only as a technical audit of one-hot encoding and model input structure.

No additional model fitting, calibration, threshold selection or internal-test evaluation is performed in this section.

In [ ]:



required_final_handoff_objects = [
    "PROJECT_ROOT",
    "raw_feature_order",
    "raw_feature_order_sha256",
    "prepared_shap_feature_order",
    "prepared_shap_feature_mapping",
    "raw_feature_shap_importance",
    "internal_test_prepared_shap_values",
    "internal_test_grouped_raw_shap_values",
    "internal_test_grouped_raw_feature_values",
    "internal_test_native_bias",
    "categorical_beeswarm_code_mapping",
    "X_mimic_test_standardised",
    "y_mimic_test",
    "internal_test_predictions",
    "locked_decision_threshold",
]


missing_final_handoff_objects = [
    object_name
    for object_name in required_final_handoff_objects
    if object_name not in globals()
]


if missing_final_handoff_objects:

    raise RuntimeError(
        "Complete the grouped SHAP analysis before creating "
        "the final Notebook 2 hand-off.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_final_handoff_objects
        )
    )



N_SHAP_ROWS = 1851

N_PREPARED_FEATURES = 2498

N_GROUPED_RAW_FEATURES = 2495


assert internal_test_prepared_shap_values.shape == (
    N_SHAP_ROWS,
    N_PREPARED_FEATURES,
)


assert internal_test_grouped_raw_shap_values.shape == (
    N_SHAP_ROWS,
    N_GROUPED_RAW_FEATURES,
)


assert internal_test_grouped_raw_feature_values.shape == (
    N_SHAP_ROWS,
    N_GROUPED_RAW_FEATURES,
)


assert internal_test_native_bias.shape == (
    N_SHAP_ROWS,
)


assert len(
    prepared_shap_feature_order
) == N_PREPARED_FEATURES


assert len(
    raw_feature_order
) == N_GROUPED_RAW_FEATURES


assert len(
    set(
        raw_feature_order
    )
) == N_GROUPED_RAW_FEATURES


assert len(
    internal_test_predictions
) == N_SHAP_ROWS


assert len(
    y_mimic_test
) == N_SHAP_ROWS


assert X_mimic_test_standardised.index.equals(
    y_mimic_test.index
)


assert X_mimic_test_standardised.index.equals(
    internal_test_predictions.index
)


assert np.isfinite(
    internal_test_prepared_shap_values
).all()


assert np.isfinite(
    internal_test_grouped_raw_shap_values
).all()


assert np.isfinite(
    internal_test_native_bias
).all()



assert np.allclose(
    internal_test_prepared_shap_values.sum(
        axis=1
    ),
    internal_test_grouped_raw_shap_values.sum(
        axis=1
    ),
    rtol=1e-5,
    atol=1e-5,
)



raw_feature_order_table = pd.DataFrame(
    {
        "raw_feature_position": np.arange(
            N_GROUPED_RAW_FEATURES,
            dtype=int,
        ),
        "source_raw_feature": list(
            raw_feature_order
        ),
    }
)


grouped_shap_feature_universe = (
    raw_feature_order_table
    .merge(
        raw_feature_shap_importance,
        on="source_raw_feature",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "raw_feature_position"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    grouped_shap_feature_universe
) == N_GROUPED_RAW_FEATURES


assert grouped_shap_feature_universe[
    "source_raw_feature"
].tolist() == list(
    raw_feature_order
)


assert grouped_shap_feature_universe[
    "mean_absolute_shap"
].notna().all()



grouped_shap_feature_universe[
    "minimum_shap"
] = np.min(
    internal_test_grouped_raw_shap_values,
    axis=0,
)


grouped_shap_feature_universe[
    "maximum_shap"
] = np.max(
    internal_test_grouped_raw_shap_values,
    axis=0,
)


grouped_shap_feature_universe[
    "median_shap"
] = np.median(
    internal_test_grouped_raw_shap_values,
    axis=0,
)


grouped_shap_feature_universe[
    "shap_p05"
] = np.percentile(
    internal_test_grouped_raw_shap_values,
    5,
    axis=0,
)


grouped_shap_feature_universe[
    "shap_p25"
] = np.percentile(
    internal_test_grouped_raw_shap_values,
    25,
    axis=0,
)


grouped_shap_feature_universe[
    "shap_p75"
] = np.percentile(
    internal_test_grouped_raw_shap_values,
    75,
    axis=0,
)


grouped_shap_feature_universe[
    "shap_p95"
] = np.percentile(
    internal_test_grouped_raw_shap_values,
    95,
    axis=0,
)


grouped_shap_feature_universe[
    "nonzero_shap_fraction"
] = np.mean(
    ~np.isclose(
        internal_test_grouped_raw_shap_values,
        0.0,
        rtol=0.0,
        atol=1e-12,
    ),
    axis=0,
)


grouped_shap_feature_universe[
    "positive_shap_fraction_recalculated"
] = np.mean(
    internal_test_grouped_raw_shap_values
    >
    0,
    axis=0,
)


grouped_shap_feature_universe[
    "negative_shap_fraction"
] = np.mean(
    internal_test_grouped_raw_shap_values
    <
    0,
    axis=0,
)



clinician_comparison_feature_universe = (
    grouped_shap_feature_universe[
        [
            "raw_feature_position",
            "source_raw_feature",
            "family",
            "timing_stage",
            "storage_type",
            "prepared_columns_grouped",
            "raw_feature_rank",
            "mean_absolute_shap",
            "global_importance_percent",
            "mean_signed_shap",
            "positive_shap_fraction",
            "negative_shap_fraction",
            "nonzero_shap_fraction",
            "minimum_shap",
            "maximum_shap",
            "median_shap",
            "shap_p05",
            "shap_p25",
            "shap_p75",
            "shap_p95",
        ]
    ]
    .copy()
)



clinician_comparison_feature_universe[
    "clinician_group"
] = pd.NA


clinician_comparison_feature_universe[
    "clinician_important"
] = pd.NA


clinician_comparison_feature_universe[
    "clinician_priority"
] = pd.NA


clinician_comparison_feature_universe[
    "clinician_comment"
] = pd.NA


assert len(
    clinician_comparison_feature_universe
) == N_GROUPED_RAW_FEATURES



shap_internal_test_row_metadata = (
    internal_test_predictions
    .copy()
)


shap_internal_test_row_metadata.insert(
    0,
    "shap_row_position",
    np.arange(
        N_SHAP_ROWS,
        dtype=int,
    ),
)


assert shap_internal_test_row_metadata[
    "shap_row_position"
].tolist() == list(
    range(
        N_SHAP_ROWS
    )
)


assert (
    shap_internal_test_row_metadata[
        "outcome_binary"
    ]
    .astype(int)
    .to_numpy()
    ==
    y_mimic_test
    .astype(int)
    .to_numpy()
).all()



FINAL_SHAP_ARRAY_DIR = (
    PROJECT_ROOT
    / "results"
    / "shap"
    / "notebook2"
)


FINAL_SHAP_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook2"
)


FINAL_SHAP_PREDICTION_DIR = (
    PROJECT_ROOT
    / "results"
    / "predictions"
    / "notebook2"
)


for output_directory in [
    FINAL_SHAP_ARRAY_DIR,
    FINAL_SHAP_TABLE_DIR,
    FINAL_SHAP_PREDICTION_DIR,
]:

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )



final_shap_handoff_paths = {
    "grouped_analysis_bundle": (
        FINAL_SHAP_ARRAY_DIR
        / "internal_test_grouped_shap_analysis_bundle.npz"
    ),
    "grouped_feature_values": (
        FINAL_SHAP_ARRAY_DIR
        / "internal_test_grouped_raw_feature_values.npz"
    ),
    "grouped_feature_universe": (
        FINAL_SHAP_TABLE_DIR
        / "shap_grouped_feature_universe.csv"
    ),
    "clinician_comparison_universe": (
        FINAL_SHAP_TABLE_DIR
        / "clinician_comparison_feature_universe.csv"
    ),
    "row_metadata": (
        FINAL_SHAP_PREDICTION_DIR
        / "shap_internal_test_row_metadata.parquet"
    ),
    "handoff_config": (
        FINAL_SHAP_ARRAY_DIR
        / "notebook2_shap_handoff_config.json"
    ),
    "handoff_manifest": (
        FINAL_SHAP_TABLE_DIR
        / "notebook2_shap_handoff_manifest.csv"
    ),
}



np.savez_compressed(
    final_shap_handoff_paths[
        "grouped_analysis_bundle"
    ],
    shap_values=(
        internal_test_grouped_raw_shap_values
    ),
    feature_values=(
        internal_test_grouped_raw_feature_values
    ),
    bias=(
        internal_test_native_bias
    ),
    raw_feature_names=np.asarray(
        raw_feature_order,
        dtype=str,
    ),
    prepared_feature_names=np.asarray(
        prepared_shap_feature_order,
        dtype=str,
    ),
    stay_ids=np.asarray(
        X_mimic_test_standardised.index
    ),
    outcome_binary=(
        y_mimic_test
        .to_numpy(
            dtype=np.int8
        )
    ),
)


np.savez_compressed(
    final_shap_handoff_paths[
        "grouped_feature_values"
    ],
    feature_values=(
        internal_test_grouped_raw_feature_values
    ),
    raw_feature_names=np.asarray(
        raw_feature_order,
        dtype=str,
    ),
    stay_ids=np.asarray(
        X_mimic_test_standardised.index
    ),
)



grouped_shap_feature_universe.to_csv(
    final_shap_handoff_paths[
        "grouped_feature_universe"
    ],
    index=False,
)


clinician_comparison_feature_universe.to_csv(
    final_shap_handoff_paths[
        "clinician_comparison_universe"
    ],
    index=False,
)


shap_internal_test_row_metadata.to_parquet(
    final_shap_handoff_paths[
        "row_metadata"
    ],
    index=True,
    engine="pyarrow",
)



notebook2_shap_handoff_config = {
    "analysis_stage": (
        "Notebook 2 grouped SHAP complete"
    ),
    "explanation_cohort": (
        "MIMIC-IV internal temporal test"
    ),
    "rows": N_SHAP_ROWS,
    "prepared_model_columns": N_PREPARED_FEATURES,
    "grouped_raw_features": N_GROUPED_RAW_FEATURES,
    "clinician_comparison_feature_count": (
        N_GROUPED_RAW_FEATURES
    ),
    "clinician_comparison_representation": (
        "Grouped raw features"
    ),
    "categorical_grouping": {
        "sex_gender": [
            "F",
            "M",
        ],
        "arrival_transport_grouped": [
            "EMS",
            "OTHER_OR_UNKNOWN",
            "SELF",
        ],
    },
    "feature_order_sha256": (
        raw_feature_order_sha256
    ),
    "shap_calculation_method": (
        "XGBoost native pred_contribs"
    ),
    "shap_output_scale": (
        "Raw XGBoost margin/log-odds"
    ),
    "later_analyses_supported": [
        "Distance from zero",
        "Minimum and maximum SHAP distance",
        "SHAP percentile filtering",
        "Positive and negative direction consistency",
        "Feature support and stability filtering",
        "Clinician-versus-model feature comparison",
        "Feature-value direction analysis",
        "Grouped SHAP beeswarm regeneration",
    ],
    "locked_model_threshold": float(
        locked_decision_threshold
    ),
    "model_refitted": False,
    "calibrator_refitted": False,
    "threshold_changed": False,
    "mcmed_used": False,
}


with final_shap_handoff_paths[
    "handoff_config"
].open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        notebook2_shap_handoff_config,
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )



reloaded_grouped_bundle = np.load(
    final_shap_handoff_paths[
        "grouped_analysis_bundle"
    ],
    allow_pickle=False,
)


assert reloaded_grouped_bundle[
    "shap_values"
].shape == (
    N_SHAP_ROWS,
    N_GROUPED_RAW_FEATURES,
)


assert reloaded_grouped_bundle[
    "feature_values"
].shape == (
    N_SHAP_ROWS,
    N_GROUPED_RAW_FEATURES,
)


assert reloaded_grouped_bundle[
    "bias"
].shape == (
    N_SHAP_ROWS,
)


assert reloaded_grouped_bundle[
    "raw_feature_names"
].tolist() == list(
    raw_feature_order
)


assert (
    reloaded_grouped_bundle[
        "outcome_binary"
    ]
    ==
    y_mimic_test.to_numpy(
        dtype=np.int8
    )
).all()


reloaded_grouped_bundle.close()


reloaded_feature_universe = pd.read_csv(
    final_shap_handoff_paths[
        "grouped_feature_universe"
    ]
)


assert len(
    reloaded_feature_universe
) == N_GROUPED_RAW_FEATURES


assert reloaded_feature_universe[
    "source_raw_feature"
].tolist() == list(
    raw_feature_order
)


reloaded_row_metadata = pd.read_parquet(
    final_shap_handoff_paths[
        "row_metadata"
    ],
    engine="pyarrow",
)


assert len(
    reloaded_row_metadata
) == N_SHAP_ROWS



def final_shap_file_sha256(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )


    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()


final_shap_handoff_records = []


for artifact_name, artifact_path in (
    final_shap_handoff_paths.items()
):

    if artifact_name == "handoff_manifest":

        continue


    if not artifact_path.exists():

        raise FileNotFoundError(
            "Expected final SHAP hand-off artifact was not "
            f"written:\n{artifact_path}"
        )


    final_shap_handoff_records.append(
        {
            "artifact": artifact_name,
            "filename": artifact_path.name,
            "directory": str(
                artifact_path.parent
            ),
            "size_bytes": int(
                artifact_path
                .stat()
                .st_size
            ),
            "sha256": final_shap_file_sha256(
                artifact_path
            ),
        }
    )


notebook2_shap_handoff_manifest = pd.DataFrame(
    final_shap_handoff_records
)


notebook2_shap_handoff_manifest.to_csv(
    final_shap_handoff_paths[
        "handoff_manifest"
    ],
    index=False,
)



notebook2_completion_audit = pd.DataFrame(
    [
        {
            "selected_algorithm": "XGBoost",
            "training_rows": 3561,
            "validation_rows": 3138,
            "internal_test_rows": 1851,
            "raw_features": 2495,
            "prepared_model_columns": 2498,
            "grouped_clinical_features": 2495,
            "calibration": "Platt scaling",
            "locked_threshold": float(
                locked_decision_threshold
            ),
            "internal_test_evaluated": True,
            "incremental_analysis_completed": True,
            "grouped_shap_completed": True,
            "robust_shap_inputs_saved": True,
            "clinician_comparison_universe_saved": True,
            "model_development_locked": True,
            "mcmed_used": False,
            "notebook2_complete": True,
        }
    ]
)


print(
    "Final grouped-SHAP hand-off manifest:"
)

display(
    notebook2_shap_handoff_manifest
)


print(
    "\nNotebook 2 completion audit:"
)

display(
    notebook2_completion_audit
)


print(
    "\nNOTEBOOK 2 COMPLETED"
)

print(
    "Prepared XGBoost columns:",
    N_PREPARED_FEATURES,
)

print(
    "Grouped clinically interpretable features:",
    N_GROUPED_RAW_FEATURES,
)

print(
    "Feature universe for clinician comparison:",
    N_GROUPED_RAW_FEATURES,
)

print(
    "\nDistance-from-zero analysis supported: YES"
)

print(
    "Minimum/maximum SHAP analysis supported: YES"
)

print(
    "Percentile and stability filtering supported: YES"
)

print(
    "Feature-value direction analysis supported: YES"
)

print(
    "Clinician-versus-model comparison supported: YES"
)

print(
    "Categorical grouping preserved: YES"
)

print(
    "Row and feature alignment verified: YES"
)

print(
    "Checksums saved: YES"
)

print(
    "\nModel refitted: NO"
)

print(
    "Calibration changed: NO"
)

print(
    "Threshold changed: NO"
)

print(
    "MC-MED loaded: NO"
)